# Neural Architecture and Extrapolation Uncertainty in Hybrid Hydrological Projections

# # Data Preparation for the 5 Model
# The output file is saved as LSTM_Prepared_v4 but it is used for all the remaining 4 models. The code was run first only first to check its workability.

In [ ]:
# =============================================================================
# CELL 1 -- SETUP
# =============================================================================

import gc
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# -----------------------------------------------------------------------------
# PORTABLE PROJECT PATHS
# -----------------------------------------------------------------------------
# Run this notebook with the project root as the current working directory.
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"              # input Excel files
RESULTS_ROOT = PROJECT_DIR / "results"       # generated files and model outputs

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

HISTORICAL_XLSX  = DATA_DIR / "Historical.xlsx"
FUTURE_SCENARIOS = {
    "SSP245": DATA_DIR / "SSP245.xlsx",
    "SSP585": DATA_DIR / "SSP585.xlsx",
}
DESCRIPTOR_XLSX = DATA_DIR / "Station_Descriptor.xlsx"
PREPARED_DIR      = RESULTS_ROOT / "LSTM_Prepared_v4"      # new folder: v3 stays untouched

# -----------------------------------------------------------------------------
# STATIONS AND LOOKBACKS
# -----------------------------------------------------------------------------
STATIONS            = ["Shebe", "Asendabo", "Abelti"]
LOOKBACK_CANDIDATES = [30, 45, 60, 90, 120, 180]

# -----------------------------------------------------------------------------
# HISTORICAL SPLITS
# -----------------------------------------------------------------------------
TRAIN_START, TRAIN_END = "2000-01-01", "2015-12-31"
VAL_START,   VAL_END   = "2016-01-01", "2018-12-31"
TEST_START,  TEST_END  = "2019-01-01", "2021-12-31"

# -----------------------------------------------------------------------------
# FEATURE GROUPS
# -----------------------------------------------------------------------------
FORCING = ["PRECIPmm", "Tmax", "Tmin"]            # PETmm removed (redundant)

# -----------------------------------------------------------------------------
# SWAT STATE VARIABLES
# ETmm removed (redundant). PETmm is not used anywhere in the features now.
# -----------------------------------------------------------------------------
SWAT_STATE = [
    "SWmm",
    "SURQmm",
    "PERCmm",
    "WYLDmm",
]

FLOW = "FLOW_OUT"
OBS  = "Observed"

# -----------------------------------------------------------------------------
# ENGINEERED FEATURES -- long-memory precipitation and multi-scale EWMA
# (half-lives 10 / 30 / 90 days).
# -----------------------------------------------------------------------------
ENGINEERED = [
    "doy_sin",
    "doy_cos",
    "precip_7d",
    "precip_30d",
    "precip_90d",
    "precip_180d",
    "precip_365d",       # see warm-up caveat in Cell 3 / 4
    "ewma_precip_10",
    "ewma_precip_30",
    "ewma_precip_90",
    "T_range",
]

# Static descriptors: pcp_mm, tmax_oc, tmin_oc removed (redundant with forcing)
DESCRIPTORS = [
    "lat", "lon", "elevation_m", "area_km2", "slope_pct",
    "forest-frac", "agri_frac", "urban_frac", "soil_index",
]

# -----------------------------------------------------------------------------
# FINAL FEATURE LIST -- deduplicated while preserving order
# -----------------------------------------------------------------------------
def _dedup(seq):
    seen = set()
    out  = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

FEATURES = _dedup(FORCING + SWAT_STATE + [FLOW] + ENGINEERED + DESCRIPTORS)

# -----------------------------------------------------------------------------
# REQUIRED COLUMNS
# -----------------------------------------------------------------------------
HISTORICAL_REQUIRED = _dedup(FORCING + SWAT_STATE + [FLOW, OBS])
FUTURE_REQUIRED     = _dedup(FORCING + SWAT_STATE + [FLOW])

# -----------------------------------------------------------------------------
# Report
# -----------------------------------------------------------------------------
print("=" * 78)
print("V4 FEATURE CONFIGURATION")
print("=" * 78)
print(f"Total input features: {len(FEATURES)}")
for i, f in enumerate(FEATURES, start=1):
    print(f"{i:02d}. {f}")
print()
for name in ("PETmm", "ETmm", "pcp_mm", "tmax_oc", "tmin_oc", "GW_Qmm"):
    print(f"{name:<8} included : {name in FEATURES}")
print("=" * 78)
print(f"DATA_DIR   : {DATA_DIR}")
print(f"PREPARED_DIR : {PREPARED_DIR}")

In [ ]:
# =============================================================================
# CELL 2 -- FUNCTION DEFINITIONS
# -----------------------------------------------------------------------------
def _ewma_alpha(half_life_days):
    return 1.0 - 0.5 ** (1.0 / float(half_life_days))


# -----------------------------------------------------------------------------
# 1) Load station descriptors
# -----------------------------------------------------------------------------
def load_descriptors():
    print("\nLoading station descriptors...")

    if not DESCRIPTOR_XLSX.exists():
        raise FileNotFoundError(f"Descriptor file not found:\n{DESCRIPTOR_XLSX}")

    df = pd.read_excel(DESCRIPTOR_XLSX, engine="openpyxl")

    if "Station" not in df.columns:
        raise ValueError("Station_Descriptor.xlsx must contain 'Station'.")

    missing_cols = [c for c in DESCRIPTORS if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing descriptor columns: {missing_cols}")

    df = df[["Station"] + DESCRIPTORS].copy().set_index("Station")

    missing_stations = [s for s in STATIONS if s not in df.index]
    if missing_stations:
        raise ValueError(f"Missing descriptor rows: {missing_stations}")

    df[DESCRIPTORS] = df[DESCRIPTORS].apply(pd.to_numeric, errors="coerce")

    miss = df[DESCRIPTORS].isna().sum()
    miss = miss[miss > 0]
    if len(miss) > 0:
        print("\nMissing descriptor values detected:")
        for c, n in miss.items():
            print(f"  {c}: {int(n)}")
        print("Applying forward fill followed by backward fill...")
        df[DESCRIPTORS] = df[DESCRIPTORS].ffill().bfill()

    rem = df[DESCRIPTORS].isna().sum()
    rem = rem[rem > 0]
    if len(rem) > 0:
        raise ValueError(f"Descriptor values remain missing after fill:\n{rem}")

    out = df.loc[STATIONS, DESCRIPTORS].copy()
    print("\nStation descriptors loaded:")
    print(out)
    return out


# -----------------------------------------------------------------------------
# 2) Load one station sheet
# -----------------------------------------------------------------------------
def load_sheet(path, station, historical=True):
    print(f"  Loading {path.name} / {station}")

    if not path.exists():
        raise FileNotFoundError(f"Input file not found:\n{path}")

    df = pd.read_excel(path, sheet_name=station, engine="openpyxl")

    if "Date" not in df.columns:
        raise ValueError(f"{path.name}/{station}: missing Date column.")

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    if df["Date"].isna().any():
        raise ValueError(
            f"{path.name}/{station}: {int(df['Date'].isna().sum())} invalid Date values."
        )
    if df["Date"].duplicated().any():
        raise ValueError(
            f"{path.name}/{station}: {int(df['Date'].duplicated().sum())} duplicate dates."
        )

    required = HISTORICAL_REQUIRED if historical else FUTURE_REQUIRED
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name}/{station}: missing required columns: {missing}")

    df[required] = df[required].apply(pd.to_numeric, errors="coerce")
    df = df.sort_values("Date").reset_index(drop=True)

    miss = df[required].isna().sum()
    miss = miss[miss > 0]
    if len(miss) > 0:
        print("\n    Missing values detected:")
        for c, n in miss.items():
            print(f"      {c}: {int(n)}")
        print("    Applying forward fill + backward fill...")
        df[required] = df[required].ffill().bfill()

    rem = df[required].isna().sum()
    rem = rem[rem > 0]
    if len(rem) > 0:
        raise ValueError(
            f"{path.name}/{station}: missing values remain after fill:\n{rem}"
        )

    expected   = pd.date_range(df["Date"].min(), df["Date"].max(), freq="D")
    miss_dates = expected.difference(df["Date"])
    if len(miss_dates) > 0:
        raise ValueError(
            f"{path.name}/{station}: {len(miss_dates)} missing calendar days."
        )

    # -------------------------------------------------------------------------
    # Non-negative checks -- include the new SWAT state variables
    # -------------------------------------------------------------------------
    nonneg = ["PRECIPmm", "PETmm", "SWmm", "SURQmm", "ETmm", "PERCmm", "WYLDmm", FLOW]
    if historical:
        nonneg.append(OBS)
    for c in nonneg:
        n_neg = int((df[c] < 0).sum())
        if n_neg > 0:
            raise ValueError(
                f"{path.name}/{station}: {n_neg} negative values in {c}."
            )

    df["station"] = station
    return df.copy()


# -----------------------------------------------------------------------------
# 3) Feature engineering
# -----------------------------------------------------------------------------
def engineer(df, descriptor_row, historical=True):
    df = df.copy()
    doy = df["Date"].dt.dayofyear.to_numpy()

    engineered = pd.DataFrame(
        {
            # Seasonal harmonics
            "doy_sin": np.sin(2.0 * np.pi * doy / 365.25),
            "doy_cos": np.cos(2.0 * np.pi * doy / 365.25),

            # Short-memory rolling precipitation
            "precip_7d":  df["PRECIPmm"].rolling(window=7,   min_periods=7).sum(),
            "precip_30d": df["PRECIPmm"].rolling(window=30,  min_periods=30).sum(),

            # Long-memory rolling precipitation (Group 1)
            "precip_90d":  df["PRECIPmm"].rolling(window=90,  min_periods=90).sum(),
            "precip_180d": df["PRECIPmm"].rolling(window=180, min_periods=180).sum(),
            "precip_365d": df["PRECIPmm"].rolling(window=365, min_periods=365).sum(),

            # Multi-scale EWMA of precipitation (baseflow proxies)
            "ewma_precip_10": df["PRECIPmm"].ewm(alpha=_ewma_alpha(10), adjust=False).mean(),
            "ewma_precip_30": df["PRECIPmm"].ewm(alpha=_ewma_alpha(30), adjust=False).mean(),
            "ewma_precip_90": df["PRECIPmm"].ewm(alpha=_ewma_alpha(90), adjust=False).mean(),

            # Temperature range
            "T_range": df["Tmax"] - df["Tmin"],
        },
        index=df.index,
    )

    descriptor_data = pd.DataFrame(
        {c: float(descriptor_row[c]) for c in DESCRIPTORS},
        index=df.index,
    )

    df = pd.concat([df, engineered, descriptor_data], axis=1)

    # -------------------------------------------------------------------------
    # Drop rows where any rolling feature is NaN
    #
    # precip_365d forces the first 364 rows to be dropped.
    # If you want to skip this and start earlier, remove precip_365d from
    # ENGINEERED in Cell 1 and from this subset list.
    # -------------------------------------------------------------------------
    df = df.dropna(
        subset=[
            "precip_7d",
            "precip_30d",
            "precip_90d",
            "precip_180d",
            "precip_365d",
        ]
    ).reset_index(drop=True)

    # -------------------------------------------------------------------------
    # Historical residual target
    # -------------------------------------------------------------------------
    if historical:
        log_obs  = np.log1p(np.maximum(df[OBS].to_numpy(dtype=np.float64),  0.0))
        log_swat = np.log1p(np.maximum(df[FLOW].to_numpy(dtype=np.float64), 0.0))
        target = pd.DataFrame(
            {
                "log_obs":         log_obs,
                "log_swat":        log_swat,
                "residual_target": log_obs - log_swat,
            },
            index=df.index,
        )
        df = pd.concat([df, target], axis=1)

    return df.copy()


# -----------------------------------------------------------------------------
# 4) Build sequences
# -----------------------------------------------------------------------------
def make_sequences(df, scaler, station_id, lookback, split=None, historical=True):
    if split == "train":
        mask = (df["Date"] >= TRAIN_START) & (df["Date"] <= TRAIN_END)
    elif split == "val":
        mask = (df["Date"] >= VAL_START)   & (df["Date"] <= VAL_END)
    elif split == "test":
        mask = (df["Date"] >= TEST_START)  & (df["Date"] <= TEST_END)
    else:
        mask = np.ones(len(df), dtype=bool)

    Xall = scaler.transform(
        df[FEATURES].to_numpy(dtype=np.float64)
    ).astype(np.float32)

    dates       = df["Date"].to_numpy()
    station_ids = np.full(len(df), station_id, dtype=np.int32)
    flow        = df[FLOW].to_numpy(dtype=np.float32)

    if historical:
        observed = df[OBS].to_numpy(dtype=np.float32)
        residual = df["residual_target"].to_numpy(dtype=np.float32)

    Xs, ys, ss, ds, fs, obs_out = [], [], [], [], [], []

    for i in np.where(mask)[0]:
        start = i - lookback + 1
        if start < 0:
            continue
        if dates[i] - dates[start] != np.timedelta64(lookback - 1, "D"):
            continue

        seq = Xall[start:i + 1]
        if not np.isfinite(seq).all():
            continue

        Xs.append(seq)
        ss.append(station_id)
        ds.append(dates[i])
        fs.append(flow[i])

        if historical:
            ys.append(residual[i])
            obs_out.append(observed[i])

    if len(Xs) == 0:
        raise ValueError(
            f"No sequences for station={station_id}, split={split}, lookback={lookback}."
        )

    X = np.asarray(Xs, dtype=np.float32)
    s = np.asarray(ss, dtype=np.int32)
    d = np.asarray(ds)
    f = np.asarray(fs, dtype=np.float32)

    if historical:
        y = np.asarray(ys, dtype=np.float32)
        o = np.asarray(obs_out, dtype=np.float32)
    else:
        y, o = None, None

    return X, y, s, d, f, o


# -----------------------------------------------------------------------------
# 5) Save one historical split
# -----------------------------------------------------------------------------
def save_split(path, X, y, station, dates, flow, observed):
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        path, X=X, y=y, station=station, dates=dates,
        FLOW_OUT=flow, Observed=observed,
    )


# -----------------------------------------------------------------------------
# 6) Prepare historical
# -----------------------------------------------------------------------------
def prepare_historical(descriptors):
    print("\n" + "=" * 78)
    print("PREPARING HISTORICAL DATA")
    print("=" * 78)

    frames = []
    for station in STATIONS:
        print(f"\nProcessing station: {station}")
        df = load_sheet(HISTORICAL_XLSX, station, historical=True)
        df = engineer(df, descriptors.loc[station], historical=True)
        print(f"  Rows after engineering: {len(df):,}")
        frames.append(df)

    pooled = pd.concat(frames, ignore_index=True).copy()

    train_rows = pooled[
        (pooled["Date"] >= TRAIN_START) & (pooled["Date"] <= TRAIN_END)
    ].copy()

    if train_rows.empty:
        raise ValueError("No historical rows found in the training period.")

    print("\n" + "-" * 78)
    print("FITTING STANDARD SCALER")
    print("-" * 78)

    scaler = StandardScaler()
    scaler.fit(train_rows[FEATURES].to_numpy(dtype=np.float64))

    print(f"Scaler fit period: {TRAIN_START} to {TRAIN_END}")
    print(f"Rows used for scaler: {len(train_rows):,}")

    station_to_id = {s: i for i, s in enumerate(STATIONS)}

    metadata = {
        "feature_cols":         FEATURES,
        "n_features":           len(FEATURES),
        "station_to_id":        station_to_id,
        "stations":             STATIONS,
        "n_stations":           len(STATIONS),
        "lookback_candidates":  LOOKBACK_CANDIDATES,
        "train_period":         [TRAIN_START, TRAIN_END],
        "val_period":           [VAL_START,   VAL_END],
        "test_period":          [TEST_START,  TEST_END],
        "target":               "log1p(Observed)-log1p(FLOW_OUT)",
        "missing_value_method": "forward_fill_then_backward_fill",
        "scaler_fit_period":    [TRAIN_START, TRAIN_END],
        "version":              "hybrid_v3_extended",
        "GW_Qmm_used":          False,
        "PETmm_used":           False,
        "SWAT_STATE_vars":      SWAT_STATE,
        "ENGINEERED_vars":      ENGINEERED,
    }

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with open(OUTPUT_DIR / "scaler.pkl", "wb") as f:
        pickle.dump({"scaler": scaler, "metadata": metadata}, f)
    print(f"\nScaler and metadata saved:\n{OUTPUT_DIR / 'scaler.pkl'}")

    station_frames = {
        s: pooled[pooled["station"] == s].sort_values("Date").reset_index(drop=True)
        for s in STATIONS
    }

    for lb in LOOKBACK_CANDIDATES:
        print("\n" + "-" * 78)
        print(f"HISTORICAL LOOKBACK = {lb} DAYS")
        print("-" * 78)

        lbdir = OUTPUT_DIR / "historical" / f"lookback_{lb:03d}"
        lbdir.mkdir(parents=True, exist_ok=True)

        for split in ["train", "val", "test"]:
            parts = [
                make_sequences(
                    station_frames[station], scaler,
                    station_to_id[station], lb, split, historical=True,
                )
                for station in STATIONS
            ]

            X        = np.concatenate([p[0] for p in parts], axis=0)
            y        = np.concatenate([p[1] for p in parts], axis=0)
            s        = np.concatenate([p[2] for p in parts], axis=0)
            d        = np.concatenate([p[3] for p in parts], axis=0)
            flow     = np.concatenate([p[4] for p in parts], axis=0)
            observed = np.concatenate([p[5] for p in parts], axis=0)

            order = np.lexsort((d, s))
            X, y, s, d, flow, observed = (
                X[order], y[order], s[order], d[order], flow[order], observed[order]
            )

            if not np.isfinite(X).all():        raise ValueError(f"L={lb}, {split}: X NaN/Inf.")
            if not np.isfinite(y).all():        raise ValueError(f"L={lb}, {split}: y NaN/Inf.")
            if not np.isfinite(flow).all():     raise ValueError(f"L={lb}, {split}: FLOW_OUT NaN/Inf.")
            if not np.isfinite(observed).all(): raise ValueError(f"L={lb}, {split}: Observed NaN/Inf.")

            save_split(lbdir / f"{split}.npz", X, y, s, d, flow, observed)
            print(f"  {split:<5} | N={len(X):,} | X={X.shape}")

            del X, y, s, d, flow, observed
            gc.collect()

    print("\nHistorical preparation complete.")
    return metadata, scaler, station_frames


# -----------------------------------------------------------------------------
# 7) Prepare future
# -----------------------------------------------------------------------------
def prepare_future(descriptors, metadata, scaler):
    print("\n" + "=" * 78)
    print("PREPARING FUTURE SCENARIOS")
    print("=" * 78)

    if not isinstance(scaler, StandardScaler):
        raise TypeError("The supplied scaler is not a sklearn StandardScaler.")

    station_to_id = metadata["station_to_id"]
    rows = []

    for scenario, path in FUTURE_SCENARIOS.items():
        print(f"\nScenario: {scenario}")
        if not path.exists():
            raise FileNotFoundError(f"Future scenario file not found:\n{path}")

        for station in STATIONS:
            print(f"  Processing station: {station}")

            df = load_sheet(path, station, historical=False)
            df = engineer(df, descriptors.loc[station], historical=False)

            for lb in LOOKBACK_CANDIDATES:
                X, _, s, d, flow, _ = make_sequences(
                    df, scaler, station_to_id[station], lb,
                    split=None, historical=False,
                )

                out = OUTPUT_DIR / "future" / scenario / f"lookback_{lb:03d}"
                out.mkdir(parents=True, exist_ok=True)

                np.savez_compressed(
                    out / f"{station}.npz",
                    X=X, station=s, dates=d, FLOW_OUT=flow,
                )

                first_pred = pd.Timestamp(d[0])  if len(d) > 0 else pd.NaT
                last_pred  = pd.Timestamp(d[-1]) if len(d) > 0 else pd.NaT

                rows.append({
                    "scenario":         scenario,
                    "station":          station,
                    "lookback":         lb,
                    "first_prediction": first_pred,
                    "last_prediction":  last_pred,
                    "n_predictions":    len(d),
                })

                print(f"    L={lb:3d} | N={len(X):,} | "
                      f"{first_pred} -> {last_pred}")

                del X, s, d, flow
                gc.collect()

    periods = pd.DataFrame(rows)
    periods.to_csv(OUTPUT_DIR / "future_prediction_periods_v3.csv", index=False)
    print("\nFuture preparation complete.")
    print(f"Saved:\n{OUTPUT_DIR / 'future_prediction_periods_v3.csv'}")


# -----------------------------------------------------------------------------
# 8) Dataset summary
# -----------------------------------------------------------------------------
def print_dataset_summary(metadata, station_frames):
    print("\n" + "=" * 78)
    print("DATASET SUMMARY")
    print("=" * 78)
    print(f'Features  : {metadata["n_features"]}')
    print(f'Stations  : {metadata["n_stations"]}')
    print(f'Lookbacks : {metadata["lookback_candidates"]}')
    print(f'Train      : {TRAIN_START} to {TRAIN_END}')
    print(f'Validation : {VAL_START} to {VAL_END}')
    print(f'Test       : {TEST_START} to {TEST_END}')
    print("\nFeatures:")
    for i, f in enumerate(FEATURES, start=1):
        print(f"  {i:02d}. {f}")
    print("\nStation row counts:")
    for station in STATIONS:
        print(f"  {station}: {len(station_frames[station]):,}")
    print()
    print(f'GW_Qmm used: {metadata["GW_Qmm_used"]}')
    print(f'PETmm used : {metadata["PETmm_used"]}')
    print(f'Scaler fit period: {TRAIN_START} -> {TRAIN_END}')
    print("=" * 78)


print("Cell 2 done -- all functions defined.")

In [ ]:
# =============================================================================
# CELL 3 -- RUN HISTORICAL PREPARATION
# =============================================================================

descriptors = load_descriptors()

metadata, scaler, station_frames = prepare_historical(descriptors)

print("\nHistorical files written:")
for p in sorted(OUTPUT_DIR.glob("historical/*/*.npz"))[:6]:
    print("  ", p)

In [ ]:
# =============================================================================
# CELL 4 -- RUN FUTURE PREPARATION
# =============================================================================

prepare_future(descriptors, metadata, scaler)

In [ ]:
# =============================================================================
# CELL 5 -- SUMMARY + VERIFICATION
# =============================================================================

print_dataset_summary(metadata, station_frames)

verify_path = OUTPUT_DIR / "historical" / "lookback_045" / "train.npz"
with np.load(verify_path, allow_pickle=False) as z:
    print("\nVerification of", verify_path.name)
    print("  Keys     :", z.files)
    print("  X shape  :", z["X"].shape)
    print("  y shape  :", z["y"].shape)
    print("  station  :", z["station"].shape,
          "| unique:", np.unique(z["station"]))
    print("  dates    :", z["dates"][0], "->", z["dates"][-1])
    recon = np.expm1(np.log1p(z["FLOW_OUT"]) + z["y"])
    err   = float(np.max(np.abs(recon - z["Observed"])))
    print(f"  Reconstruction max |error|: {err:.3e}  (should be ~0)")

print("\n" + "=" * 78)
print("DATA PREPARATION COMPLETE")
print("=" * 78)
print(f"Output directory: {OUTPUT_DIR}")

# LSTM_Only

In [ ]:
# =============================================================================
# CELL 1 -- SETUP
# =============================================================================

import os
import gc
import json
import pickle
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Embedding,
    Flatten, Concatenate, LayerNormalization,
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau,
    ModelCheckpoint, CSVLogger,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

import matplotlib
matplotlib.use("Agg")   # notebook-safe; comment out if you want inline plots
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# =============================================================================
# PATHS -- LOCAL WINDOWS
# =============================================================================
DATA_DIR = PROJECT_DIR / "data"

# If you are on Colab, comment the line above and uncomment these:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)
# DATA_DIR = Path('/content/drive/MyDrive/Wateshed-Memory1/Daily')

PREPARED_DIR = RESULTS_ROOT / "LSTM_Prepared_v4"          # <-- same as data-prep Cell 1
RESULTS_DIR  = RESULTS_ROOT / "LSTM_Results_v4"
MODELS_DIR      = RESULTS_DIR / "models"
PREDICTIONS_DIR = RESULTS_DIR / "predictions"
PLOTS_DIR       = RESULTS_DIR / "plots"

for d in [RESULTS_DIR, MODELS_DIR, PREDICTIONS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =============================================================================
# RUN CONFIGURATION
# =============================================================================

# Shortlist of lookbacks that actually exist in the prepared data.
# Set to the full set if you want the full search.
LOOKBACKS = [30, 45, 60, 90, 120, 180]

# Multi-seed search. Set to [42] for a quick run.
SEEDS = [42, 123, 456]

# =============================================================================
# MODEL CONFIGURATIONS
# =============================================================================
MODEL_CONFIGS = {
    1: {"name": "LSTM64_D32",            "lstm1": 64,  "lstm2": None, "dense": 32, "dropout": 0.20, "learning_rate": 0.0010, "batch_size": 64, "l2": 1e-5},
    2: {"name": "LSTM128_D32",           "lstm1": 128, "lstm2": None, "dense": 32, "dropout": 0.20, "learning_rate": 0.0010, "batch_size": 64, "l2": 1e-5},
    3: {"name": "LSTM128_D64",           "lstm1": 128, "lstm2": None, "dense": 64, "dropout": 0.20, "learning_rate": 0.0005, "batch_size": 64, "l2": 1e-5},
    4: {"name": "LSTM64_32_D32",         "lstm1": 64,  "lstm2": 32,   "dense": 32, "dropout": 0.20, "learning_rate": 0.0010, "batch_size": 64, "l2": 1e-5},
    5: {"name": "LSTM128_64_D64",        "lstm1": 128, "lstm2": 64,   "dense": 64, "dropout": 0.15, "learning_rate": 0.0005, "batch_size": 64, "l2": 1e-5},
    6: {"name": "LSTM128_64_D64_lowLR",  "lstm1": 128, "lstm2": 64,   "dense": 64, "dropout": 0.10, "learning_rate": 0.0003, "batch_size": 64, "l2": 1e-5},
}

# =============================================================================
# TRAINING PARAMETERS
# =============================================================================
EPOCHS            = 60
PATIENCE          = 15
LR_PATIENCE       = 6
MIN_LR            = 1e-6
GRADIENT_CLIPNORM = 1.0

# =============================================================================
# LOSS PARAMETERS
# =============================================================================
RESIDUAL_LOSS_WEIGHT = 0.50
FLOW_LOSS_WEIGHT     = 0.50
HIGH_FLOW_WEIGHT     = 1.50
HIGH_FLOW_QUANTILE   = 0.90
MAX_SAMPLE_WEIGHT    = 4.0

# =============================================================================
# GPU (optional)
# =============================================================================
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU detected: {len(gpus)}")
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
else:
    print("No GPU detected. Using CPU.")

print(f"PREPARED_DIR : {PREPARED_DIR}")
print(f"RESULTS_DIR  : {RESULTS_DIR}")
print(f"Lookbacks    : {LOOKBACKS}")
print(f"Seeds        : {SEEDS}")
print(f"Configs      : {len(MODEL_CONFIGS)}")

In [ ]:
# =============================================================================
# CELL 2 -- FUNCTION DEFINITIONS
# =============================================================================

# -----------------------------------------------------------------------------
# Reproducibility
# -----------------------------------------------------------------------------
def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


# -----------------------------------------------------------------------------
# Load metadata / splits
# -----------------------------------------------------------------------------
def load_metadata():
    """Read scaler.pkl produced by the data preparation notebook."""
    scaler_path = PREPARED_DIR / "scaler.pkl"
    if not scaler_path.exists():
        raise FileNotFoundError(
            f"Cannot find:\n{scaler_path}\n\n"
            "Run the V3 data-preparation notebook first."
        )
    with open(scaler_path, "rb") as f:
        saved = pickle.load(f)
    return saved["scaler"], saved["metadata"]


def load_split(lookback, split):
    path = PREPARED_DIR / "historical" / f"lookback_{lookback:03d}" / f"{split}.npz"
    if not path.exists():
        raise FileNotFoundError(f"Cannot find:\n{path}")
    with np.load(path, allow_pickle=False) as data:
        X        = data["X"].astype(np.float32)
        y        = data["y"].astype(np.float32)
        station  = data["station"].astype(np.int32)
        dates    = data["dates"]
        flow     = data["FLOW_OUT"].astype(np.float32)
        observed = data["Observed"].astype(np.float32)
    return X, y, station, dates, flow, observed


def check_data(X, y, station, dates, flow, observed, name):
    print(f"\n{name}")
    print(f"  X:        {X.shape}")
    print(f"  y:        {y.shape}")
    print(f"  station:  {station.shape}")
    print(f"  dates:    {dates.shape}")
    print(f"  FLOW_OUT: {flow.shape}")
    print(f"  Observed: {observed.shape}")

    if not np.isfinite(X).all():        raise ValueError(f"{name}: X contains NaN/Inf.")
    if not np.isfinite(y).all():        raise ValueError(f"{name}: y contains NaN/Inf.")
    if not np.isfinite(flow).all():     raise ValueError(f"{name}: FLOW_OUT NaN/Inf.")
    if not np.isfinite(observed).all(): raise ValueError(f"{name}: Observed NaN/Inf.")
    if (flow < 0).any():                raise ValueError(f"{name}: negative FLOW_OUT.")
    if (observed < 0).any():            raise ValueError(f"{name}: negative Observed.")


# -----------------------------------------------------------------------------
# High-flow weighting
# -----------------------------------------------------------------------------
def calculate_high_flow_weights(observed_train):
    """
    Weight increases for observations above the TRAINING 90th percentile.
    Threshold is derived from training only to avoid leakage.
    """
    q90 = np.quantile(observed_train, HIGH_FLOW_QUANTILE)
    q75 = np.quantile(observed_train, 0.75)
    scale = max(q90 - q75, 1e-6)

    weights = np.ones_like(observed_train, dtype=np.float32)
    high_mask = observed_train > q90
    weights[high_mask] = 1.0 + HIGH_FLOW_WEIGHT * (
        (observed_train[high_mask] - q90) / scale
    )
    weights = np.clip(weights, 1.0, MAX_SAMPLE_WEIGHT)
    return weights.astype(np.float32), float(q90), float(q75)


# -----------------------------------------------------------------------------
# Model
# -----------------------------------------------------------------------------
def build_model(lookback, n_features, n_stations, config):

    sequence_input = Input(shape=(lookback, n_features), name="sequence_input")
    station_input  = Input(shape=(1,), dtype="int32", name="station_input")

    x = sequence_input
    if config["lstm2"] is None:
        x = LSTM(
            config["lstm1"], return_sequences=False,
            kernel_regularizer=l2(config["l2"]), name="lstm_1",
        )(x)
    else:
        x = LSTM(
            config["lstm1"], return_sequences=True,
            kernel_regularizer=l2(config["l2"]), name="lstm_1",
        )(x)
        x = Dropout(config["dropout"])(x)
        x = LSTM(
            config["lstm2"], return_sequences=False,
            kernel_regularizer=l2(config["l2"]), name="lstm_2",
        )(x)

    x = LayerNormalization(name="lstm_normalization")(x)

    embedding_dim = 4
    station_embedding = Embedding(
        input_dim=n_stations, output_dim=embedding_dim,
        name="station_embedding",
    )(station_input)
    station_embedding = Flatten(name="station_embedding_flat")(station_embedding)

    x = Concatenate(name="sequence_station_concat")([x, station_embedding])
    x = Dense(
        config["dense"], activation="relu",
        kernel_regularizer=l2(config["l2"]), name="dense_1",
    )(x)
    x = Dropout(config["dropout"])(x)

    residual_output = Dense(1, activation="linear", name="residual_output")(x)

    return Model(inputs=[sequence_input, station_input], outputs=residual_output)


# -----------------------------------------------------------------------------
# Custom hybrid loss
# -----------------------------------------------------------------------------
def make_hybrid_loss():
    """
    y_true contains four columns:
        0 = residual target
        1 = Observed discharge
        2 = SWAT FLOW_OUT
        3 = sample weight
    y_pred = predicted residual
    """
    def hybrid_loss(y_true, y_pred):

        residual_true = y_true[:, 0:1]
        observed      = y_true[:, 1:2]
        swat          = y_true[:, 2:3]
        weights       = y_true[:, 3:4]

        residual_pred = y_pred

        # Residual term (weighted MSE)
        residual_sq = tf.square(residual_pred - residual_true)
        residual_loss = tf.reduce_mean(residual_sq * weights)

        # Flow-space term (weighted Huber in log1p space)
        corrected_log = tf.math.log1p(tf.maximum(swat, 0.0)) + residual_pred
        observed_log  = tf.math.log1p(tf.maximum(observed, 0.0))
        flow_error    = corrected_log - observed_log

        abs_error = tf.abs(flow_error)
        delta     = 1.0
        quadratic = tf.minimum(abs_error, delta)
        linear    = abs_error - quadratic
        huber     = 0.5 * tf.square(quadratic) + delta * linear

        flow_loss = tf.reduce_mean(huber * weights)

        return RESIDUAL_LOSS_WEIGHT * residual_loss + FLOW_LOSS_WEIGHT * flow_loss

    return hybrid_loss


def compile_model(model, config):
    optimizer = Adam(learning_rate=config["learning_rate"], clipnorm=GRADIENT_CLIPNORM)
    model.compile(optimizer=optimizer, loss=make_hybrid_loss())
    return model


def make_loss_target(residual, observed, swat, weights):
    return np.column_stack([residual, observed, swat, weights]).astype(np.float32)


# -----------------------------------------------------------------------------
# Prediction / reconstruction
# -----------------------------------------------------------------------------
def predict_residual(model, X, station, batch_size):
    pred = model.predict(
        {"sequence_input": X, "station_input": station.reshape(-1, 1)},
        batch_size=batch_size, verbose=0,
    )
    return pred.reshape(-1)


def reconstruct_flow(swat, residual_pred):
    log_corrected = np.log1p(np.maximum(swat, 0)) + residual_pred
    log_corrected = np.clip(log_corrected, -20, 20)
    corrected = np.expm1(log_corrected)
    return np.maximum(corrected, 0).astype(np.float32)


# -----------------------------------------------------------------------------
# Metrics
# -----------------------------------------------------------------------------
def nse(observed, simulated):
    o = np.asarray(observed,  dtype=np.float64)
    s = np.asarray(simulated, dtype=np.float64)
    den = np.sum((o - o.mean()) ** 2)
    if den <= 0:
        return np.nan
    return 1.0 - np.sum((o - s) ** 2) / den


def kge(observed, simulated):
    o = np.asarray(observed,  dtype=np.float64)
    s = np.asarray(simulated, dtype=np.float64)
    if len(o) < 2:
        return np.nan
    mo, ms = np.mean(o), np.mean(s)
    so, ss = np.std(o, ddof=1), np.std(s, ddof=1)
    if so <= 0 or mo == 0:
        return np.nan
    r = np.corrcoef(o, s)[0, 1]
    alpha = ss / so
    beta  = ms / mo
    return 1.0 - np.sqrt((r - 1.0) ** 2 + (alpha - 1.0) ** 2 + (beta - 1.0) ** 2)


def pbias(observed, simulated):
    o = np.asarray(observed,  dtype=np.float64)
    s = np.asarray(simulated, dtype=np.float64)
    den = np.sum(o)
    if den == 0:
        return np.nan
    return 100.0 * np.sum(s - o) / den


def rmse(observed, simulated):
    return float(np.sqrt(np.mean((observed - simulated) ** 2)))


def mae(observed, simulated):
    return float(np.mean(np.abs(observed - simulated)))


def evaluate(observed, simulated, swat):
    return {
        "NSE":        nse(observed, simulated),
        "KGE":        kge(observed, simulated),
        "PBIAS":      pbias(observed, simulated),
        "RMSE":       rmse(observed, simulated),
        "MAE":        mae(observed, simulated),
        "SWAT_NSE":   nse(observed, swat),
        "SWAT_KGE":   kge(observed, swat),
        "SWAT_PBIAS": pbias(observed, swat),
    }


def evaluate_by_station(observed, simulated, swat, station):
    station_names = {0: "Shebe", 1: "Asendabo", 2: "Abelti"}
    rows = []
    for sid in sorted(np.unique(station)):
        mask = (station == sid)
        if mask.sum() < 2:
            continue
        o, s, w = observed[mask], simulated[mask], swat[mask]
        rows.append({
            "station":    station_names.get(int(sid), str(sid)),
            "N":          int(mask.sum()),
            "NSE":        nse(o, s),
            "KGE":        kge(o, s),
            "PBIAS":      pbias(o, s),
            "RMSE":       rmse(o, s),
            "SWAT_NSE":   nse(o, w),
            "SWAT_KGE":   kge(o, w),
            "SWAT_PBIAS": pbias(o, w),
        })
    return pd.DataFrame(rows)


def composite_score(metrics):
    nse_v   = metrics["NSE"]   if np.isfinite(metrics["NSE"])   else -10
    kge_v   = metrics["KGE"]   if np.isfinite(metrics["KGE"])   else -10
    pbias_v = metrics["PBIAS"] if np.isfinite(metrics["PBIAS"]) else 1000
    pbias_score = 1.0 - abs(pbias_v) / 100.0
    return 0.50 * nse_v + 0.30 * kge_v + 0.20 * pbias_score


# -----------------------------------------------------------------------------
# Save predictions / plots
# -----------------------------------------------------------------------------
def save_predictions(path, dates, station, observed, swat,
                     residual_true, residual_pred, corrected):
    station_names = {0: "Shebe", 1: "Asendabo", 2: "Abelti"}
    df = pd.DataFrame({
        "Date":           pd.to_datetime(dates),
        "Station_ID":     station,
        "Station":        [station_names.get(int(s), str(s)) for s in station],
        "Observed":       observed,
        "FLOW_OUT":       swat,
        "Residual_True":  residual_true,
        "Residual_Pred":  residual_pred,
        "Corrected_Flow": corrected,
    }).sort_values(["Station_ID", "Date"])
    df.to_csv(path, index=False)


def plot_test_results(df, title, output_path):
    for station in df["Station"].unique():
        sub = df[df["Station"] == station].sort_values("Date")
        plt.figure(figsize=(14, 5))
        plt.plot(sub["Date"], sub["Observed"],       label="Observed",   linewidth=1.2)
        plt.plot(sub["Date"], sub["FLOW_OUT"],       label="SWAT",       linewidth=1.0)
        plt.plot(sub["Date"], sub["Corrected_Flow"], label="V3 LSTM-SWAT", linewidth=1.2)
        plt.xlabel("Date", fontsize=12)
        plt.ylabel("Streamflow", fontsize=12)
        plt.title(f"{title} - {station}", fontsize=12)
        plt.legend()
        plt.grid(alpha=0.25)
        plt.tight_layout()
        filename = output_path.parent / f"{output_path.stem}_{station}.png"
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close()


# -----------------------------------------------------------------------------
# Train one model
# -----------------------------------------------------------------------------
def train_one_model(lookback, config_id, seed,
                    X_train, y_train, station_train, flow_train, observed_train,
                    X_val,   y_val,   station_val,   flow_val,   observed_val,
                    n_features, n_stations):

    config = MODEL_CONFIGS[config_id]
    set_seed(seed)

    model_name = f"L{lookback}_C{config_id}_S{seed}"
    print("\n" + "=" * 78)
    print(f"TRAINING {model_name}")
    print("=" * 78)

    # High-flow weights from TRAIN only
    train_weights, q90, q75 = calculate_high_flow_weights(observed_train)
    scale = max(q90 - q75, 1e-6)

    # Validation weights use TRAIN-derived thresholds (no leakage)
    val_weights = np.ones_like(observed_val, dtype=np.float32)
    val_high = observed_val > q90
    val_weights[val_high] = 1.0 + HIGH_FLOW_WEIGHT * ((observed_val[val_high] - q90) / scale)
    val_weights = np.clip(val_weights, 1.0, MAX_SAMPLE_WEIGHT).astype(np.float32)

    y_train_loss = make_loss_target(y_train, observed_train, flow_train, train_weights)
    y_val_loss   = make_loss_target(y_val,   observed_val,   flow_val,   val_weights)

    model = build_model(lookback, n_features, n_stations, config)
    model = compile_model(model, config)

    model_path = MODELS_DIR / f"{model_name}.keras"

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=PATIENCE,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                          patience=LR_PATIENCE, min_lr=MIN_LR, verbose=1),
        ModelCheckpoint(model_path, monitor="val_loss",
                        save_best_only=True, verbose=0),
    ]

    history = model.fit(
        {"sequence_input": X_train, "station_input": station_train.reshape(-1, 1)},
        y_train_loss,
        validation_data=(
            {"sequence_input": X_val, "station_input": station_val.reshape(-1, 1)},
            y_val_loss,
        ),
        epochs=EPOCHS,
        batch_size=config["batch_size"],
        shuffle=True,
        callbacks=callbacks,
        verbose=1,
    )

    residual_pred = predict_residual(model, X_val, station_val, config["batch_size"])
    corrected = reconstruct_flow(flow_val, residual_pred)

    metrics = evaluate(observed_val, corrected, flow_val)
    metrics["composite"]  = composite_score(metrics)
    metrics["lookback"]   = lookback
    metrics["config_id"]  = config_id
    metrics["config_name"] = config["name"]
    metrics["seed"]       = seed
    metrics["epochs"]     = len(history.history["loss"])
    metrics["q90_train"]  = q90

    print(f"\nValidation NSE={metrics['NSE']:.4f} | "
          f"KGE={metrics['KGE']:.4f} | "
          f"PBIAS={metrics['PBIAS']:.2f}% | "
          f"Composite={metrics['composite']:.4f} | "
          f"SWAT NSE={metrics['SWAT_NSE']:.4f}")

    del model
    tf.keras.backend.clear_session()
    gc.collect()

    return metrics


# -----------------------------------------------------------------------------
# Full search across lookbacks, configs, seeds
# -----------------------------------------------------------------------------
def run_model_search():
    print("\n" + "=" * 78)
    print("WATERSHED-MEMORY-INFORMED LSTM-SWAT HYBRID V3 -- TRAIN/VALIDATION SEARCH")
    print("=" * 78)

    scaler, metadata = load_metadata()
    feature_cols = metadata["feature_cols"]
    n_features   = len(feature_cols)
    n_stations   = metadata["n_stations"]

    print(f"\nFeatures : {n_features}")
    print(f"Stations : {n_stations}")
    print(f"Lookbacks: {LOOKBACKS}")
    print(f"Configs  : {len(MODEL_CONFIGS)}")
    print(f"Seeds    : {SEEDS}")
    print(f"Total models: {len(LOOKBACKS) * len(MODEL_CONFIGS) * len(SEEDS)}")

    results = []
    best_score = -np.inf
    best_info  = None

    for lookback in LOOKBACKS:
        print("\n" + "#" * 78)
        print(f"LOOKBACK = {lookback} DAYS")
        print("#" * 78)

        X_train, y_train, s_train, d_train, f_train, o_train = load_split(lookback, "train")
        X_val,   y_val,   s_val,   d_val,   f_val,   o_val   = load_split(lookback, "val")

        check_data(X_train, y_train, s_train, d_train, f_train, o_train,
                   f"L={lookback} TRAIN")
        check_data(X_val, y_val, s_val, d_val, f_val, o_val,
                   f"L={lookback} VALIDATION")

        for config_id in MODEL_CONFIGS:
            for seed in SEEDS:
                metrics = train_one_model(
                    lookback, config_id, seed,
                    X_train, y_train, s_train, f_train, o_train,
                    X_val,   y_val,   s_val,   f_val,   o_val,
                    n_features, n_stations,
                )
                results.append(metrics)

                if np.isfinite(metrics["composite"]) and metrics["composite"] > best_score:
                    best_score = metrics["composite"]
                    best_info  = metrics.copy()
                    print("\n" + "*" * 78)
                    print("NEW BEST MODEL")
                    print(f"  Lookback   : {lookback}")
                    print(f"  Config     : {config_id} {MODEL_CONFIGS[config_id]['name']}")
                    print(f"  Seed       : {seed}")
                    print(f"  NSE        : {metrics['NSE']:.4f}")
                    print(f"  KGE        : {metrics['KGE']:.4f}")
                    print(f"  PBIAS      : {metrics['PBIAS']:.2f}%")
                    print(f"  Composite  : {metrics['composite']:.4f}")
                    print("*" * 78)

        del (X_train, y_train, s_train, d_train, f_train, o_train,
             X_val,   y_val,   s_val,   d_val,   f_val,   o_val)
        gc.collect()

    results_df = pd.DataFrame(results).sort_values("composite", ascending=False)
    results_path = RESULTS_DIR / "validation_results.csv"
    results_df.to_csv(results_path, index=False)

    if best_info is None:
        raise RuntimeError("No valid model was produced.")

    pd.DataFrame([best_info]).to_csv(RESULTS_DIR / "best_model_summary.csv", index=False)

    print("\n" + "=" * 78)
    print("MODEL SEARCH COMPLETE")
    print("=" * 78)
    print(f"Best lookback   : {best_info['lookback']}")
    print(f"Best config     : {best_info['config_name']} (id={best_info['config_id']})")
    print(f"Best seed       : {best_info['seed']}")
    print(f"Validation NSE  : {best_info['NSE']:.4f}")
    print(f"Validation KGE  : {best_info['KGE']:.4f}")
    print(f"Validation PBIAS: {best_info['PBIAS']:.2f}%")
    print(f"Composite       : {best_info['composite']:.4f}")
    print(f"\nResults saved to:\n{results_path}")

    return results_df, best_info


# -----------------------------------------------------------------------------
# Final independent test of the selected model
# -----------------------------------------------------------------------------
def test_best_model(best_info):
    print("\n" + "=" * 78)
    print("FINAL INDEPENDENT TEST")
    print("=" * 78)

    lookback  = int(best_info["lookback"])
    config_id = int(best_info["config_id"])
    seed      = int(best_info["seed"])
    config    = MODEL_CONFIGS[config_id]

    model_name = f"L{lookback}_C{config_id}_S{seed}"
    model_path = MODELS_DIR / f"{model_name}.keras"
    if not model_path.exists():
        raise FileNotFoundError(f"Best model not found:\n{model_path}")

    X_test, y_test, s_test, d_test, f_test, o_test = load_split(lookback, "test")
    check_data(X_test, y_test, s_test, d_test, f_test, o_test, "FINAL TEST")

    set_seed(seed)
    model = tf.keras.models.load_model(
        model_path,
        custom_objects={"hybrid_loss": make_hybrid_loss()},
    )

    residual_pred = predict_residual(model, X_test, s_test, config["batch_size"])
    corrected     = reconstruct_flow(f_test, residual_pred)

    metrics = evaluate(o_test, corrected, f_test)
    metrics["composite"]   = composite_score(metrics)
    metrics["lookback"]    = lookback
    metrics["config_id"]   = config_id
    metrics["config_name"] = config["name"]
    metrics["seed"]        = seed
    metrics["period"]      = "test"

    print("\n" + "=" * 60)
    print("FINAL TEST PERFORMANCE")
    print("=" * 60)
    print(f"NSE       = {metrics['NSE']:.4f}")
    print(f"KGE       = {metrics['KGE']:.4f}")
    print(f"PBIAS     = {metrics['PBIAS']:.2f}%")
    print(f"RMSE      = {metrics['RMSE']:.4f}")
    print(f"MAE       = {metrics['MAE']:.4f}")
    print(f"Composite = {metrics['composite']:.4f}")
    print("\nSWAT baseline")
    print(f"SWAT NSE  = {metrics['SWAT_NSE']:.4f}")
    print(f"SWAT KGE  = {metrics['SWAT_KGE']:.4f}")
    print(f"SWAT PBIAS= {metrics['SWAT_PBIAS']:.2f}%")
    print("\nImprovement over SWAT")
    print(f"NSE improvement = {metrics['NSE'] - metrics['SWAT_NSE']:.4f}")
    print(f"KGE improvement = {metrics['KGE'] - metrics['SWAT_KGE']:.4f}")

    station_results = evaluate_by_station(o_test, corrected, f_test, s_test)
    print("\n" + "=" * 78)
    print("STATION-WISE TEST RESULTS")
    print("=" * 78)
    print(station_results.to_string(index=False))
    station_results.to_csv(RESULTS_DIR / "test_station_results.csv", index=False)

    prediction_path = PREDICTIONS_DIR / f"{model_name}_test_predictions.csv"
    save_predictions(
        prediction_path, d_test, s_test, o_test, f_test,
        y_test, residual_pred, corrected,
    )

    pred_df = pd.read_csv(prediction_path)
    plot_test_results(
        pred_df,
        f"V3 LSTM-SWAT (L={lookback}, {config['name']})",
        PLOTS_DIR / f"{model_name}_test",
    )

    pd.DataFrame([metrics]).to_csv(RESULTS_DIR / "test_results.csv", index=False)

    del model
    tf.keras.backend.clear_session()
    gc.collect()

    return metrics, station_results


# -----------------------------------------------------------------------------
# Save run configuration
# -----------------------------------------------------------------------------
def save_run_configuration(metadata):
    config = {
        "lookbacks":               LOOKBACKS,
        "seeds":                   SEEDS,
        "model_configs":           MODEL_CONFIGS,
        "epochs":                  EPOCHS,
        "early_stopping_patience": PATIENCE,
        "lr_patience":             LR_PATIENCE,
        "min_lr":                  MIN_LR,
        "gradient_clipnorm":       GRADIENT_CLIPNORM,
        "residual_loss_weight":    RESIDUAL_LOSS_WEIGHT,
        "flow_loss_weight":        FLOW_LOSS_WEIGHT,
        "high_flow_weight":        HIGH_FLOW_WEIGHT,
        "high_flow_quantile":      HIGH_FLOW_QUANTILE,
        "max_sample_weight":       MAX_SAMPLE_WEIGHT,
        "feature_count":           len(metadata["feature_cols"]),
        "feature_cols":            metadata["feature_cols"],
    }
    with open(RESULTS_DIR / "run_configuration.json", "w") as f:
        json.dump(config, f, indent=4, default=str)


print("Cell 2 done -- all functions defined.")

In [ ]:
# =============================================================================
# CELL 3 -- RUN PIPELINE
# =============================================================================

print("=" * 78)
print("WATERSHED-MEMORY-INFORMED LSTM-SWAT HYBRID V3")
print("=" * 78)
print(f"\nPrepared data : {PREPARED_DIR}")
print(f"Results       : {RESULTS_DIR}")

scaler, metadata = load_metadata()
save_run_configuration(metadata)

print("\n" + "=" * 78)
print(f"Number of features : {len(metadata['feature_cols'])}")
print(f"Number of stations : {metadata['n_stations']}")
print(f"Feature list       :")
for i, f in enumerate(metadata["feature_cols"], start=1):
    print(f"  {i:02d}. {f}")
print("=" * 78)

# -----------------------------------------------------------------------------
# Search
# -----------------------------------------------------------------------------
validation_results, best_info = run_model_search()

# -----------------------------------------------------------------------------
# Final test on the selected model
# -----------------------------------------------------------------------------
test_metrics, station_results = test_best_model(best_info)

# -----------------------------------------------------------------------------
# Summary
# -----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("V3 TRAINING COMPLETE")
print("=" * 78)
print(f"\nBest validation model:")
print(f"  Lookback      = {best_info['lookback']}")
print(f"  Configuration = {best_info['config_name']}")
print(f"  Seed          = {best_info['seed']}")
print(f"\nValidation:")
print(f"  NSE   = {best_info['NSE']:.4f}")
print(f"  KGE   = {best_info['KGE']:.4f}")
print(f"  PBIAS = {best_info['PBIAS']:.2f}%")
print(f"\nIndependent test:")
print(f"  NSE   = {test_metrics['NSE']:.4f}")
print(f"  KGE   = {test_metrics['KGE']:.4f}")
print(f"  PBIAS = {test_metrics['PBIAS']:.2f}%")
print(f"\nSWAT baseline on test:")
print(f"  NSE   = {test_metrics['SWAT_NSE']:.4f}")
print(f"  KGE   = {test_metrics['SWAT_KGE']:.4f}")
print(f"\nResults saved to:\n{RESULTS_DIR}")
print("\nDONE.")

In [ ]:
# Four Model

In [ ]:
# =============================================================================
# CELL 4 -- GRU, TCN, MLP, LSTM-FCN (run after Cells 1-3)
# =============================================================================
from tensorflow.keras.layers import (
    GRU, Conv1D, Add, Activation, BatchNormalization,
    GlobalAveragePooling1D, Cropping1D,
)

LSTM_RESULTS_DIR = RESULTS_DIR                 # your finished LSTM run (for the comparison table)
ARCHS            = ["gru", "tcn", "mlp", "lstm_fcn"]

# Shorter search to save time: 5 lookbacks x 6 configs x 3 seeds = 90 models per architecture
LOOKBACKS = [30, 60, 90, 120, 180]
SEEDS     = [42, 123, 456]                     # use [42] for a quick run (30 models per architecture)

# -----------------------------------------------------------------------------
# Configurations (same keys for every architecture)
#   units  : width of the first layer / LSTM units / TCN filters
#   units2 : second layer width (GRU, MLP) or None
#   blocks : number of dilated residual blocks (TCN only)
#   filters: base number of conv filters (LSTM-FCN only)
# -----------------------------------------------------------------------------
def _cfg(name, units, units2, dense, dropout, lr, **extra):
    d = {"name": name, "units": units, "units2": units2, "dense": dense,
         "dropout": dropout, "learning_rate": lr, "batch_size": 64, "l2": 1e-5}
    d.update(extra)
    return d

ARCH_CONFIGS = {
    "gru": {
        1: _cfg("GRU64_D32",           64,  None, 32, 0.20, 1e-3),
        2: _cfg("GRU128_D32",          128, None, 32, 0.20, 1e-3),
        3: _cfg("GRU128_D64",          128, None, 64, 0.20, 5e-4),
        4: _cfg("GRU64_32_D32",        64,  32,   32, 0.20, 1e-3),
        5: _cfg("GRU128_64_D64",       128, 64,   64, 0.15, 5e-4),
        6: _cfg("GRU128_64_D64_lowLR", 128, 64,   64, 0.10, 3e-4),
    },
    "mlp": {
        1: _cfg("MLP64_D32",            64,  None, 32, 0.20, 1e-3),
        2: _cfg("MLP128_D32",           128, None, 32, 0.20, 1e-3),
        3: _cfg("MLP256_D64",           256, None, 64, 0.25, 5e-4),
        4: _cfg("MLP128_64_D32",        128, 64,   32, 0.20, 1e-3),
        5: _cfg("MLP256_128_D64",       256, 128,  64, 0.25, 5e-4),
        6: _cfg("MLP256_128_D64_lowLR", 256, 128,  64, 0.20, 3e-4),
    },
    "tcn": {
        1: _cfg("TCN32_B4",       32,  None, 32, 0.20, 1e-3, blocks=4),
        2: _cfg("TCN64_B5",       64,  None, 32, 0.20, 1e-3, blocks=5),
        3: _cfg("TCN64_B6",       64,  None, 64, 0.20, 5e-4, blocks=6),
        4: _cfg("TCN128_B5",      128, None, 64, 0.20, 5e-4, blocks=5),
        5: _cfg("TCN128_B6",      128, None, 64, 0.15, 5e-4, blocks=6),
        6: _cfg("TCN128_B6_lowLR",128, None, 64, 0.10, 3e-4, blocks=6),
    },
    "lstm_fcn": {
        1: _cfg("LSTMFCN32_F32",       32,  None, 32, 0.20, 1e-3, filters=32),
        2: _cfg("LSTMFCN64_F32",       64,  None, 32, 0.20, 1e-3, filters=32),
        3: _cfg("LSTMFCN64_F64",       64,  None, 64, 0.20, 5e-4, filters=64),
        4: _cfg("LSTMFCN128_F64",      128, None, 64, 0.20, 5e-4, filters=64),
        5: _cfg("LSTMFCN128_F128",     128, None, 64, 0.15, 5e-4, filters=128),
        6: _cfg("LSTMFCN128_F128_lowLR",128,None, 64, 0.10, 3e-4, filters=128),
    },
}

KEEP_CONFIGS = [1, 3, 5]      # small, medium, large (two-layer / wider) for each architecture
ARCH_CONFIGS = {a: {k: v for k, v in c.items() if k in KEEP_CONFIGS}
                for a, c in ARCH_CONFIGS.items()}


# -----------------------------------------------------------------------------
# Backbones (one block per architecture); everything else is shared
# -----------------------------------------------------------------------------
def backbone(seq, config, lookback):
    u, u2, do = config["units"], config["units2"], config["dropout"]
    reg = l2(config["l2"])

    if ARCH == "gru":
        if u2 is None:
            x = GRU(u, kernel_regularizer=reg, name="gru_1")(seq)
        else:
            x = GRU(u, return_sequences=True, kernel_regularizer=reg, name="gru_1")(seq)
            x = Dropout(do)(x)
            x = GRU(u2, kernel_regularizer=reg, name="gru_2")(x)

    elif ARCH == "mlp":
        x = Flatten(name="mlp_flatten")(seq)
        x = Dense(u, activation="relu", kernel_regularizer=reg, name="mlp_1")(x)
        x = Dropout(do)(x)
        if u2 is not None:
            x = Dense(u2, activation="relu", kernel_regularizer=reg, name="mlp_2")(x)

    elif ARCH == "tcn":
        x = Conv1D(u, 1, padding="same", name="tcn_in")(seq)
        for b in range(config["blocks"]):
            h = Conv1D(u, 3, padding="causal", dilation_rate=2 ** b,
                       activation="relu", kernel_regularizer=reg, name=f"tcn_conv_{b}")(x)
            h = Dropout(do)(h)
            x = Add(name=f"tcn_add_{b}")([x, h])
        x = Cropping1D(cropping=(lookback - 1, 0), name="tcn_last_step")(x)   # keep last day
        x = Flatten(name="tcn_flat")(x)

    elif ARCH == "lstm_fcn":
        a = LSTM(u, kernel_regularizer=reg, name="lf_lstm")(seq)
        a = Dropout(do)(a)
        c = seq
        for k, (mult, ks) in enumerate(((1, 8), (2, 5), (1, 3))):
            c = Conv1D(config["filters"] * mult, ks, padding="same",
                       kernel_regularizer=reg, name=f"lf_conv_{k}")(c)
            c = BatchNormalization(name=f"lf_bn_{k}")(c)
            c = Activation("relu")(c)
        c = GlobalAveragePooling1D(name="lf_gap")(c)
        x = Concatenate(name="lf_concat")([a, c])

    else:
        raise ValueError(f"Unknown ARCH: {ARCH}")
    return x


def build_model(lookback, n_features, n_stations, config):
    sequence_input = Input(shape=(lookback, n_features), name="sequence_input")
    station_input  = Input(shape=(1,), dtype="int32", name="station_input")

    x = backbone(sequence_input, config, lookback)
    x = LayerNormalization(name="backbone_normalization")(x)

    emb = Embedding(input_dim=n_stations, output_dim=4, name="station_embedding")(station_input)
    emb = Flatten(name="station_embedding_flat")(emb)

    x = Concatenate(name="sequence_station_concat")([x, emb])
    x = Dense(config["dense"], activation="relu",
              kernel_regularizer=l2(config["l2"]), name="dense_1")(x)
    x = Dropout(config["dropout"])(x)
    residual_output = Dense(1, activation="linear", name="residual_output")(x)
    return Model(inputs=[sequence_input, station_input], outputs=residual_output)


# Plot label follows the architecture
def plot_test_results(df, title, output_path):
    for station in df["Station"].unique():
        sub = df[df["Station"] == station].sort_values("Date")
        plt.figure(figsize=(14, 5))
        plt.plot(sub["Date"], sub["Observed"],       label="Observed", linewidth=1.2)
        plt.plot(sub["Date"], sub["FLOW_OUT"],       label="SWAT",     linewidth=1.0)
        plt.plot(sub["Date"], sub["Corrected_Flow"], label=f"{ARCH.upper()}-SWAT", linewidth=1.2)
        plt.xlabel("Date", fontsize=12); plt.ylabel("Streamflow", fontsize=12)
        plt.title(f"{title} - {station}", fontsize=12)
        plt.legend(); plt.grid(alpha=0.25); plt.tight_layout()
        plt.savefig(output_path.parent / f"{output_path.stem}_{station}.png",
                    dpi=300, bbox_inches="tight")
        plt.close()


# -----------------------------------------------------------------------------
# Switch architecture (sets the globals that the Cell 2 functions read)
# -----------------------------------------------------------------------------
def set_arch(arch):
    global ARCH, MODEL_CONFIGS, RESULTS_DIR, MODELS_DIR, PREDICTIONS_DIR, PLOTS_DIR
    ARCH = arch
    MODEL_CONFIGS   = ARCH_CONFIGS[arch]
    RESULTS_DIR     = RESULTS_ROOT / f"{arch.upper()}_Results_v4"
    MODELS_DIR      = RESULTS_DIR / "models"
    PREDICTIONS_DIR = RESULTS_DIR / "predictions"
    PLOTS_DIR       = RESULTS_DIR / "plots"
    for d in (RESULTS_DIR, MODELS_DIR, PREDICTIONS_DIR, PLOTS_DIR):
        d.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# Run all four architectures
# -----------------------------------------------------------------------------
summary_rows = []

def _row(name, val, test):
    return {"model": name, "lookback": val["lookback"], "config": val["config_name"],
            "val_NSE": val["NSE"], "val_KGE": val["KGE"],
            "test_NSE": test["NSE"], "test_KGE": test["KGE"],
            "test_PBIAS": test["PBIAS"], "test_RMSE": test["RMSE"],
            "SWAT_test_NSE": test["SWAT_NSE"], "SWAT_test_KGE": test["SWAT_KGE"]}

for arch in ARCHS:
    set_arch(arch)
    print("\n" + "=" * 78)
    print(f"ARCHITECTURE: {arch.upper()}   ->   {RESULTS_DIR}")
    print("=" * 78)

    done = RESULTS_DIR / "test_results.csv"
    if done.exists():                                   # skip finished architectures on restart
        print("Already finished, loading saved results.")
        t = pd.read_csv(done).iloc[0].to_dict()
        v = pd.read_csv(RESULTS_DIR / "best_model_summary.csv").iloc[0].to_dict()
        summary_rows.append(_row(arch.upper() + "-SWAT", v, t))
        continue

    scaler, metadata = load_metadata()
    save_run_configuration(metadata)
    validation_results, best_info = run_model_search()
    test_metrics, station_results = test_best_model(best_info)
    summary_rows.append(_row(arch.upper() + "-SWAT", best_info, test_metrics))

# Add the finished LSTM run (Cells 1-3) to the comparison, if present
lt, lv = LSTM_RESULTS_DIR / "test_results.csv", LSTM_RESULTS_DIR / "best_model_summary.csv"
if lt.exists() and lv.exists():
    summary_rows.insert(0, _row("LSTM-SWAT",
                                pd.read_csv(lv).iloc[0].to_dict(),
                                pd.read_csv(lt).iloc[0].to_dict()))

comparison = pd.DataFrame(summary_rows)
comparison.to_csv(RESULTS_ROOT / "DL_architecture_comparison.csv", index=False)
print("\n" + "=" * 78)
print("ARCHITECTURE COMPARISON (test period 2019-2021)")
print("=" * 78)
print(comparison.round(3).to_string(index=False))

# Comparsion of the 5 DL model

In [ ]:
# =============================================================================
# CELL 5 -- COMPARE THE 5 DL MODELS ON THE TEST PERIOD (2019-2021)
# =============================================================================
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = PROJECT_DIR / "data"

# --- CHANGED: added "lstm"
ARCHS    = ["gru", "tcn", "mlp", "lstm_fcn", "lstm"]
# --- CHANGED: added lstm colour
COLORS   = {"gru": "tab:green", "tcn": "tab:red", "mlp": "tab:purple",
            "lstm_fcn": "tab:orange", "lstm": "tab:blue"}
STATIONS = ["Shebe", "Asendabo", "Abelti"]

def comp(nse_v, kge_v, pb):
    return 0.50 * nse_v + 0.30 * kge_v + 0.20 * (1.0 - abs(pb) / 100.0)

# --- CHANGED: v5 to avoid clobbering the v4 run
OUT = RESULTS_ROOT / "DL_comparison_v5"
OUT.mkdir(exist_ok=True)

# ---------------------------------------------------------------- load results
pooled_rows, station_rows, pred = [], [], {}
for a in ARCHS:
    d = RESULTS_ROOT / f"{a.upper()}_Results_v4"
    t = pd.read_csv(d / "test_results.csv").iloc[0]
    v = pd.read_csv(d / "best_model_summary.csv").iloc[0]
    s = pd.read_csv(d / "test_station_results.csv")

    # --- CHANGED: fall back to computed composite if column absent
    val_comp = v["composite"] if "composite" in v.index else comp(v["NSE"], v["KGE"], v["PBIAS"])

    pooled_rows.append({
        "model": a.upper(), "lookback": int(t["lookback"]), "config": t["config_name"],
        "val_NSE": v["NSE"], "val_KGE": v["KGE"], "val_composite": val_comp,
        "test_NSE": t["NSE"], "test_KGE": t["KGE"], "test_PBIAS": t["PBIAS"],
        "test_RMSE": t["RMSE"], "test_MAE": t["MAE"],
        "test_composite": comp(t["NSE"], t["KGE"], t["PBIAS"]),
    })
    for _, r in s.iterrows():
        station_rows.append({"model": a.upper(), "station": r["station"], "NSE": r["NSE"],
                             "KGE": r["KGE"], "PBIAS": r["PBIAS"], "RMSE": r["RMSE"]})

    name = f"L{int(t['lookback'])}_C{int(t['config_id'])}_S{int(t['seed'])}"
    pred[a] = pd.read_csv(d / "predictions" / f"{name}_test_predictions.csv", parse_dates=["Date"])

pooled  = pd.DataFrame(pooled_rows)
station = pd.DataFrame(station_rows)

# ---------------------------------------------------------------- ranking
for m in ("test_NSE", "test_KGE", "test_RMSE", "test_composite", "val_composite"):
    pooled["rank_" + m] = pooled[m].rank(ascending=(m == "test_RMSE"), method="min").astype(int)
pooled["abs_PBIAS"] = pooled["test_PBIAS"].abs()
pooled["rank_PBIAS"] = pooled["abs_PBIAS"].rank(method="min").astype(int)
pooled["mean_rank"] = pooled[["rank_test_NSE", "rank_test_KGE", "rank_test_RMSE", "rank_PBIAS"]].mean(axis=1)
pooled = pooled.sort_values("test_composite", ascending=False).reset_index(drop=True)

print("=" * 90)
print("POOLED TEST PERFORMANCE (2019-2021), best first")
print("=" * 90)
cols = ["model", "lookback", "config", "test_NSE", "test_KGE", "test_PBIAS", "test_RMSE",
        "test_composite", "mean_rank", "val_composite"]
print(pooled[cols].round(3).to_string(index=False))

print("\n" + "=" * 90)
print("STATION-WISE TEST NSE / KGE / PBIAS")
print("=" * 90)
for m in ("NSE", "KGE", "PBIAS"):
    print(f"\n{m}")
    print(station.pivot(index="model", columns="station", values=m)[STATIONS].round(3).to_string())

best_by_station = station.loc[station.groupby("station")["NSE"].idxmax()]
print("\nBest NSE per station:")
print(best_by_station[["station", "model", "NSE"]].round(3).to_string(index=False))

pooled.to_csv(OUT / "pooled_test_comparison.csv", index=False)
station.to_csv(OUT / "station_test_comparison.csv", index=False)

# ---------------------------------------------------------------- figure 1: bar charts
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, m in zip(axes, ("NSE", "KGE", "PBIAS")):
    piv = station.pivot(index="station", columns="model", values=m).reindex(STATIONS)
    pv  = pooled.set_index("model")["test_" + m]
    piv.loc["Pooled"] = pv
    piv = piv.reindex(STATIONS + ["Pooled"])
    piv[[a.upper() for a in ARCHS]].plot.bar(
        ax=ax, color=[COLORS[a] for a in ARCHS], width=0.8)
    ax.set_title(f"Test {m}"); ax.set_xlabel(""); ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=0)
    if m == "PBIAS":
        ax.axhline(0, color="k", lw=0.8); ax.set_ylabel("PBIAS (%)")
axes[0].legend(title="Model", fontsize=9)
plt.tight_layout()
plt.savefig(OUT / "metrics_comparison.png", dpi=300, bbox_inches="tight"); plt.close()

# ---------------------------------------------------------------- figure 2: hydrographs
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
for ax, st in zip(axes, STATIONS):
    obs = pred[ARCHS[0]].query("Station == @st").sort_values("Date")
    ax.plot(obs["Date"], obs["Observed"], color="k", lw=1.4, label="Observed")
    for a in ARCHS:
        sub = pred[a].query("Station == @st").sort_values("Date")
        ax.plot(sub["Date"], sub["Corrected_Flow"], color=COLORS[a], lw=0.9,
                alpha=0.85, label=a.upper())
    ax.set_title(st); ax.set_ylabel("Streamflow"); ax.grid(alpha=0.25)
axes[0].legend(ncol=len(ARCHS) + 1, fontsize=9)
plt.tight_layout()
plt.savefig(OUT / "test_hydrographs.png", dpi=300, bbox_inches="tight"); plt.close()

# ---------------------------------------------------------------- choose the model for projection
winner = pooled.iloc[0]
print("\n" + "=" * 90)
print(f"BEST ON TEST COMPOSITE : {winner['model']}  (lookback {winner['lookback']}, {winner['config']})")
print(f"BEST ON VALIDATION     : {pooled.sort_values('val_composite', ascending=False).iloc[0]['model']}")
gap = pooled["test_composite"].iloc[0] - pooled["test_composite"].iloc[1]
print(f"Margin over 2nd (test composite): {gap:.3f}")
print("=" * 90)

arch = winner["model"].lower()
d = RESULTS_ROOT / f"{arch.upper()}_Results_v4"
t = pd.read_csv(d / "test_results.csv").iloc[0]
with open(OUT / "best_architecture.json", "w") as f:
    json.dump({"architecture": arch, "results_dir": str(d), "lookback": int(t["lookback"]),
               "config_id": int(t["config_id"]), "config_name": t["config_name"],
               "seed": int(t["seed"]),
               "model_file": f"L{int(t['lookback'])}_C{int(t['config_id'])}_S{int(t['seed'])}.keras"}, f, indent=2)
print(f"Saved: {OUT}")

In [ ]:
import numpy as np, pandas as pd, pickle, tensorflow as tf
from pathlib import Path

DATA_DIR     = PROJECT_DIR / "data"
PREPARED_DIR = RESULTS_ROOT / "LSTM_Prepared_v4"
OUT          = RESULTS_ROOT / "Ablation_v4"; OUT.mkdir(exist_ok=True)
MODEL_DIRS = {"LSTM":"LSTM_Results_v4","GRU":"GRU_Results_v4","TCN":"TCN_Results_v4",
              "MLP":"MLP_Results_v4","LSTM-FCN":"LSTM_FCN_Results_v4"}
SPLIT, N_REP, SEED = "test", 5, 0          # "val" avoids using the test years

with open(PREPARED_DIR / "scaler.pkl", "rb") as f:
    FEATS = pickle.load(f)["metadata"]["feature_cols"]

GROUPS = {
 "Forcing (P, Tmax, Tmin)":      ["PRECIPmm", "Tmax", "Tmin"],
 "SWAT states":                  ["SWmm", "SURQmm", "PERCmm", "WYLDmm"],
 "SWAT flow":                    ["FLOW_OUT"],
 "Seasonality + T range":        ["doy_sin", "doy_cos", "T_range"],
 "Short precip memory (7-30 d)": ["precip_7d", "precip_30d"],
 "Long precip memory (90-365 d, EWMA)": ["precip_90d", "precip_180d", "precip_365d",
                                   "ewma_precip_10", "ewma_precip_30", "ewma_precip_90"],
 "Static descriptors":           ["lat", "lon", "elevation_m", "area_km2", "slope_pct",
                                  "forest-frac", "agri_frac", "urban_frac", "soil_index"],
}
used = sum(GROUPS.values(), [])
print("Features not in any group:", [f for f in FEATS if f not in used])
print("Group names not in the data:", [f for f in used if f not in FEATS])
IDX = {g: [FEATS.index(f) for f in fs if f in FEATS] for g, fs in GROUPS.items()}

def nse(o, s): o, s = np.asarray(o, float), np.asarray(s, float); return 1 - ((o-s)**2).sum() / ((o-o.mean())**2).sum()
def kge(o, s):
    o, s = np.asarray(o, float), np.asarray(s, float)
    r = np.corrcoef(o, s)[0, 1]
    return 1 - np.sqrt((r-1)**2 + (s.std()/o.std()-1)**2 + (s.mean()/o.mean()-1)**2)

def flow(model, X, st, swat):
    r = model.predict({"sequence_input": X, "station_input": st.reshape(-1, 1)},
                      batch_size=512, verbose=0).reshape(-1)
    return np.maximum(np.expm1(np.clip(np.log1p(np.maximum(swat, 0)) + r, -20, 20)), 0)

def permute(X, st, cols, static, rng):
    Xp, L = X.copy(), X.shape[1]
    groups = [np.arange(len(st))] if static else [np.where(st == s)[0] for s in np.unique(st)]
    for ids in groups:
        p = rng.permutation(ids)
        Xp[np.ix_(ids, np.arange(L), cols)] = X[np.ix_(p, np.arange(L), cols)]
    return Xp

rng, rows = np.random.default_rng(SEED), []
for name, d in MODEL_DIRS.items():
    b = pd.read_csv(RESULTS_ROOT / d / "best_model_summary.csv").iloc[0]
    lb = int(b["lookback"]); mn = f"L{lb}_C{int(b['config_id'])}_S{int(b['seed'])}"
    model = tf.keras.models.load_model(RESULTS_ROOT / d / "models" / f"{mn}.keras", compile=False)
    z = np.load(PREPARED_DIR / "historical" / f"lookback_{lb:03d}" / f"{SPLIT}.npz", allow_pickle=False)
    X, st, swat, obs = z["X"].astype(np.float32), z["station"].astype(np.int32), z["FLOW_OUT"], z["Observed"]
    base = flow(model, X, st, swat); bN, bK = nse(obs, base), kge(obs, base)
    print(f"{name} ({mn}): baseline NSE {bN:.3f}  KGE {bK:.3f}")
    for g, cols in IDX.items():
        dN, dK = [], []
        for _ in range(N_REP):
            q = flow(model, permute(X, st, cols, g == "Static descriptors", rng), st, swat)
            dN.append(nse(obs, q) - bN); dK.append(kge(obs, q) - bK)
        rows.append({"Model": name, "Group": g, "dNSE": np.mean(dN), "dNSE_sd": np.std(dN),
                     "dKGE": np.mean(dK), "dKGE_sd": np.std(dK)})
    tf.keras.backend.clear_session()

res = pd.DataFrame(rows); res.to_csv(OUT / "permutation_reliance.csv", index=False)
for m in ("dNSE", "dKGE"):
    print(f"\nChange in {m[1:]} after permuting each group ({SPLIT} period); more negative = model relies more on it")
    print(res.pivot(index="Group", columns="Model", values=m).reindex(list(GROUPS)).round(3).to_string())

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon

DATA_DIR = PROJECT_DIR / "data"
dirs = {"LSTM":"LSTM_Results_v4","GRU":"GRU_Results_v4","TCN":"TCN_Results_v4",
        "MLP":"MLP_Results_v4","LSTM-FCN":"LSTM_FCN_Results_v4"}
SHARED_LB, SHARED_CONFIG = [30, 60, 90, 120, 180], [1, 3, 5]

rows = []
for name, d in dirs.items():
    v = pd.read_csv(RESULTS_ROOT / d / "validation_results.csv")
    v = v[v["lookback"].isin(SHARED_LB) & v["config_id"].isin(SHARED_CONFIG)]
    for metric in ("NSE", "KGE"):
        piv = v.pivot_table(index=["config_id", "seed"], columns="lookback", values=metric)
        for lb in [60, 90, 120, 180]:
            diff = (piv[lb] - piv[30]).dropna()          # gain relative to the 30-day window
            p = wilcoxon(diff).pvalue if (diff != 0).any() else np.nan
            rows.append({"Model": name, "Metric": metric, "Lookback vs 30 d": lb,
                         "Mean gain": diff.mean(), "SD": diff.std(),
                         "Pairs better": int((diff > 0).sum()), "n pairs": len(diff),
                         "Wilcoxon p": p})

out = pd.DataFrame(rows)
out.to_csv(RESULTS_ROOT / "lookback_paired_tests.csv", index=False)
for metric in ("NSE", "KGE"):
    print(f"\n{metric}: gain relative to the 30-day lookback (paired by configuration and seed)")
    print(out[out.Metric == metric].drop(columns="Metric").round(3).to_string(index=False))

# Figure Generation

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
plt.rcParams.update({
    "font.family":       "serif",
    "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size":         12,
    "axes.titlesize":    12,
    "axes.labelsize":    12,
    "xtick.labelsize":   12,
    "ytick.labelsize":   12,
    "legend.fontsize":   11,
    "axes.linewidth":    0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size":  3,
    "ytick.major.size":  3,
    "figure.dpi":        300,
    "savefig.dpi":       600,
})

DATA_DIR = PROJECT_DIR / "data"
res = pd.read_csv(RESULTS_ROOT / "Ablation_v4" / "permutation_reliance.csv")

models = ["LSTM", "GRU", "TCN", "MLP", "LSTM-FCN"]
groups = ["SWAT flow", "SWAT states", "Forcing (P, Tmax, Tmin)", "Seasonality + T range",
          "Short precip memory (7-30 d)", "Long precip memory (90-365 d, EWMA)", "Static descriptors"]
VLIM = 1.0     # colour scale is clipped to +/-1 so one large value (TCN, SWAT flow) does not wash out the rest

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for ax, (m, title) in zip(axes, [("dNSE", "(a) Change in NSE"), ("dKGE", "(b) Change in KGE")]):
    val = res.pivot(index="Group", columns="Model", values=m).reindex(index=groups, columns=models)
    sd  = res.pivot(index="Group", columns="Model", values=m + "_sd").reindex(index=groups, columns=models)
    im = ax.imshow(val.values, cmap="RdBu", vmin=-VLIM, vmax=VLIM, aspect="auto")
    for i in range(val.shape[0]):
        for j in range(val.shape[1]):
            v, s = val.values[i, j], sd.values[i, j]
            noisy = abs(v) <= s                       # change smaller than the repeat SD
            ax.text(j, i, f"{v:+.2f}", ha="center", va="center", fontsize=9,
                    color="0.45" if noisy else "black", style="italic" if noisy else "normal",
                    weight="normal" if noisy else "bold")
    ax.set_xticks(range(len(models))); ax.set_xticklabels(models)
    ax.set_yticks(range(len(groups))); ax.set_yticklabels(groups)
    ax.set_title(title, loc="left")
    ax.set_xticks(np.arange(-.5, len(models), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(groups), 1), minor=True)
    ax.grid(which="minor", color="white", lw=1.5); ax.tick_params(which="minor", length=0)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
    cb.set_label("Change after permutation (red = skill falls)")


plt.tight_layout()
plt.savefig(RESULTS_ROOT / "Ablation_v4" / "permutation_reliance_heatmap.png", dpi=600, bbox_inches="tight")
plt.show()

# Validation NSE for Lookback comparsion

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = PROJECT_DIR / "data"
dirs = {"LSTM":"LSTM_Results_v4","GRU":"GRU_Results_v4","TCN":"TCN_Results_v4",
        "MLP":"MLP_Results_v4","LSTM-FCN":"LSTM_FCN_Results_v4"}

# Fair comparison: LSTM searched 6 lookbacks x 6 configs, the others 5 x 3.
# Restrict to what all five share (set to False to use everything).
SHARED_ONLY   = True
SHARED_LB     = [30, 60, 90, 120, 180]
SHARED_CONFIG = [1, 3, 5]

metrics = [("NSE", "Validation NSE"), ("KGE", "Validation KGE")]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True)

for ax, (col, ylabel) in zip(axes, metrics):
    for name, d in dirs.items():
        v = pd.read_csv(RESULTS_ROOT / d / "validation_results.csv")
        if SHARED_ONLY:
            v = v[v["lookback"].isin(SHARED_LB) & v["config_id"].isin(SHARED_CONFIG)]
        g = v.groupby("lookback")[col].agg(["mean", "std"])
        ax.errorbar(g.index, g["mean"], yerr=g["std"], marker="o",
                    capsize=3, label=name)
    ax.set_xlabel("Lookback (days)")
    ax.set_ylabel(ylabel)
    ax.set_xticks(SHARED_LB if SHARED_ONLY else sorted(v["lookback"].unique()))
    ax.grid(alpha=.3)

axes[0].set_title("(a) NSE", loc="left")
axes[1].set_title("(b) KGE", loc="left")
axes[0].legend(frameon=False)
plt.tight_layout()
plt.savefig(RESULTS_ROOT / "lookback_sensitivity_NSE_KGE.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = PROJECT_DIR / "data"
dirs = {"LSTM":"LSTM_Results_v4","GRU":"GRU_Results_v4","TCN":"TCN_Results_v4",
        "MLP":"MLP_Results_v4","LSTM-FCN":"LSTM_FCN_Results_v4"}

# Fair comparison: LSTM searched 6 lookbacks x 6 configs, the others 5 x 3.
SHARED_ONLY   = True
SHARED_LB     = [30, 60, 90, 120, 180]
SHARED_CONFIG = [1, 3, 5]
DECIMALS      = 3

def fmt(m, s):
    return f"{m:.{DECIMALS}f} ± {s:.{DECIMALS}f}"

tables, counts = {}, {}
for col in ("NSE", "KGE"):
    cols = {}
    for name, d in dirs.items():
        v = pd.read_csv(RESULTS_ROOT / d / "validation_results.csv")
        if SHARED_ONLY:
            v = v[v["lookback"].isin(SHARED_LB) & v["config_id"].isin(SHARED_CONFIG)]
        g = v.groupby("lookback")[col].agg(["mean", "std", "count"])
        cols[name] = g.apply(lambda r: fmt(r["mean"], r["std"]), axis=1)
        counts[name] = g["count"]
    t = pd.DataFrame(cols)
    t.index.name = "Lookback (days)"
    tables[col] = t

# models per cell (so you can state it in the table note)
n = pd.DataFrame(counts); n.index.name = "Lookback (days)"

for col, t in tables.items():
    print(f"\nValidation {col}: mean ± std over configurations and seeds")
    print(t.to_string())
print("\nNumber of models behind each cell:")
print(n.to_string())

# Save: one CSV per metric and one Excel file with both
for col, t in tables.items():
    t.to_csv(RESULTS_ROOT / f"lookback_sensitivity_{col}.csv")
with pd.ExcelWriter(RESULTS_ROOT / "lookback_sensitivity_tables.xlsx") as xw:
    for col, t in tables.items():
        t.to_excel(xw, sheet_name=col)
    n.to_excel(xw, sheet_name="n_models")
print("\nSaved CSV and Excel files to", RESULTS_ROOT)

# Future_Predicion

In [ ]:

# =============================================================================
import numpy as np, pandas as pd, tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR     = PROJECT_DIR / "data"
PREPARED_DIR = RESULTS_ROOT / "LSTM_Prepared_v4"
OUT          = RESULTS_ROOT / "Projection_v4"; OUT.mkdir(exist_ok=True)

MODEL_DIRS = {                       # model name -> results folder
    "LSTM":     RESULTS_ROOT / "LSTM_Results_v4",
    "GRU":      RESULTS_ROOT / "GRU_Results_v4",
    "TCN":      RESULTS_ROOT / "TCN_Results_v4",
    "MLP":      RESULTS_ROOT / "MLP_Results_v4",
    "LSTM_FCN": RESULTS_ROOT / "LSTM_FCN_Results_v4",
}
SCENARIOS = ["SSP245", "SSP585"]
STATIONS  = {0: "Shebe", 1: "Asendabo", 2: "Abelti"}
LOG_CLIP  = 20

# Future periods: leave as None to split the future record into three equal
# blocks (near / mid / far), or set e.g. {"Near": ("2021","2040"), ...}
PERIODS = {"Near": (2041, 2070), "Far": (2071,2100)}

# ------------------------------------------------------------------ helpers
def find_future_files(scenario, lookback):
    tag = f"lookback_{lookback:03d}"
    cands = [p for p in PREPARED_DIR.rglob("*.npz")
             if scenario.lower() in str(p).lower() and tag in str(p)]
    if not cands:
        print(f"\nNo file found for {scenario}, {tag}. Files under {PREPARED_DIR}:")
        for p in sorted(PREPARED_DIR.rglob("*")):
            if p.is_file(): print("  ", p.relative_to(PREPARED_DIR))
        raise FileNotFoundError(f"{scenario} / {tag}")
    return sorted(cands)

def load_future(scenario, lookback):
    parts = [load_npz(p) for p in find_future_files(scenario, lookback)]
    out = {k: np.concatenate([d[k] for d in parts]) for k in ("X", "station", "swat")}
    out["Date"] = pd.DatetimeIndex(np.concatenate([d["Date"].values for d in parts]))
    ids = sorted(set(out["station"].tolist()))
    print(f"   {scenario} L{lookback}: {len(parts)} file(s), stations {ids}, rows {len(out['station']):,}")
    assert ids == [0, 1, 2], f"expected station ids [0, 1, 2], got {ids}"
    return out

def load_npz(path):
    with np.load(path, allow_pickle=False) as z:
        date_key = "dates" if "dates" in z.files else ("Date" if "Date" in z.files else None)
        need = ["X", "station", "FLOW_OUT"]
        miss = [k for k in need if k not in z.files] + ([] if date_key else ["dates"])
        if miss: raise KeyError(f"{path.name} lacks {miss}; has {z.files}")
        return {"X": z["X"].astype(np.float32), "station": z["station"].astype(np.int32),
                "Date": pd.to_datetime(z[date_key]), "swat": z["FLOW_OUT"].astype(np.float32)}

def load_historical(lookback):
    parts = []
    for s in ("train", "val", "test"):
        p = PREPARED_DIR / "historical" / f"lookback_{lookback:03d}" / f"{s}.npz"
        parts.append(load_npz(p))
    out = {k: np.concatenate([d[k] for d in parts]) for k in ("X", "station", "swat")}
    out["Date"] = pd.DatetimeIndex(np.concatenate([d["Date"].values for d in parts]))
    return out

def predict_flow(model, d):
    r = model.predict({"sequence_input": d["X"],
                       "station_input": d["station"].reshape(-1, 1)},
                      batch_size=512, verbose=0).reshape(-1)
    lg = np.clip(np.log1p(np.maximum(d["swat"], 0)) + r, -LOG_CLIP, LOG_CLIP)
    return np.maximum(np.expm1(lg), 0).astype(np.float32)

def stats(q):
    q = np.asarray(q, float)
    return {"mean": q.mean(), "Q5_high": np.quantile(q, 0.95),
            "Q95_low": np.quantile(q, 0.05), "max": q.max()}

# ------------------------------------------------------------------ run
daily, rows = [], []
for name, folder in MODEL_DIRS.items():
    b = pd.read_csv(folder / "best_model_summary.csv").iloc[0]
    lb = int(b["lookback"])
    mname = f"L{lb}_C{int(b['config_id'])}_S{int(b['seed'])}"
    model = tf.keras.models.load_model(folder / "models" / f"{mname}.keras", compile=False)
    print(f"\n{name}: {mname}")

    hist = load_historical(lb)
    sets = {"Historical": hist}
    for sc in SCENARIOS:
        sets[sc] = load_future(sc, lb)

    for key, d in sets.items():
        q = predict_flow(model, d)
        df = pd.DataFrame({"Date": d["Date"], "Station": [STATIONS[int(s)] for s in d["station"]],
                           "Scenario": key, "Model": name, "SWAT": d["swat"], "Hybrid": q})
        daily.append(df)
    del model; tf.keras.backend.clear_session()

daily = pd.concat(daily, ignore_index=True)
daily.to_csv(OUT / "daily_projections_all_models.csv", index=False)

# ------------------------------------------------------------------ periods
fut = daily[daily["Scenario"] != "Historical"]
y0, y1 = fut["Date"].dt.year.min(), fut["Date"].dt.year.max()
print(f"\nFuture record: {y0}-{y1}")
if PERIODS is None:
    edges = np.linspace(y0, y1 + 1, 4).astype(int)
    PERIODS = {"Near": (edges[0], edges[1] - 1), "Mid": (edges[1], edges[2] - 1),
               "Far": (edges[2], y1)}
print("Periods:", PERIODS)

# ------------------------------------------------------------------ change table
for st in STATIONS.values():
    for m in list(MODEL_DIRS) + ["SWAT"]:
        col  = "SWAT" if m == "SWAT" else "Hybrid"
        base = daily[(daily.Station == st) & (daily.Scenario == "Historical") &
                     ((daily.Model == m) if m != "SWAT" else (daily.Model == "GRU"))][col]
        b = stats(base)
        for sc in SCENARIOS:
            for per, (a, z) in PERIODS.items():
                sub = daily[(daily.Station == st) & (daily.Scenario == sc) &
                            ((daily.Model == m) if m != "SWAT" else (daily.Model == "GRU")) &
                            (daily.Date.dt.year.between(a, z))][col]
                if len(sub) == 0: continue
                s = stats(sub)
                rows.append({"Station": st, "Model": m, "Scenario": sc, "Period": per,
                             **{f"{k}_change_%": 100 * (s[k] - b[k]) / b[k] for k in s}})
chg = pd.DataFrame(rows)
chg.to_csv(OUT / "percent_change_by_model.csv", index=False)

# ------------------------------------------------------------------ spread: models vs scenarios
dl = chg[chg.Model != "SWAT"]
spread = (dl.groupby(["Station", "Scenario", "Period"])["mean_change_%"]
            .agg(model_min="min", model_max="max", model_mean="mean").reset_index())
spread["model_range"] = spread.model_max - spread.model_min
scen = (dl.groupby(["Station", "Period", "Scenario"])["mean_change_%"].mean()
          .unstack("Scenario"))
scen["scenario_diff"] = (scen[SCENARIOS[1]] - scen[SCENARIOS[0]]).abs()
spread = spread.merge(scen["scenario_diff"].reset_index(), on=["Station", "Period"])
spread.to_csv(OUT / "model_vs_scenario_spread.csv", index=False)
print("\nMEAN-FLOW CHANGE (%): range across models vs difference between scenarios")
print(spread.round(1).to_string(index=False))

# ------------------------------------------------------------------ figure
fig, axes = plt.subplots(len(SCENARIOS), 3, figsize=(15, 8), sharey=True)
mods = list(MODEL_DIRS) + ["SWAT"]
for i, sc in enumerate(SCENARIOS):
    for j, st in enumerate(STATIONS.values()):
        ax = axes[i, j]
        p = (chg[(chg.Scenario == sc) & (chg.Station == st)]
             .pivot(index="Period", columns="Model", values="mean_change_%")
             .reindex(list(PERIODS))[mods])
        p.plot.bar(ax=ax, width=0.85, legend=(i == 0 and j == 0))
        ax.axhline(0, color="k", lw=0.8); ax.set_title(f"{st} - {sc}")
        ax.set_xlabel(""); ax.tick_params(axis="x", rotation=0)
        if j == 0: ax.set_ylabel("Change in mean flow (%)")
plt.tight_layout(); plt.savefig(OUT / "mean_flow_change.png", dpi=300, bbox_inches="tight")
plt.close(); print(f"\nSaved to {OUT}")

In [ ]:
d2 = daily.copy()
d2["Per"] = np.where(d2.Scenario == "Historical", "Hist",
                     np.where(d2.Date.dt.year <= 2070, "Near", "Far"))
s = d2.groupby(["Model", "Station", "Scenario", "Per"])[["Hybrid", "SWAT"]].mean().reset_index()
s["ratio"] = s.Hybrid / s.SWAT

hist = s[s.Scenario == "Historical"][["Model", "Station", "ratio"]].rename(columns={"ratio": "hist_ratio"})
fut = s[s.Scenario != "Historical"].merge(hist, on=["Model", "Station"])
fut["drift"] = fut.ratio / fut.hist_ratio

tab = fut.pivot_table(index=["Station", "Scenario", "Per"], columns="Model", values="drift")
tab["range"] = tab.max(axis=1) - tab.min(axis=1)
print(tab.round(2).to_string())
tab.round(3).to_csv(OUT / "correction_drift.csv")

In [ ]:
# =============================================================================
# FIGURE: SSP245 mean-flow change, hybrid models only
# Needs percent_change_by_model.csv from Cell 6.
# =============================================================================
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

OUT       = RESULTS_ROOT / "Projection_v4"
SCENARIO  = "SSP245"
STATIONS  = ["Shebe", "Asendabo", "Abelti"]
PERIODS   = ["Near", "Far"]
PERIOD_LABELS = {"Near": "Near (2041-2070)", "Far": "Far (2071-2100)"}
MODELS    = ["LSTM", "GRU", "TCN", "MLP", "LSTM_FCN"]
COLORS    = {"LSTM": "#1f77b4", "GRU": "#ff7f0e", "TCN": "#2ca02c",
             "MLP": "#9467bd", "LSTM_FCN": "#8c564b"}

chg = pd.read_csv(OUT / "percent_change_by_model.csv")
chg = chg[(chg.Scenario == SCENARIO) & (chg.Model.isin(MODELS))]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
n = len(MODELS) + 1                      # models + ensemble mean
w = 0.8 / n
for ax, st in zip(axes, STATIONS):
    t = (chg[chg.Station == st]
         .pivot(index="Period", columns="Model", values="mean_change_%")
         .reindex(PERIODS)[MODELS])
    ens = t.mean(axis=1)
    x = np.arange(len(PERIODS))
    for i, m in enumerate(MODELS):
        ax.bar(x + (i - n / 2 + 0.5) * w, t[m].values, w, color=COLORS[m],
               label=m if st == STATIONS[0] else None)
    ax.bar(x + (len(MODELS) - n / 2 + 0.5) * w, ens.values, w, color="black",
           label="Ensemble mean" if st == STATIONS[0] else None)
    for xi, v in zip(x, ens.values):                 # label the ensemble mean
        ax.text(xi + (len(MODELS) - n / 2 + 0.5) * w, v + (4 if v >= 0 else -4),
                f"{v:.0f}%", ha="center", va="bottom" if v >= 0 else "top", fontsize=8)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels([PERIOD_LABELS[p] for p in PERIODS])
    ax.set_title(f"{st} - {SCENARIO}")
    ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("Change in mean flow vs historical (%)")
axes[0].legend(fontsize=8, ncol=2, loc="upper left")
plt.tight_layout()
plt.savefig(OUT / f"mean_flow_change_{SCENARIO}_hybrid.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved to {OUT / f'mean_flow_change_{SCENARIO}_hybrid.png'}")

In [ ]:
cols = [
    "PRECIPmm", "Tmax", "Tmin", "SWmm",
    "SURQmm", "PERCmm", "WYLDmm", "FLOW_OUT"
]

res = []

for f in ["Historical", "SSP245", "SSP585"]:
    xl = pd.ExcelFile(DATA_DIR / f"{f}.xlsx")

    for st in STATIONS:
        df = xl.parse(st)

        # Find date column
        dc = [c for c in df.columns if "date" in str(c).lower()][0]

        # Convert to datetime
        df[dc] = pd.to_datetime(df[dc])

        # Define period
        df["Per"] = (
            "Hist"
            if f == "Historical"
            else np.where(df[dc].dt.year <= 2070, "Near", "Far")
        )

        # Select available variables
        use = [c for c in cols if c in df.columns]

        # Calculate mean
        m = df.groupby("Per")[use].mean()

        # Add scenario and station
        m["Scenario"] = f
        m["Station"] = st

        res.append(m.reset_index())

# Combine all results
out = pd.concat(res, ignore_index=True)

# Convert daily precipitation to approximate annual precipitation
out["Annual_P_mm"] = out["PRECIPmm"] * 365

print(out.round(1).to_string(index=False))

# Daily graph all model

In [ ]:
# =============================================================================
#  DAILY FLOW PLOTS: Historical + SSP245 + SSP585
#           Observed, SWAT and the five hybrid models (run after Cell 6)
# =============================================================================
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

DATA_DIR     = PROJECT_DIR / "data"
PREPARED_DIR = RESULTS_ROOT / "LSTM_Prepared_v4"
OUT          = RESULTS_ROOT / "Projection_v4"
PLOT_DIR     = OUT / "daily_flow_plots"; PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- options
SMOOTH_DAYS = 1        # 1 = raw daily flow; 30 = 30-day rolling mean (much easier to read for 30-yr records)
LOG_Y       = False    # True = log scale (shows low flows better)
MODELS      = ["LSTM", "GRU", "TCN", "MLP", "LSTM_FCN"]
STATIONS    = ["Shebe", "Asendabo", "Abelti"]
SCENARIOS   = ["Historical", "SSP245", "SSP585"]

COLORS = {"Observed": "black", "SWAT": "#7f7f7f",
          "LSTM": "#1f77b4", "GRU": "#ff7f0e", "TCN": "#2ca02c",
          "MLP": "#d62728", "LSTM_FCN": "#9467bd"}
LW     = {"Observed": 1.0, "SWAT": 0.9}

# ---------------------------------------------------------------- load data
daily = pd.read_csv(OUT / "daily_projections_all_models.csv", parse_dates=["Date"])

# Observed flow comes from the prepared historical files (any lookback; 30 keeps the most rows)
STATION_IDS = {0: "Shebe", 1: "Asendabo", 2: "Abelti"}
obs_parts = []
for split in ("train", "val", "test"):
    with np.load(PREPARED_DIR / "historical" / "lookback_030" / f"{split}.npz",
                 allow_pickle=False) as z:
        obs_parts.append(pd.DataFrame({
            "Date":     pd.to_datetime(z["dates"]),
            "Station":  [STATION_IDS[int(s)] for s in z["station"]],
            "Observed": z["Observed"].astype(float)}))
obs = pd.concat(obs_parts, ignore_index=True)

def series(station, scenario):
    """Return a DataFrame indexed by Date with columns Observed (hist only), SWAT and each model."""
    sub = daily[(daily.Station == station) & (daily.Scenario == scenario)]
    # SWAT is identical across models for the same date -> take the mean per date
    out = sub.groupby("Date")["SWAT"].mean().to_frame("SWAT")
    for m in MODELS:
        s = sub[sub.Model == m].set_index("Date")["Hybrid"]
        out[m] = s
    if scenario == "Historical":
        o = obs[obs.Station == station].set_index("Date")["Observed"]
        out["Observed"] = o
    out = out.sort_index()
    if SMOOTH_DAYS > 1:
        out = out.rolling(SMOOTH_DAYS, min_periods=max(1, SMOOTH_DAYS // 2)).mean()
    return out

# ---------------------------------------------------------------- figure 1:
# one figure per station, three stacked panels (Historical / SSP245 / SSP585)
ylab = "Streamflow" + (f" ({SMOOTH_DAYS}-day mean)" if SMOOTH_DAYS > 1 else " (daily)")
tag  = f"smooth{SMOOTH_DAYS}" + ("_log" if LOG_Y else "")

for st in STATIONS:
    fig, axes = plt.subplots(len(SCENARIOS), 1, figsize=(16, 11))
    for ax, sc in zip(axes, SCENARIOS):
        df = series(st, sc)
        order = (["Observed"] if sc == "Historical" else []) + ["SWAT"] + MODELS
        for c in order:
            if c in df and df[c].notna().any():
                ax.plot(df.index, df[c], label=c, color=COLORS[c],
                        lw=LW.get(c, 0.7), alpha=0.9 if c in ("Observed", "SWAT") else 0.75)
        ax.set_title(f"{st} - {sc}", fontsize=12, loc="left")
        ax.set_ylabel(ylab)
        if LOG_Y: ax.set_yscale("log")
        ax.grid(alpha=0.25)
        ax.xaxis.set_major_locator(mdates.YearLocator(5 if sc != "Historical" else 2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    axes[0].legend(ncol=7, loc="upper center", bbox_to_anchor=(0.5, 1.28), frameon=False)
    plt.tight_layout()
    fig.savefig(PLOT_DIR / f"daily_flow_{st}_{tag}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

# ---------------------------------------------------------------- figure 2:
# one figure per station x scenario, one panel per series (easier to compare shapes)
for st in STATIONS:
    for sc in SCENARIOS:
        df = series(st, sc)
        cols = (["Observed"] if sc == "Historical" else []) + ["SWAT"] + MODELS
        cols = [c for c in cols if c in df and df[c].notna().any()]
        fig, axes = plt.subplots(len(cols), 1, figsize=(16, 2.0 * len(cols)), sharex=True, sharey=True)
        for ax, c in zip(np.atleast_1d(axes), cols):
            ax.plot(df.index, df[c], color=COLORS[c], lw=0.7)
            ax.set_ylabel(c, rotation=0, ha="right", va="center")
            if LOG_Y: ax.set_yscale("log")
            ax.grid(alpha=0.25)
        np.atleast_1d(axes)[0].set_title(f"{st} - {sc}  ({ylab})", loc="left")
        plt.tight_layout()
        fig.savefig(PLOT_DIR / f"daily_flow_panels_{st}_{sc}_{tag}.png", dpi=300, bbox_inches="tight")
        plt.close(fig)

print(f"Saved plots to {PLOT_DIR}")

In [ ]:
# =============================================================================
# DIAGNOSTICS: is the projected change SWAT or the DL correction?
# Run after Cell 6. Uses saved CSVs; only part D re-reads the .npz inputs.
# =============================================================================
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR     = PROJECT_DIR / "data"
PREPARED_DIR = RESULTS_ROOT / "LSTM_Prepared_v4"
OUT          = RESULTS_ROOT / "Projection_v4"
SCENARIOS    = ["SSP245", "SSP585"]
PERIODS      = {"Near": (2041, 2070), "Far": (2071, 2100)}
STATIONS     = ["Shebe", "Asendabo", "Abelti"]
CHECK_LOOKBACK = 30          # lookback used for the input-shift check (part D)

chg   = pd.read_csv(OUT / "percent_change_by_model.csv")
daily = pd.read_csv(OUT / "daily_projections_all_models.csv", parse_dates=["Date"])
DL    = [m for m in daily.Model.unique()]

# -----------------------------------------------------------------------------
# A. SWAT-only change vs hybrid (DL ensemble) change
# -----------------------------------------------------------------------------
print("\n=== A. SWAT-only vs hybrid change in MEAN flow (%) ===")
sw  = chg[chg.Model == "SWAT"].set_index(["Station", "Scenario", "Period"])["mean_change_%"]
hy  = chg[chg.Model != "SWAT"].groupby(["Station", "Scenario", "Period"])["mean_change_%"]
tabA = pd.DataFrame({"SWAT": sw, "DL_mean": hy.mean(), "DL_min": hy.min(), "DL_max": hy.max()})
tabA["DL_minus_SWAT"] = tabA.DL_mean - tabA.SWAT
print(tabA.round(1).to_string())
tabA.round(2).to_csv(OUT / "diag_A_swat_vs_hybrid.csv")

print("\n=== A2. Same comparison for the tails (DL ensemble mean) ===")
for stat in ["Q95_low_change_%", "Q5_high_change_%", "max_change_%"]:
    s = chg[chg.Model == "SWAT"].set_index(["Station", "Scenario", "Period"])[stat]
    h = chg[chg.Model != "SWAT"].groupby(["Station", "Scenario", "Period"])[stat].mean()
    print(f"\n{stat}")
    print(pd.DataFrame({"SWAT": s, "DL_mean": h}).round(1).to_string())

# -----------------------------------------------------------------------------
# B. Residual drift: what the DL layer adds, historical vs future (log space)
#    residual = log1p(Hybrid) - log1p(SWAT)
# -----------------------------------------------------------------------------
daily["resid"] = np.log1p(daily.Hybrid.clip(lower=0)) - np.log1p(daily.SWAT.clip(lower=0))
daily["Year"]  = daily.Date.dt.year

def label_period(y):
    for k, (a, z) in PERIODS.items():
        if a <= y <= z: return k
    return None
daily["Period"] = daily.Year.map(label_period)

hist_res = (daily[daily.Scenario == "Historical"]
            .groupby(["Station", "Model"]).resid.mean().rename("resid_hist"))
fut_res  = (daily[daily.Scenario != "Historical"].dropna(subset=["Period"])
            .groupby(["Station", "Model", "Scenario", "Period"]).resid.mean().rename("resid_fut"))
tabB = fut_res.reset_index().merge(hist_res.reset_index(), on=["Station", "Model"])
tabB["drift"] = tabB.resid_fut - tabB.resid_hist
tabB["approx_%_flow_effect"] = 100 * (np.exp(tabB.drift) - 1)

print("\n=== B. Mean log-residual: historical vs future (drift = future - hist) ===")
print("   A drift of -0.5 means the DL layer alone cuts flow by ~39% relative to SWAT.")
print(tabB.pivot_table(index=["Station", "Model"], columns=["Scenario", "Period"],
                       values="drift").round(3).to_string())
tabB.to_csv(OUT / "diag_B_residual_drift.csv", index=False)

print("\n=== Mean historical SWAT flow per station (context for low-flow amplification) ===")
print(daily[(daily.Scenario == "Historical") & (daily.Model == DL[0])]
      .groupby("Station").SWAT.agg(["mean", "median", "min", "max"]).round(3).to_string())

# -----------------------------------------------------------------------------
# C. Like-for-like dates: recompute mean change on dates common to all models
# -----------------------------------------------------------------------------
print("\n=== C. Mean-flow change (%) on COMMON dates only (DL ensemble mean vs SWAT) ===")
rowsC = []
for st in STATIONS:
    h = daily[(daily.Station == st) & (daily.Scenario == "Historical")]
    common_h = set.intersection(*[set(h[h.Model == m].Date) for m in DL])
    hh = h[h.Date.isin(common_h)]
    base_sw = hh[hh.Model == DL[0]].SWAT.mean()
    base_dl = hh.groupby("Model").Hybrid.mean()
    for sc in SCENARIOS:
        f = daily[(daily.Station == st) & (daily.Scenario == sc)]
        common_f = set.intersection(*[set(f[f.Model == m].Date) for m in DL])
        ff = f[f.Date.isin(common_f)]
        for per, (a, z) in PERIODS.items():
            g = ff[ff.Year.between(a, z)]
            sw_chg = 100 * (g[g.Model == DL[0]].SWAT.mean() - base_sw) / base_sw
            dl_chg = [100 * (g[g.Model == m].Hybrid.mean() - base_dl[m]) / base_dl[m] for m in DL]
            rowsC.append({"Station": st, "Scenario": sc, "Period": per,
                          "SWAT": sw_chg, "DL_mean": np.mean(dl_chg),
                          "DL_min": np.min(dl_chg), "DL_max": np.max(dl_chg)})
tabC = pd.DataFrame(rowsC)
print(tabC.round(1).to_string(index=False))
tabC.round(2).to_csv(OUT / "diag_C_common_dates.csv", index=False)

# -----------------------------------------------------------------------------
# D. Input shift: how far do future predictors fall outside the training range?
# -----------------------------------------------------------------------------
def load_X(path):
    with np.load(path, allow_pickle=False) as z:
        print(f"   {path.name} keys: {z.files}")
        return z["X"].astype(np.float32), z["station"].astype(int)

print(f"\n=== D. Input shift (lookback {CHECK_LOOKBACK}), last-timestep features ===")
tag = f"lookback_{CHECK_LOOKBACK:03d}"
Xtr, st_tr = load_X(PREPARED_DIR / "historical" / tag / "train.npz")
def last(X): return X[:, -1, :] if X.ndim == 3 else X
Ftr = last(Xtr)
lo, hi = np.percentile(Ftr, 0.5, axis=0), np.percentile(Ftr, 99.5, axis=0)
mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-9

rowsD = []
for sc in SCENARIOS:
    for p in sorted(PREPARED_DIR.rglob("*.npz")):
        if sc.lower() in str(p).lower() and tag in str(p):
            X, s = load_X(p); F = last(X)
            for k, name in {0: "Shebe", 1: "Asendabo", 2: "Abelti"}.items():
                m = s == k
                if not m.any(): continue
                Fk = F[m]
                for j in range(Fk.shape[1]):
                    rowsD.append({"Scenario": sc, "File": p.name, "Station": name, "Feature": j,
                                  "mean_shift_in_train_SD": (Fk[:, j].mean() - mu[j]) / sd[j],
                                  "pct_outside_train_0.5-99.5": 100 * np.mean((Fk[:, j] < lo[j]) | (Fk[:, j] > hi[j]))})
tabD = pd.DataFrame(rowsD)
agg = tabD.groupby(["Scenario", "Station", "Feature"])[["mean_shift_in_train_SD",
                                                        "pct_outside_train_0.5-99.5"]].mean()
print("\nFeatures with the biggest shift (|mean shift| in training SDs):")
print(agg.reindex(agg.mean_shift_in_train_SD.abs().sort_values(ascending=False).index)
         .head(15).round(2).to_string())
tabD.to_csv(OUT / "diag_D_input_shift.csv", index=False)
print("   Rule of thumb: >1 SD mean shift, or >10% outside range, means DL extrapolation is a concern.")

# -----------------------------------------------------------------------------
# Figure: SWAT vs DL ensemble, mean-flow change
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(len(SCENARIOS), 3, figsize=(14, 7), sharey=True)
for i, sc in enumerate(SCENARIOS):
    for j, st in enumerate(STATIONS):
        ax = axes[i, j]
        t = tabC[(tabC.Scenario == sc) & (tabC.Station == st)].set_index("Period").reindex(list(PERIODS))
        x = np.arange(len(t)); w = 0.35
        ax.bar(x - w/2, t.SWAT, w, label="SWAT only", color="tab:gray")
        ax.bar(x + w/2, t.DL_mean, w, label="DL ensemble mean", color="tab:blue",
               yerr=[t.DL_mean - t.DL_min, t.DL_max - t.DL_mean], capsize=3)
        ax.axhline(0, color="k", lw=0.8); ax.set_xticks(x); ax.set_xticklabels(t.index)
        ax.set_title(f"{st} - {sc}")
        if j == 0: ax.set_ylabel("Change in mean flow (%)")
        if i == 0 and j == 0: ax.legend()
plt.tight_layout(); plt.savefig(OUT / "diag_swat_vs_hybrid.png", dpi=300, bbox_inches="tight")
plt.close(); print(f"\nSaved diagnostics to {OUT}")

In [ ]:
# =============================================================================
# (1) historical skill, (2) bounded / static residual variants,
#           (3) feature-shift report.   Run after Cells 6 and 7.
# =============================================================================
import numpy as np, pandas as pd, tensorflow as tf
from pathlib import Path

DATA_DIR     = PROJECT_DIR / "data"
PREPARED_DIR = RESULTS_ROOT / "LSTM_Prepared_v4"
OUT          = RESULTS_ROOT / "Projection_v4"; OUT.mkdir(exist_ok=True)

MODEL_DIRS = {
    "LSTM":     RESULTS_ROOT / "LSTM_Results_v4",
    "GRU":      RESULTS_ROOT / "GRU_Results_v4",
    "TCN":      RESULTS_ROOT / "TCN_Results_v4",
    "MLP":      RESULTS_ROOT / "MLP_Results_v4",
    "LSTM_FCN": RESULTS_ROOT / "LSTM_FCN_Results_v4",
}
SCENARIOS = ["SSP245", "SSP585"]
STATIONS  = {0: "Shebe", 1: "Asendabo", 2: "Abelti"}
PERIODS   = {"Near": (2041, 2070), "Far": (2071, 2100)}
LOG_CLIP  = 20
CLIP_Q    = (0.005, 0.995)      # residual bounds = these quantiles of the HISTORICAL residual

# Fill with your predictor names in column order to get readable output in part 3
FEATURE_NAMES = None            # e.g. ["P", "Tmax", "Tmin", "SW", ...]

# ------------------------------------------------------------------ helpers
def load_npz(path):
    with np.load(path, allow_pickle=False) as z:
        dk = "dates" if "dates" in z.files else "Date"
        d = {"X": z["X"].astype(np.float32), "station": z["station"].astype(np.int32),
             "Date": pd.to_datetime(z[dk]), "swat": z["FLOW_OUT"].astype(np.float32)}
        d["obs"] = z["Observed"].astype(np.float64) if "Observed" in z.files else None
        return d

def concat(parts):
    out = {k: np.concatenate([p[k] for p in parts]) for k in ("X", "station", "swat")}
    out["Date"] = pd.DatetimeIndex(np.concatenate([p["Date"].values for p in parts]))
    out["obs"] = (np.concatenate([p["obs"] for p in parts])
                  if all(p["obs"] is not None for p in parts) else None)
    return out

def load_hist_split(lb, split):
    return load_npz(PREPARED_DIR / "historical" / f"lookback_{lb:03d}" / f"{split}.npz")

def load_future(sc, lb):
    tag = f"lookback_{lb:03d}"
    files = sorted(p for p in PREPARED_DIR.rglob("*.npz")
                   if sc.lower() in str(p).lower() and tag in str(p))
    return concat([load_npz(p) for p in files])

def predict_resid(model, d):
    return model.predict({"sequence_input": d["X"], "station_input": d["station"].reshape(-1, 1)},
                         batch_size=512, verbose=0).reshape(-1).astype(np.float64)

def to_flow(swat, r):
    lg = np.clip(np.log1p(np.maximum(swat, 0)) + r, -LOG_CLIP, LOG_CLIP)
    return np.maximum(np.expm1(lg), 0)

def nse(o, s):  return 1 - np.sum((s - o) ** 2) / np.sum((o - o.mean()) ** 2)
def pbias(o, s): return 100 * np.sum(s - o) / np.sum(o)
def kge(o, s):
    r = np.corrcoef(o, s)[0, 1]; a = s.std() / o.std(); b = s.mean() / o.mean()
    return 1 - np.sqrt((r - 1) ** 2 + (a - 1) ** 2 + (b - 1) ** 2)

def summarize(q):
    return dict(mean=q.mean(), q05=np.quantile(q, 0.05), q95=np.quantile(q, 0.95), max=q.max())

# ------------------------------------------------------------------ main loop
skill_rows, stat_rows = [], []
for name, folder in MODEL_DIRS.items():
    b  = pd.read_csv(folder / "best_model_summary.csv").iloc[0]
    lb = int(b["lookback"])
    mname = f"L{lb}_C{int(b['config_id'])}_S{int(b['seed'])}"
    model = tf.keras.models.load_model(folder / "models" / f"{mname}.keras", compile=False)
    print(f"\n{name}: {mname}")

    # ---- (1) skill per split and station
    splits = {s: load_hist_split(lb, s) for s in ("train", "val", "test")}
    for s, d in splits.items():
        d["r"] = predict_resid(model, d)
        if d["obs"] is None: continue
        hyb = to_flow(d["swat"], d["r"])
        for k, st in STATIONS.items():
            m = (d["station"] == k) & np.isfinite(d["obs"])
            if m.sum() < 10: continue
            o = d["obs"][m]
            skill_rows.append({"Model": name, "Split": s, "Station": st, "n": int(m.sum()),
                               "NSE_SWAT": nse(o, d["swat"][m]), "NSE_Hybrid": nse(o, hyb[m]),
                               "KGE_SWAT": kge(o, d["swat"][m]), "KGE_Hybrid": kge(o, hyb[m]),
                               "PBIAS_SWAT": pbias(o, d["swat"][m]), "PBIAS_Hybrid": pbias(o, hyb[m])})

    # ---- (2) residual variants, bounds learned from historical only
    hist = concat(list(splits.values()))
    hist["r"] = np.concatenate([splits[s]["r"] for s in ("train", "val", "test")])
    bounds, monthly = {}, {}
    for k in STATIONS:
        m = hist["station"] == k
        bounds[k]  = tuple(np.quantile(hist["r"][m], CLIP_Q))
        monthly[k] = pd.Series(hist["r"][m]).groupby(hist["Date"][m].month).mean().to_dict()

    sets = {"Historical": hist}
    for sc in SCENARIOS:
        d = load_future(sc, lb); d["r"] = predict_resid(model, d); sets[sc] = d

    for key, d in sets.items():
        r_raw  = d["r"]
        r_clip = np.empty_like(r_raw); r_stat = np.empty_like(r_raw)
        for k in STATIONS:
            m = d["station"] == k
            r_clip[m] = np.clip(r_raw[m], *bounds[k])
            r_stat[m] = d["Date"][m].month.map(monthly[k]).values
        variants = {"SWAT": d["swat"].astype(np.float64),
                    "Raw": to_flow(d["swat"], r_raw),
                    "Clipped": to_flow(d["swat"], r_clip),
                    "StaticBias": to_flow(d["swat"], r_stat)}
        yrs = d["Date"].year.values
        for vname, q in variants.items():
            for k, st in STATIONS.items():
                ms = d["station"] == k
                if key == "Historical":
                    stat_rows.append({"Model": name, "Variant": vname, "Station": st,
                                      "Scenario": "Historical", "Period": "Hist", **summarize(q[ms])})
                else:
                    for per, (a, z) in PERIODS.items():
                        mp = ms & (yrs >= a) & (yrs <= z)
                        if mp.any():
                            stat_rows.append({"Model": name, "Variant": vname, "Station": st,
                                              "Scenario": key, "Period": per, **summarize(q[mp])})
    del model; tf.keras.backend.clear_session()

# ------------------------------------------------------------------ report: skill
skill = pd.DataFrame(skill_rows)
skill.to_csv(OUT / "skill_by_model_station.csv", index=False)
print("\n=== 1. HISTORICAL SKILL (test split) -- hybrid should beat SWAT, esp. at Shebe/Asendabo ===")
cols = ["NSE_SWAT", "NSE_Hybrid", "KGE_SWAT", "KGE_Hybrid", "PBIAS_SWAT", "PBIAS_Hybrid"]
if len(skill):
    print(skill[skill.Split == "test"].set_index(["Station", "Model"])[cols].round(2).to_string())
else:
    print("No 'Observed' array found in historical files.")

# ------------------------------------------------------------------ report: variants
S = pd.DataFrame(stat_rows)
base = S[S.Scenario == "Historical"].set_index(["Model", "Variant", "Station"])
fut  = S[S.Scenario != "Historical"].copy()
for c in ("mean", "q05", "q95", "max"):
    b0 = base[c].reindex(pd.MultiIndex.from_frame(fut[["Model", "Variant", "Station"]])).values
    fut[f"{c}_chg_%"] = 100 * (fut[c].values - b0) / b0
fut.to_csv(OUT / "percent_change_variants_by_model.csv", index=False)

print("\n=== 2. CHANGE (%) BY VARIANT: ensemble mean [min, max] across models ===")
for c, label in [("mean", "MEAN flow"), ("q05", "LOW flow (5th pct)"), ("q95", "HIGH flow (95th pct)")]:
    g = fut.groupby(["Station", "Scenario", "Period", "Variant"])[f"{c}_chg_%"]
    t = pd.DataFrame({"mean": g.mean(), "min": g.min(), "max": g.max()}).round(1)
    t = t.assign(txt=t["mean"].astype(str) + " [" + t["min"].astype(str) + ", " + t["max"].astype(str) + "]")
    print(f"\n{label}")
    print(t["txt"].unstack("Variant")[["SWAT", "Raw", "Clipped", "StaticBias"]].to_string())
print("\nSWAT = no DL; Raw = Cell 6; Clipped = residual bounded to historical range;"
      "\nStaticBias = month-of-year mean historical residual (no input dependence, cannot extrapolate).")

# ------------------------------------------------------------------ report: feature shift
LB = 30
tag = f"lookback_{LB:03d}"
Xtr = load_hist_split(LB, "train")
F0  = Xtr["X"][:, -1, :] if Xtr["X"].ndim == 3 else Xtr["X"]
lo, hi = np.percentile(F0, 0.5, axis=0), np.percentile(F0, 99.5, axis=0)
mu, sd = F0.mean(0), F0.std(0) + 1e-9
names = FEATURE_NAMES or [f"f{j}" for j in range(F0.shape[1])]
rows = []
for sc in SCENARIOS:
    d = load_future(sc, LB)
    F = d["X"][:, -1, :] if d["X"].ndim == 3 else d["X"]
    for k, st in STATIONS.items():
        m = d["station"] == k
        for j in range(F.shape[1]):
            rows.append({"Scenario": sc, "Station": st, "Feature": names[j],
                         "shift_SD": (F[m, j].mean() - mu[j]) / sd[j],
                         "pct_outside": 100 * np.mean((F[m, j] < lo[j]) | (F[m, j] > hi[j]))})
fs = pd.DataFrame(rows)
fs.to_csv(OUT / "feature_shift_report.csv", index=False)
w = fs.pivot_table(index=["Station", "Feature"], columns="Scenario", values=["shift_SD", "pct_outside"])
w[("diff", "SSP585-SSP245 shift_SD")] = w[("shift_SD", "SSP585")] - w[("shift_SD", "SSP245")]
w = w.reindex(w[("shift_SD", "SSP245")].abs().sort_values(ascending=False).index)
print("\n=== 3. FEATURE SHIFT (lookback 30): worst 20 ===")
print(w.head(20).round(2).to_string())
print("\nLarge SSP585-SSP245 differences in shift_SD can mean the two scenarios' inputs were built differently.")
print(f"\nSaved to {OUT}")

In [ ]:
# =============================================================================
# CELL 10 -- RAW SWAT CHECK: recompute historical vs future SWAT straight from
# the .npz files (no models, no CSVs) to settle Excel vs code.
# =============================================================================
import numpy as np, pandas as pd
from pathlib import Path

P   = RESULTS_ROOT / "LSTM_Prepared_v4"
OUT = RESULTS_ROOT / "Projection_v4"
LB  = 30
PERIODS = {"Near": (2041, 2070), "Far": (2071, 2100)}

def read(p):
    with np.load(p, allow_pickle=False) as z:
        dk = "dates" if "dates" in z.files else "Date"
        print(f"      keys: {z.files}")
        return pd.DataFrame({"Date": pd.to_datetime(z[dk]),
                             "sid": z["station"].astype(int),
                             "swat": z["FLOW_OUT"].astype(float)})

frames = []
print("=== FILES USED ===")
for p in sorted(P.rglob("*.npz")):
    s = str(p).lower()
    if f"lookback_{LB:03d}" not in s: continue
    scen = ("SSP245" if "ssp245" in s else "SSP585" if "ssp585" in s
            else "Historical" if "historical" in s else None)
    if scen is None: continue
    d = read(p); d["Scenario"] = scen; d["file"] = p.stem
    print(f"{scen:10s} {str(p.relative_to(P)):55s} ids={sorted(d.sid.unique())} n={len(d):>7,} "
          f"{d.Date.min().date()}..{d.Date.max().date()}  mean={d.swat.mean():.2f}")
    frames.append(d)
A = pd.concat(frames, ignore_index=True)
A["Year"] = A.Date.dt.year

print("\nDuplicate (scenario, station, date) rows:", int(A.duplicated(["Scenario", "sid", "Date"]).sum()))
print("\nStation id <-> file name (a file should map to ONE id, same id in historical and future):")
print(A.groupby(["file", "Scenario"]).sid.agg(lambda x: sorted(set(x))).unstack("Scenario").to_string())

# ---- period labels
A["Period"] = "Hist"
for per, (a, z) in PERIODS.items():
    A.loc[(A.Scenario != "Historical") & A.Year.between(a, z), "Period"] = per
A = A[~((A.Scenario != "Historical") & (A.Period == "Hist"))]    # drop future years outside Near/Far

# ---- distribution summary: mean must be consistent with the quantiles
g = A.groupby(["sid", "Scenario", "Period"]).swat
S = g.agg(n="size", mean="mean", p5=lambda x: x.quantile(.05), median="median",
          p95=lambda x: x.quantile(.95), max="max")
print("\n=== RAW SWAT DISTRIBUTION BY STATION ID (0=Shebe? 1=Asendabo? 2=Abelti?) ===")
print(S.round(2).to_string())

base = S.xs("Hist", level="Period").droplevel("Scenario")["mean"]
chg = S["mean"].reset_index()
chg = chg[chg.Period != "Hist"]
chg["hist_mean"] = chg.sid.map(base)
chg["mean_change_%"] = 100 * (chg["mean"] - chg.hist_mean) / chg.hist_mean
print("\n=== MEAN CHANGE FROM RAW FILES (%) ===")
print(chg.pivot_table(index="sid", columns=["Scenario", "Period"], values="mean_change_%").round(1).to_string())

# ---- annual means for pasting next to your Excel numbers
ann = A.groupby(["sid", "Scenario", "Year"]).swat.mean().unstack(["sid"])
ann.round(3).to_csv(OUT / "annual_mean_swat_check.csv")
print(f"\nAnnual means saved to {OUT / 'annual_mean_swat_check.csv'} -- compare with your Excel columns.")

In [ ]:
# =============================================================================
# MEAN FLOW: Observed vs SWAT vs DL models, historical / near / far
# Needs Cell 6 outputs (daily_projections_all_models.csv). No model re-run.
# =============================================================================
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR     = PROJECT_DIR / "data"
PREPARED_DIR = RESULTS_ROOT / "LSTM_Prepared_v4"
OUT          = RESULTS_ROOT / "Projection_v4"
STATIONS     = {0: "Shebe", 1: "Asendabo", 2: "Abelti"}
SCENARIOS    = ["SSP245", "SSP585"]
PERIODS      = {"Near": (2041, 2070), "Far": (2071, 2100)}
REF_MODEL    = "MLP"      # SWAT values are taken from this model's rows (lookback 30 = longest coverage)
OBS_LOOKBACK = 30         # historical files used to read the Observed array
UNITS        = "flow units of FLOW_OUT"   # change to "m3/s" if that is what it is

daily = pd.read_csv(OUT / "daily_projections_all_models.csv", parse_dates=["Date"])
daily["Year"] = daily.Date.dt.year
models = [m for m in daily.Model.unique()]

# ------------------------------------------------------------------ observed
parts = []
for s in ("train", "val", "test"):
    p = PREPARED_DIR / "historical" / f"lookback_{OBS_LOOKBACK:03d}" / f"{s}.npz"
    with np.load(p, allow_pickle=False) as z:
        dk = "dates" if "dates" in z.files else "Date"
        parts.append(pd.DataFrame({"Date": pd.to_datetime(z[dk]),
                                   "Station": [STATIONS[int(k)] for k in z["station"]],
                                   "Observed": z["Observed"].astype(float)}))
obs = pd.concat(parts).dropna().drop_duplicates(["Date", "Station"])
print(f"Observed: {obs.Date.min().date()} to {obs.Date.max().date()}, {len(obs):,} station-days")

# ------------------------------------------------------------------ mean-flow table
COLS = ["Historical"] + [f"{sc}_{per}" for sc in SCENARIOS for per in PERIODS]

def means_for(model, col):
    d = daily[daily.Model == model]
    out = {"Historical": d[d.Scenario == "Historical"].groupby("Station")[col].mean()}
    for sc in SCENARIOS:
        for per, (a, z) in PERIODS.items():
            m = d[(d.Scenario == sc) & d.Year.between(a, z)]
            out[f"{sc}_{per}"] = m.groupby("Station")[col].mean()
    return pd.DataFrame(out)

tabs = {"Observed": pd.DataFrame({"Historical": obs.groupby("Station").Observed.mean()}),
        "SWAT": means_for(REF_MODEL, "SWAT")}
for m in models:
    tabs[m] = means_for(m, "Hybrid")
dl = pd.concat([tabs[m] for m in models]).groupby(level=0).mean()
tabs["DL_ensemble"] = dl

full = pd.concat(tabs, names=["Source", "Station"]).reindex(columns=COLS)
order = ["Observed", "SWAT"] + models + ["DL_ensemble"]
full = full.reindex(order, level="Source").reindex(list(STATIONS.values()), level="Station")
full = full.swaplevel().sort_index(level="Station", sort_remaining=False)
full = full.reindex(order, level="Source")
full.round(2).to_csv(OUT / "mean_flow_table.csv")
print(f"\n=== MEAN FLOW ({UNITS}) ===")
for st in STATIONS.values():
    print(f"\n{st}")
    print(full.loc[st].round(1).to_string())

# ------------------------------------------------------------------ % change vs own historical
chg = full[COLS[1:]].div(full["Historical"], axis=0).sub(1).mul(100)
chg.round(1).to_csv(OUT / "mean_flow_pct_change_vs_own_hist.csv")

# ------------------------------------------------------------------ matched to observed dates
print("\n=== HISTORICAL MEAN ON DATES WITH OBSERVATIONS (like-for-like) ===")
rows = []
for st in STATIONS.values():
    o = obs[obs.Station == st]
    r = {"Station": st, "Observed": o.Observed.mean()}
    h = daily[(daily.Scenario == "Historical") & (daily.Station == st)]
    ref = h[h.Model == REF_MODEL].merge(o, on=["Date", "Station"])
    r["SWAT"] = ref.SWAT.mean()
    for m in models:
        r[m] = h[h.Model == m].merge(o, on=["Date", "Station"]).Hybrid.mean()
    r["n_days"] = len(ref)
    rows.append(r)
matched = pd.DataFrame(rows).set_index("Station")
matched.round(2).to_csv(OUT / "mean_flow_hist_matched_to_obs.csv")
print(matched.round(1).to_string())

# ------------------------------------------------------------------ figure
colors = {"Historical": "0.55", "SSP245_Near": "#9ecae1", "SSP245_Far": "#3182bd",
          "SSP585_Near": "#fcae91", "SSP585_Far": "#de2d26"}
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=False)
for ax, st in zip(axes, STATIONS.values()):
    t = full.loc[st].reindex(order)
    x = np.arange(len(t)); w = 0.16
    for i, c in enumerate(COLS):
        ax.bar(x + (i - 2) * w, t[c].values, w, label=c.replace("_", " "), color=colors[c])
    ax.set_xticks(x); ax.set_xticklabels(order, rotation=30, ha="right")
    ax.set_title(st); ax.set_ylabel(f"Mean flow ({UNITS})")
axes[0].legend(fontsize=8, ncol=1)
plt.tight_layout(); plt.savefig(OUT / "mean_flow_obs_swat_models.png", dpi=300, bbox_inches="tight")
plt.close(); print(f"\nSaved to {OUT}")

In [ ]:
#   Fig X1: change in mean flow by architecture (+ SWAT only), station x scenario
#   Fig X2: SWAT only vs DL ensemble mean with min-max whiskers (common dates)
# =============================================================================
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from pathlib import Path

OUT       = RESULTS_ROOT / "Projection_v4"
SCENARIOS = ["SSP245", "SSP585"]
STATIONS  = ["Shebe", "Asendabo", "Abelti"]
PERIODS   = ["Near", "Far"]
PERIOD_LABEL = {"Near": "Near (2041-2070)", "Far": "Far (2071-2100)"}
MODELS    = ["LSTM", "GRU", "TCN", "MLP", "LSTM_FCN"]
MODEL_LABEL = {"LSTM_FCN": "LSTM-FCN"}

# Colour-blind-safe palette (Okabe-Ito); SWAT in neutral grey
COL = {"LSTM": "#0072B2", "GRU": "#E69F00", "TCN": "#009E73",
       "MLP": "#CC79A7", "LSTM_FCN": "#D55E00", "SWAT": "#7F7F7F"}

plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10, "axes.labelsize": 10, "axes.titlesize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "legend.frameon": False,
})
letters = "abcdefghijkl"

# -----------------------------------------------------------------------------
# Figure X1
# -----------------------------------------------------------------------------
chg  = pd.read_csv(OUT / "percent_change_by_model.csv")
bars = MODELS + ["SWAT"]
nb   = len(bars); w = 0.8 / nb
x    = np.arange(len(PERIODS))

fig, axes = plt.subplots(len(SCENARIOS), len(STATIONS), figsize=(11, 6.2),
                         sharey=True, constrained_layout=True)
k = 0
for i, sc in enumerate(SCENARIOS):
    for j, st in enumerate(STATIONS):
        ax = axes[i, j]
        sub = chg[(chg.Scenario == sc) & (chg.Station == st)]
        for b, m in enumerate(bars):
            vals = [sub[(sub.Model == m) & (sub.Period == p)]["mean_change_%"].mean() for p in PERIODS]
            ax.bar(x + (b - (nb - 1) / 2) * w, vals, w, color=COL[m],
                   hatch="//" if m == "SWAT" else None,
                   edgecolor="white" if m != "SWAT" else "k", linewidth=0.4)
        ax.axhline(0, color="k", lw=0.8)
        ax.set_xticks(x); ax.set_xticklabels([PERIOD_LABEL[p] for p in PERIODS])
        ax.set_title(f"({letters[k]}) {st} - {sc}", loc="left", fontweight="bold"); k += 1
        if j == 0: ax.set_ylabel("Change in mean flow (%)")
handles = [Patch(facecolor=COL[m], hatch="//" if m == "SWAT" else None,
                  edgecolor="k" if m == "SWAT" else "white",
                  label=("SWAT only" if m == "SWAT" else MODEL_LABEL.get(m, m))) for m in bars]
fig.legend(handles=handles, loc="outside lower center", ncol=nb)
fig.savefig(OUT / "Fig_X1_mean_flow_change.png", dpi=300)
fig.savefig(OUT / "Fig_X1_mean_flow_change.pdf")
plt.close(fig)

# -----------------------------------------------------------------------------
# Figure X2  (needs diag_C_common_dates.csv from Cell 7)
# -----------------------------------------------------------------------------
tabC = pd.read_csv(OUT / "diag_C_common_dates.csv")
fig, axes = plt.subplots(len(SCENARIOS), len(STATIONS), figsize=(11, 6.2),
                         sharey=True, constrained_layout=True)
k = 0; w = 0.36
for i, sc in enumerate(SCENARIOS):
    for j, st in enumerate(STATIONS):
        ax = axes[i, j]
        t = (tabC[(tabC.Scenario == sc) & (tabC.Station == st)]
             .set_index("Period").reindex(PERIODS))
        xx = np.arange(len(t))
        ax.bar(xx - w / 2, t.SWAT, w, color=COL["SWAT"], hatch="//", edgecolor="k", linewidth=0.4)
        ax.bar(xx + w / 2, t.DL_mean, w, color="#0072B2", edgecolor="white", linewidth=0.4,
               yerr=[t.DL_mean - t.DL_min, t.DL_max - t.DL_mean],
               error_kw=dict(ecolor="k", elinewidth=0.9, capsize=3))
        ax.axhline(0, color="k", lw=0.8)
        ax.set_xticks(xx); ax.set_xticklabels([PERIOD_LABEL[p] for p in PERIODS])
        ax.set_title(f"({letters[k]}) {st} - {sc}", loc="left", fontweight="bold"); k += 1
        if j == 0: ax.set_ylabel("Change in mean flow (%)")
handles = [Patch(facecolor=COL["SWAT"], hatch="//", edgecolor="k", label="SWAT only"),
           Patch(facecolor="#0072B2", label="DL ensemble mean"),
           Line2D([0], [0], color="k", marker="_", markersize=8, lw=1, label="Min-max across architectures")]
fig.legend(handles=handles, loc="outside lower center", ncol=3)
fig.savefig(OUT / "Fig_X2_swat_vs_hybrid.png", dpi=300)
fig.savefig(OUT / "Fig_X2_swat_vs_hybrid.pdf")
plt.close(fig)
print(f"Saved Fig_X1 and Fig_X2 (PNG 300 dpi + PDF) to {OUT}")

# ARCHITECTURAL UNCERTAINTY ANALYSIS 

In [ ]:
# =============================================================================
# ARCHITECTURAL UNCERTAINTY ANALYSIS  
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# =============================================================================
# 1. PATHS AND SETTINGS
# =============================================================================

DATA_DIR = PROJECT_DIR / "data"

OUT = RESULTS_ROOT / "Projection_v4"
FILE_ANNUAL = OUT / "annual_change" / "annual_change_period_summary.csv"

FIG_DIR = OUT / "Architectural_Uncertainty_Figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

MODELS = ["LSTM", "GRU", "TCN", "MLP", "LSTM_FCN"]
MODEL_LABELS = ["LSTM", "GRU", "TCN", "MLP", "LSTM-FCN"]
STATIONS = ["Shebe", "Asendabo", "Abelti"]
SCENARIOS = ["SSP245", "SSP585"]
PERIODS = ["Near", "Far"]

PERIOD_LABELS = {
    "Near": "Near future (2041\u20132070)",
    "Far": "Far future (2071\u20132100)",
}

MODEL_COLORS = {
    "LSTM": "#2878B5",
    "GRU": "#E07A1F",
    "TCN": "#32965A",
    "MLP": "#9B59B6",
    "LSTM_FCN": "#C94C4C",
}

SCENARIO_COLORS = {"SSP245": "#2878B5", "SSP585": "#D9534F"}

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 18,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 18,
    "figure.titlesize": 18,
    "axes.linewidth": 0.9,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "figure.dpi": 300,
    "savefig.dpi": 600,
})

# =============================================================================
# 2. LOAD DATA FROM THE ANNUAL-CHANGE OUTPUT
# =============================================================================

if not FILE_ANNUAL.exists():
    raise FileNotFoundError(
        f"Missing {FILE_ANNUAL}\n"
        "Run annual_change_from_daily_projections.py first."
    )

annual = pd.read_csv(FILE_ANNUAL)

need = ["Station", "Scenario", "Model", "Period", "Period_change_%"]
missing = [c for c in need if c not in annual.columns]
if missing:
    raise ValueError(f"annual_change_period_summary.csv lacks columns: {missing}")

for c in ["Station", "Scenario", "Model", "Period"]:
    annual[c] = annual[c].astype(str).str.strip()

# Five architectures -> same column name the rest of the script uses
chg = annual[annual["Model"].isin(MODELS)].copy()
chg = chg.rename(columns={"Period_change_%": "mean_change_%"})

swat = (annual[annual["Model"] == "SWAT"]
        .set_index(["Station", "Scenario", "Period"])["Period_change_%"])
ens = (annual[annual["Model"] == "Ensemble"]
       .set_index(["Station", "Scenario", "Period"])["Period_change_%"])

if ens.empty:
    raise ValueError("No 'Ensemble' rows found in the annual summary.")

# Standardise categorical order
for col, cats in (("Station", STATIONS), ("Scenario", SCENARIOS),
                  ("Period", PERIODS)):
    chg[col] = pd.Categorical(chg[col], categories=cats, ordered=True)

# Completeness check
for st in STATIONS:
    for sc in SCENARIOS:
        for per in PERIODS:
            tmp = chg[(chg.Station == st) & (chg.Scenario == sc)
                      & (chg.Period == per)]
            if set(tmp.Model.astype(str)) != set(MODELS):
                raise ValueError(
                    f"Missing architecture results: {st}, {sc}, {per}")

# Architectural range per station/scenario/period
range_data = (
    chg.groupby(["Station", "Scenario", "Period"], observed=False)
    ["mean_change_%"]
    .agg(minimum="min", maximum="max", mean="mean", sd="std")
    .reset_index()
)
range_data["Architecture_Range"] = range_data["maximum"] - range_data["minimum"]

# Scenario difference from the ensemble (same ensemble as the annual figure)
rows = []
for st in STATIONS:
    for per in PERIODS:
        e245 = float(ens.loc[(st, "SSP245", per)])
        e585 = float(ens.loc[(st, "SSP585", per)])
        rows.append({
            "Station": st, "Period": per,
            "Ens_SSP245": e245, "Ens_SSP585": e585,
            "scenario_diff": abs(e245 - e585),
        })
scenario = pd.DataFrame(rows)

# Table used by Figure Z (one row per station/scenario/period)
spread = range_data.merge(scenario, on=["Station", "Period"], how="left")
spread = spread.rename(columns={"Architecture_Range": "model_range"})
for col, cats in (("Station", STATIONS), ("Scenario", SCENARIOS),
                  ("Period", PERIODS)):
    spread[col] = pd.Categorical(spread[col].astype(str), categories=cats,
                                 ordered=True)

print("Data loaded from:", FILE_ANNUAL)
print("Models:", MODELS)
print("Stations:", STATIONS)

# =============================================================================
# FIGURE X -- INTER-ARCHITECTURE VARIABILITY
# =============================================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=False)

x = np.arange(len(MODELS))
width = 0.34

for i, sc in enumerate(SCENARIOS):
    for j, st in enumerate(STATIONS):

        ax = axes[i, j]
        sub = chg[(chg.Scenario == sc) & (chg.Station == st)]

        near = (sub[sub.Period == "Near"].set_index("Model")
                .reindex(MODELS)["mean_change_%"])
        far = (sub[sub.Period == "Far"].set_index("Model")
               .reindex(MODELS)["mean_change_%"])

        for k, model in enumerate(MODELS):
            ax.bar(x[k] - width / 2, near[model], width,
                   color=MODEL_COLORS[model], edgecolor="black",
                   linewidth=0.5, alpha=0.75)
            ax.bar(x[k] + width / 2, far[model], width,
                   color=MODEL_COLORS[model], edgecolor="black",
                   linewidth=0.5, alpha=1.0)

        ax.axhline(0, color="black", linewidth=0.9)
        ax.set_xticks(x)
        ax.set_xticklabels(MODEL_LABELS, rotation=0)
        ax.set_title(f"{st} \u2013 {sc}", pad=9)
        ax.set_ylabel("Change in mean streamflow (%)" if j == 0 else "")
        ax.grid(axis="y", linestyle="--", alpha=0.30)
        ax.set_axisbelow(True)

model_handles = [
    Patch(facecolor=MODEL_COLORS[m], edgecolor="black", label=lab)
    for m, lab in zip(MODELS, MODEL_LABELS)
]
period_handles = [
    Patch(facecolor="gray", alpha=0.65, edgecolor="black", label="Near future"),
    Patch(facecolor="gray", alpha=1.0, edgecolor="black", label="Far future"),
]

fig.legend(handles=model_handles + period_handles, loc="lower center",
           ncol=7, frameon=False, bbox_to_anchor=(0.5, -0.015))

fig.suptitle("Inter-architecture variability in projected mean streamflow",
             fontsize=18, fontweight="bold", y=1.01)

plt.tight_layout(rect=[0, 0.07, 1, 0.97])
fig.savefig(FIG_DIR / "Figure_X_Inter_Architecture.png", dpi=600,
            bbox_inches="tight")
plt.show()
plt.close(fig)

# =============================================================================
# FIGURE Y -- SPATIAL AND TEMPORAL DEPENDENCE
# =============================================================================

heat = range_data.pivot_table(
    index="Station", columns=["Period", "Scenario"],
    values="Architecture_Range", observed=False,
)

vmin = np.nanmin(heat.values)
vmax = np.nanmax(heat.values)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), sharey=True,
                         constrained_layout=True)

for ax, per in zip(axes, PERIODS):

    data = heat[per].reindex(index=STATIONS, columns=SCENARIOS)

    im = ax.imshow(data.values, aspect="auto", cmap="YlOrRd",
                   vmin=vmin, vmax=vmax, interpolation="nearest")

    ax.set_xticks(np.arange(len(SCENARIOS)))
    ax.set_xticklabels(SCENARIOS)
    ax.set_yticks(np.arange(len(STATIONS)))
    ax.set_yticklabels(STATIONS)
    ax.set_title(PERIOD_LABELS[per], fontweight="bold", pad=10)

    for r in range(len(STATIONS)):
        for c in range(len(SCENARIOS)):
            value = data.iloc[r, c]
            norm = (value - vmin) / (vmax - vmin) if vmax > vmin else 0
            ax.text(c, r, f"{value:.1f}", ha="center", va="center",
                    fontsize=18, fontweight="bold",
                    color="white" if norm > 0.65 else "black")

    ax.set_xticks(np.arange(-0.5, len(SCENARIOS), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(STATIONS), 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=2)
    ax.grid(which="major", visible=False)
    ax.tick_params(which="minor", bottom=False, left=False)

axes[0].set_ylabel("Monitoring station")
axes[0].set_xlabel("Climate scenario")
axes[1].set_xlabel("Climate scenario")

cbar = fig.colorbar(im, ax=axes, fraction=0.04, pad=0.03)
cbar.set_label("Inter-architecture range (percentage points)",
               rotation=90, labelpad=12)

fig.savefig(FIG_DIR / "Figure_Y_Spatial_Temporal_Uncertainty.png", dpi=600,
            bbox_inches="tight")
plt.show()
plt.close(fig)

# =============================================================================
# FIGURE Z -- ARCHITECTURAL VS CLIMATE-SCENARIO UNCERTAINTY
# =============================================================================

ymax = float(spread["model_range"].max()) * 1.15

fig = plt.figure(figsize=(15, 8.6), constrained_layout=True)
subfigs = fig.subfigures(2, 1)

letters = "abcdef"
n = 0

for i, per in enumerate(PERIODS):

    subfig = subfigs[i]
    subfig.suptitle(PERIOD_LABELS[per], fontsize=18)
    axes = subfig.subplots(1, 3, sharey=True)

    for j, st in enumerate(STATIONS):

        ax = axes[j]
        sub = spread[(spread.Station == st) & (spread.Period == per)]

        vals = {}
        for sc in SCENARIOS:
            v = sub.loc[sub.Scenario == sc, "model_range"]
            if len(v) != 1:
                raise ValueError(
                    f"Expected one architecture range for {st}, {sc}, {per}")
            vals[sc] = float(v.iloc[0])

        scen_value = float(
            scenario[(scenario.Station == st) & (scenario.Period == per)]
            ["scenario_diff"].iloc[0]
        )

        positions = np.array([0, 1])
        heights = [vals["SSP245"], vals["SSP585"]]

        bars = ax.bar(
            positions, heights, width=0.55,
            color=[SCENARIO_COLORS["SSP245"], SCENARIO_COLORS["SSP585"]],
            edgecolor="black", linewidth=0.7, alpha=0.85, zorder=2,
        )

        ax.axhline(scen_value, color="black", linestyle="--",
                   linewidth=1.8, zorder=3)

        for bar, val in zip(bars, heights):
            cx = bar.get_x() + bar.get_width() / 2
            if val <= scen_value <= val + ymax * 0.09:
                ax.text(cx, val - ymax * 0.015, f"{val:.1f}",
                        ha="center", va="top", fontsize=18, color="white", zorder=5)
            else:
                ax.text(cx, val + ymax * 0.015, f"{val:.1f}",
                        ha="center", va="bottom", zorder=5,
                        bbox=dict(facecolor="white", edgecolor="none",
                                  pad=1.2, alpha=0.9))

        ax.set_xlim(-0.6, 1.6)
        ax.set_ylim(0, ymax)
        ax.set_xticks(positions)
        ax.set_xticklabels(SCENARIOS)

        if j == 0:
            ax.set_ylabel("Inter-architecture range")

        ax.set_title(
            f"({letters[n]}) {st}\nScenario difference = {scen_value:.1f} pp",
            loc="left", pad=6,
        )
        n += 1

        ax.grid(axis="y", linestyle="--", alpha=0.30)
        ax.set_axisbelow(True)

handles = [
    Patch(facecolor=SCENARIO_COLORS["SSP245"], edgecolor="black",
          label="Architectural range: SSP245"),
    Patch(facecolor=SCENARIO_COLORS["SSP585"], edgecolor="black",
          label="Architectural range: SSP585"),
    Line2D([0], [0], color="black", linestyle="--", linewidth=1.8,
           label="SSP245\u2013SSP585 difference in ensemble-mean change"),
]

fig.legend(handles=handles, loc="outside lower center", ncol=3,
           frameon=False)

fig.savefig(FIG_DIR / "Figure_Z_Architecture_vs_Scenario.png", dpi=600,
            bbox_inches="tight")
plt.show()
plt.close(fig)

# =============================================================================
# 3. SUMMARY AND TABLE X1
# =============================================================================

summary = range_data.merge(scenario[["Station", "Period", "scenario_diff"]],
                           on=["Station", "Period"], how="left")

summary["Architecture_to_Scenario_Ratio"] = np.where(
    summary["scenario_diff"] > 0,
    summary["Architecture_Range"] / summary["scenario_diff"], np.nan)
summary["Architecture_Exceeds_Scenario"] = (
    summary["Architecture_Range"] > summary["scenario_diff"])

summary = summary[["Station", "Scenario", "Period", "minimum", "maximum",
                   "mean", "sd", "Architecture_Range", "scenario_diff",
                   "Architecture_to_Scenario_Ratio",
                   "Architecture_Exceeds_Scenario"]]
summary.to_csv(FIG_DIR / "Architectural_Uncertainty_Summary.csv", index=False)


def sign_label(lo, hi):
    if lo > 0:
        return "All increase"
    if hi < 0:
        return "All decrease"
    return "Mixed"


t = []
for _, r in range_data.iterrows():
    key = (str(r["Station"]), str(r["Scenario"]), str(r["Period"]))
    sd_ = float(scenario[(scenario.Station == key[0])
                         & (scenario.Period == key[2])]["scenario_diff"].iloc[0])
    t.append({
        "Station": key[0], "Period": key[2], "Scenario": key[1],
        "SWAT_only_%": float(swat.loc[key]),
        "DL_ensemble_%": float(ens.loc[key]),
        "DL_min_%": r["minimum"], "DL_max_%": r["maximum"],
        "Model_range_pp": r["Architecture_Range"],
        "Sign_of_five_architectures": sign_label(r["minimum"], r["maximum"]),
        "Scenario_diff_pp": sd_,
        "Range_exceeds_scenario_diff": r["Architecture_Range"] > sd_,
    })

table_x1 = pd.DataFrame(t)
table_x1["Station"] = pd.Categorical(table_x1["Station"], STATIONS,
                                     ordered=True)
table_x1["Period"] = pd.Categorical(table_x1["Period"], PERIODS, ordered=True)
table_x1 = table_x1.sort_values(["Station", "Period", "Scenario"])
table_x1.to_csv(FIG_DIR / "Table_X1_Architecture_vs_Scenario.csv", index=False)

# =============================================================================
# 4. PRINT RESULTS FOR MANUSCRIPT
# =============================================================================

print("\n" + "=" * 85)
print("TABLE X1 -- ARCHITECTURE VS SCENARIO (annual-consistent)")
print("=" * 85)
print(table_x1.round(1).to_string(index=False))

n_exceed = int(table_x1["Range_exceeds_scenario_diff"].sum())
n_mixed = int((table_x1["Sign_of_five_architectures"] == "Mixed").sum())
print(f"\nArchitectural range exceeds scenario difference in {n_exceed} of "
      f"{len(table_x1)} cases.")
print(f"Architectures disagree on the sign of the change in {n_mixed} of "
      f"{len(table_x1)} cases.")
print(f"Model range: {table_x1['Model_range_pp'].min():.1f} to "
      f"{table_x1['Model_range_pp'].max():.1f} pp")
print(f"Ensemble minus SWAT-only: "
      f"{(table_x1['DL_ensemble_%'] - table_x1['SWAT_only_%']).min():.1f} to "
      f"{(table_x1['DL_ensemble_%'] - table_x1['SWAT_only_%']).max():.1f} pp")

print("\nFigures saved in:", FIG_DIR)
for f in sorted(FIG_DIR.iterdir()):
    print("  ", f.name)

In [ ]:
# =============================================================================
#  DAILY FLOW, ALL STATIONS IN ONE FIGURE

# =============================================================================
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from pathlib import Path

DATA_DIR     = PROJECT_DIR / "data"
PREPARED_DIR = RESULTS_ROOT / "LSTM_Prepared_v4"
OUT          = RESULTS_ROOT / "Projection_v4"
PLOT_DIR     = OUT / "daily_flow_plots"; PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- options
SMOOTH_DAYS = 30       # 1 = raw daily; 30 = 30-day rolling mean (recommended for 30-60 yr records)
LOG_Y       = False
MODELS      = ["LSTM", "GRU", "TCN", "MLP", "LSTM_FCN"]
MODEL_LABEL = {"LSTM_FCN": "LSTM-FCN"}
STATIONS    = ["Shebe", "Asendabo", "Abelti"]
SCENARIOS   = ["Historical", "SSP245", "SSP585"]
PERIODS     = {"Near": (2041, 2070), "Far": (2071, 2100)}
PERIOD_NAME = {"Near": "Near future", "Far": "Far future"}
PERIOD_FILL = {"Near": "#FDE0B5", "Far": "#CFE3F5"}     # background shading

COLORS = {"Observed": "black", "SWAT": "#7f7f7f",
          "LSTM": "#1f77b4", "GRU": "#ff7f0e", "TCN": "#2ca02c",
          "MLP": "#d62728", "LSTM_FCN": "#9467bd"}
ENS_COLOR = "#111111"

plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 12, "axes.labelsize": 12, "axes.titlesize": 12,
    "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 12,
    "axes.spines.top": False, "axes.spines.right": False,
})

# ---------------------------------------------------------------- load data
daily = pd.read_csv(OUT / "daily_projections_all_models.csv", parse_dates=["Date"])

STATION_IDS = {0: "Shebe", 1: "Asendabo", 2: "Abelti"}
obs_parts = []
for split in ("train", "val", "test"):
    with np.load(PREPARED_DIR / "historical" / "lookback_030" / f"{split}.npz",
                 allow_pickle=False) as z:
        obs_parts.append(pd.DataFrame({
            "Date":     pd.to_datetime(z["dates"]),
            "Station":  [STATION_IDS[int(s)] for s in z["station"]],
            "Observed": z["Observed"].astype(float)}))
obs = pd.concat(obs_parts, ignore_index=True)

def series(station, scenario):
    """Daily DataFrame: SWAT, each model, Ensemble (mean of the 5 models), Observed (hist only).
    The ensemble uses only dates on which all five models have a value."""
    sub = daily[(daily.Station == station) & (daily.Scenario == scenario)]
    out = sub.groupby("Date")["SWAT"].mean().to_frame("SWAT")
    for m in MODELS:
        out[m] = sub[sub.Model == m].set_index("Date")["Hybrid"]
    out["Ensemble"] = out[MODELS].mean(axis=1, skipna=False)      # NaN unless all 5 present
    if scenario == "Historical":
        out["Observed"] = obs[obs.Station == station].set_index("Date")["Observed"]
    return out.sort_index()

def smooth(df):
    return df.rolling(SMOOTH_DAYS, min_periods=max(1, SMOOTH_DAYS // 2)).mean() if SMOOTH_DAYS > 1 else df

data = {(st, sc): series(st, sc) for st in STATIONS for sc in SCENARIOS}

# ---------------------------------------------------------------- figure
spans = []
for sc in SCENARIOS:
    d0 = min(data[(st, sc)].index.min() for st in STATIONS)
    d1 = max(data[(st, sc)].index.max() for st in STATIONS)
    spans.append((d0, d1))
ratios = [(b - a).days for a, b in spans]          # column width proportional to time span

fig, axes = plt.subplots(len(STATIONS), len(SCENARIOS), figsize=(17, 10),
                         sharey="row", constrained_layout=True,
                         gridspec_kw={"width_ratios": ratios})
letters = "abcdefghi"; n = 0

for i, st in enumerate(STATIONS):
    hist_mean = data[(st, "Historical")]["Ensemble"].mean()
    for j, sc in enumerate(SCENARIOS):
        ax = axes[i, j]
        df = data[(st, sc)]
        sm = smooth(df)

        # shaded Near / Far background (future columns only)
        if sc != "Historical":
            for per, (a, z) in PERIODS.items():
                ax.axvspan(pd.Timestamp(a, 1, 1), pd.Timestamp(z, 12, 31),
                           color=PERIOD_FILL[per], alpha=0.75, lw=0, zorder=0)

        # individual series (thin), then Observed, then the ensemble mean (thick)
        order = ["SWAT"] + MODELS
        for c in order:
            if sm[c].notna().any():
                ax.plot(sm.index, sm[c], color=COLORS[c], lw=0.7 if c != "SWAT" else 0.9,
                        alpha=0.55 if c in MODELS else 0.9, zorder=2)
        if sc == "Historical" and sm["Observed"].notna().any():
            ax.plot(sm.index, sm["Observed"], color=COLORS["Observed"], lw=1.0, zorder=3)
        ax.plot(sm.index, sm["Ensemble"], color=ENS_COLOR, lw=2.0, zorder=4)

        # mean flow of the ensemble: dashed level + printed value
        blocks = {"Hist": (df.index.min(), df.index.max())} if sc == "Historical" else \
                 {p: (pd.Timestamp(a, 1, 1), pd.Timestamp(z, 12, 31)) for p, (a, z) in PERIODS.items()}
        for key, (t0, t1) in blocks.items():
            e = df.loc[t0:t1, "Ensemble"].dropna()
            if e.empty: continue
            mval = e.mean()
            ax.hlines(mval, t0, t1, colors=ENS_COLOR, linestyles="--", lw=1.2, zorder=5)
            txt = f"Mean {mval:.1f}" + ("" if key == "Hist" else f" ({100 * (mval - hist_mean) / hist_mean:+.0f}%)")
            tx = t0 + (t1 - t0) / 2
            ax.text(tx, 0.97, txt, transform=ax.get_xaxis_transform(), ha="center", va="top",
                    fontsize=12, fontweight="bold", zorder=6,
                    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.5))

        # Near / Far written once, on the top row only
        if i == 0 and sc != "Historical":
            for per, (a, z) in PERIODS.items():
                mid = pd.Timestamp(a, 1, 1) + (pd.Timestamp(z, 12, 31) - pd.Timestamp(a, 1, 1)) / 2
                ax.text(mid, 1.02, f"{PERIOD_NAME[per]} ({a}\u2013{z})", transform=ax.get_xaxis_transform(),
                        ha="center", va="bottom", fontsize=12, fontweight="bold")

        ax.set_xlim(spans[j])
        if LOG_Y: ax.set_yscale("log")
        ax.grid(alpha=0.25, zorder=1)
        ax.xaxis.set_major_locator(mdates.YearLocator(5 if sc == "Historical" else 10))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        # column name on the top row, station name on the left column
        if i == 0:
            ax.set_title(sc, fontweight="bold", pad=24 if sc != "Historical" else 6)
        if j == 0:
            ax.set_ylabel(f"{st}\nStreamflow" + (f" ({SMOOTH_DAYS}-day mean)" if SMOOTH_DAYS > 1 else ""))
        else:
            ax.tick_params(labelleft=False)
        ax.text(0.01, 0.97, f"({letters[n]})", transform=ax.transAxes, ha="left", va="top",
                fontweight="bold", zorder=6); n += 1

# ---------------------------------------------------------------- legend
handles = [Line2D([0], [0], color=COLORS["Observed"], lw=1.2, label="Observed"),
           Line2D([0], [0], color=COLORS["SWAT"], lw=1.2, label="SWAT")] + \
          [Line2D([0], [0], color=COLORS[m], lw=1.2, alpha=0.7, label=MODEL_LABEL.get(m, m)) for m in MODELS] + \
          [Line2D([0], [0], color=ENS_COLOR, lw=2.2, label="Ensemble mean (5 models)"),
           Line2D([0], [0], color=ENS_COLOR, lw=1.2, ls="--", label="Period mean flow"),
           Patch(facecolor=PERIOD_FILL["Near"], label="Near future"),
           Patch(facecolor=PERIOD_FILL["Far"], label="Far future")]
fig.legend(handles=handles, loc="outside lower center", ncol=6, frameon=False)

tag = f"smooth{SMOOTH_DAYS}" + ("_log" if LOG_Y else "")
fig.savefig(PLOT_DIR / f"daily_flow_all_stations_{tag}.png", dpi=600, bbox_inches="tight")
plt.show(); plt.close(fig)
print(f"Saved to {PLOT_DIR / f'daily_flow_all_stations_{tag}.png'}")

In [ ]:
# =============================================================================
# SEASONAL COMPARISON: observed vs future ensemble (5 hybrid models)
#   Fig A: seasonal mean flow  (Observed | Near | Far), whiskers = min-max across models
#   Fig B: seasonal change (%) relative to the OBSERVED seasonal mean
#   Table: Seasonal_comparison_summary.csv
# =============================================================================
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from pathlib import Path

DATA_DIR     = PROJECT_DIR / "data"
PREPARED_DIR = RESULTS_ROOT / "LSTM_Prepared_v4"
OUT          = RESULTS_ROOT / "Projection_v4"
FIG_DIR      = OUT / "seasonal_figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)

MODELS    = ["LSTM", "GRU", "TCN", "MLP", "LSTM_FCN"]
STATIONS  = ["Shebe", "Asendabo", "Abelti"]
SCENARIOS = ["SSP245", "SSP585"]
PERIODS   = {"Near": (2041, 2070), "Far": (2071, 2100)}
PERIOD_NAME = {"Near": "Near future (2041\u20132070)", "Far": "Far future (2071\u20132100)"}

# Ethiopian seasons (edit here if you use a different definition)
SEASONS = {"Bega (Oct\u2013Jan)": [10, 11, 12, 1],
           "Belg (Feb\u2013May)": [2, 3, 4, 5],
           "Kiremt (Jun\u2013Sep)": [6, 7, 8, 9]}
SEASON_ORDER = list(SEASONS)
MONTH_TO_SEASON = {m: s for s, ms in SEASONS.items() for m in ms}

C_OBS, C_NEAR, C_FAR = "#4D4D4D", "#F0A33A", "#5B9BD5"

plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 12, "axes.labelsize": 12, "axes.titlesize": 12,
    "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
})

# ---------------------------------------------------------------- load
daily = pd.read_csv(OUT / "daily_projections_all_models.csv", parse_dates=["Date"])
daily["Season"] = daily.Date.dt.month.map(MONTH_TO_SEASON)

STATION_IDS = {0: "Shebe", 1: "Asendabo", 2: "Abelti"}
parts = []
for split in ("train", "val", "test"):
    with np.load(PREPARED_DIR / "historical" / "lookback_030" / f"{split}.npz",
                 allow_pickle=False) as z:
        parts.append(pd.DataFrame({"Date": pd.to_datetime(z["dates"]),
                                   "Station": [STATION_IDS[int(s)] for s in z["station"]],
                                   "Observed": z["Observed"].astype(float)}))
obs = pd.concat(parts, ignore_index=True)
obs["Season"] = obs.Date.dt.month.map(MONTH_TO_SEASON)

# observed seasonal mean per station (baseline)
obs_seas = obs.groupby(["Station", "Season"]).Observed.mean()

# ---------------------------------------------------------------- seasonal means per model
rows = []
for st in STATIONS:
    for sc in SCENARIOS:
        for per, (a, z) in PERIODS.items():
            sub = daily[(daily.Station == st) & (daily.Scenario == sc) &
                        (daily.Date.dt.year.between(a, z))]
            # keep only dates on which all five models exist (like-for-like)
            cnt = sub.groupby("Date").Model.nunique()
            sub = sub[sub.Date.isin(cnt[cnt == len(MODELS)].index)]
            m = sub.groupby(["Model", "Season"]).Hybrid.mean()
            for season in SEASON_ORDER:
                vals = m.xs(season, level="Season").reindex(MODELS)     # one value per model
                base = obs_seas[(st, season)]
                rows.append({"Station": st, "Scenario": sc, "Period": per, "Season": season,
                             "Observed_mean": base,
                             "Ens_mean": vals.mean(), "Model_min": vals.min(), "Model_max": vals.max(),
                             "Ens_change_%": 100 * (vals.mean() - base) / base,
                             "Model_min_change_%": 100 * (vals.min() - base) / base,
                             "Model_max_change_%": 100 * (vals.max() - base) / base})
tab = pd.DataFrame(rows)
tab.round(2).to_csv(FIG_DIR / "Seasonal_comparison_summary.csv", index=False)
print(tab.round(1).to_string(index=False))

# ---------------------------------------------------------------- Fig A: seasonal mean flow
def grid(title_prefix):
    fig, axes = plt.subplots(len(SCENARIOS), len(STATIONS), figsize=(15, 8.5),
                             constrained_layout=True)
    return fig, axes

fig, axes = grid("A"); k = 0
x = np.arange(len(SEASON_ORDER)); w = 0.26
for i, sc in enumerate(SCENARIOS):
    for j, st in enumerate(STATIONS):
        ax = axes[i, j]
        t = tab[(tab.Station == st) & (tab.Scenario == sc)]
        ob = [obs_seas[(st, s)] for s in SEASON_ORDER]
        ax.bar(x - w, ob, w, color=C_OBS, edgecolor="black", linewidth=0.5)
        for off, per, col in ((0, "Near", C_NEAR), (w, "Far", C_FAR)):
            tp = t[t.Period == per].set_index("Season").reindex(SEASON_ORDER)
            ax.bar(x + off, tp.Ens_mean, w, color=col, edgecolor="black", linewidth=0.5,
                   yerr=[tp.Ens_mean - tp.Model_min, tp.Model_max - tp.Ens_mean],
                   error_kw=dict(ecolor="black", elinewidth=0.9, capsize=3))
        ax.set_xticks(x); ax.set_xticklabels([s.replace(" (", "\n(") for s in SEASON_ORDER])
        ax.set_title(f"({'abcdef'[k]}) {st} \u2013 {sc}", loc="left", fontweight="bold"); k += 1
        if j == 0: ax.set_ylabel("Seasonal mean streamflow")
handles = [Patch(facecolor=C_OBS, edgecolor="black", label="Observed (historical)"),
           Patch(facecolor=C_NEAR, edgecolor="black", label=PERIOD_NAME["Near"] + ", ensemble mean"),
           Patch(facecolor=C_FAR, edgecolor="black", label=PERIOD_NAME["Far"] + ", ensemble mean"),
           Line2D([0], [0], color="k", marker="_", markersize=9, lw=1, label="Min\u2013max across models")]
fig.legend(handles=handles, loc="outside lower center", ncol=4, frameon=False)
fig.savefig(FIG_DIR / "Fig_seasonal_mean_flow.png", dpi=600, bbox_inches="tight")
plt.show(); plt.close(fig)

# ---------------------------------------------------------------- Fig B: % change vs observed
fig, axes = grid("B"); k = 0
w = 0.36
for i, sc in enumerate(SCENARIOS):
    for j, st in enumerate(STATIONS):
        ax = axes[i, j]
        t = tab[(tab.Station == st) & (tab.Scenario == sc)]
        for off, per, col in ((-w / 2, "Near", C_NEAR), (w / 2, "Far", C_FAR)):
            tp = t[t.Period == per].set_index("Season").reindex(SEASON_ORDER)
            ax.bar(x + off, tp["Ens_change_%"], w, color=col, edgecolor="black", linewidth=0.5,
                   yerr=[tp["Ens_change_%"] - tp["Model_min_change_%"],
                         tp["Model_max_change_%"] - tp["Ens_change_%"]],
                   error_kw=dict(ecolor="black", elinewidth=0.9, capsize=3))
        ax.axhline(0, color="black", lw=0.9)
        ax.set_xticks(x); ax.set_xticklabels([s.replace(" (", "\n(") for s in SEASON_ORDER])
        ax.set_title(f"({'abcdef'[k]}) {st} \u2013 {sc}", loc="left", fontweight="bold"); k += 1
        if j == 0: ax.set_ylabel("Change from observed seasonal mean (%)")
handles = [Patch(facecolor=C_NEAR, edgecolor="black", label=PERIOD_NAME["Near"] + ", ensemble mean"),
           Patch(facecolor=C_FAR, edgecolor="black", label=PERIOD_NAME["Far"] + ", ensemble mean"),
           Line2D([0], [0], color="k", marker="_", markersize=9, lw=1, label="Min\u2013max across models")]
fig.legend(handles=handles, loc="outside lower center", ncol=3, frameon=False)
fig.savefig(FIG_DIR / "Fig_seasonal_change_vs_observed.png", dpi=600, bbox_inches="tight")
plt.show(); plt.close(fig)

print(f"\nSaved to {FIG_DIR}")

In [ ]:

# =============================================================================
# RAW DAILY FUTURE STREAMFLOW, ALL STATIONS

# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from pathlib import Path

# ---------------------------------------------------------------- PATHS

DATA_DIR = PROJECT_DIR / "data"
OUT      = RESULTS_ROOT / "Projection_v4"
PLOT_DIR = OUT / "daily_flow_plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- OPTIONS

LOG_Y = False

MODELS = ["LSTM", "GRU", "TCN", "MLP", "LSTM_FCN"]

MODEL_LABEL = {
    "LSTM_FCN": "LSTM-FCN"
}

STATIONS = ["Shebe", "Asendabo", "Abelti"]

SCENARIOS = ["SSP245", "SSP585"]

PERIODS = {
    "Near": (2041, 2070),
    "Far":  (2071, 2100)
}

PERIOD_NAME = {
    "Near": "Near future",
    "Far":  "Far future"
}

# ---------------------------------------------------------------- COLOR PALETTE

# Future-period background colors
PERIOD_FILL = {
    "Near": "#FFF0C2",   # warm champagne / golden cream
    "Far":  "#DCEBFA"    # pale blue
}

# Individual model colors
COLORS = {
    "SWAT":     "#777777",  # neutral grey
    "LSTM":     "#2468A2",  # strong blue
    "GRU":      "#008C95",  # vivid teal
    "TCN":      "#32965C",  # emerald green
    "MLP":      "#A05AA1",  # purple
    "LSTM_FCN": "#D46A32"   # burnt orange
}

# Ensemble mean color
ENS_COLOR = "#8B0000"

# ---------------------------------------------------------------- PLOT SETTINGS

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.9,
    "figure.facecolor": "white",
    "axes.facecolor": "white"
})

# ---------------------------------------------------------------- LOAD DAILY DATA

daily = pd.read_csv(
    OUT / "daily_projections_all_models.csv",
    parse_dates=["Date"]
)

# Ensure numeric flow values
daily["SWAT"] = pd.to_numeric(daily["SWAT"], errors="coerce")
daily["Hybrid"] = pd.to_numeric(daily["Hybrid"], errors="coerce")

# ---------------------------------------------------------------- DAILY SERIES FUNCTION

def series(station, scenario):
    """
    Return original daily streamflow values for:
        SWAT
        LSTM
        GRU
        TCN
        MLP
        LSTM_FCN
        Ensemble

    No smoothing or temporal averaging is applied.

    The ensemble is the arithmetic mean of the five hybrid
    models on each date, requiring all five values to be present.
    """

    sub = daily[
        (daily["Station"] == station) &
        (daily["Scenario"] == scenario)
    ].copy()

    if sub.empty:
        raise ValueError(
            f"No data found for {station} under {scenario}."
        )

    # SWAT daily flow
    out = (
        sub.groupby("Date")["SWAT"]
        .mean()
        .to_frame("SWAT")
    )

    # Individual hybrid model daily flows
    for model in MODELS:

        model_data = (
            sub[sub["Model"] == model]
            .set_index("Date")["Hybrid"]
        )

        # Avoid silently accepting duplicate dates
        if model_data.index.has_duplicates:
            raise ValueError(
                f"Duplicate dates found for {station}, "
                f"{scenario}, {model}."
            )

        out[model] = model_data

    # Daily ensemble mean across the five hybrid models
    out["Ensemble"] = out[MODELS].mean(
        axis=1,
        skipna=False
    )

    # Sort chronologically; do not smooth or resample
    out = out.sort_index()

    return out


# ---------------------------------------------------------------- PREPARE ALL STATIONS AND SCENARIOS

data = {
    (station, scenario): series(station, scenario)
    for station in STATIONS
    for scenario in SCENARIOS
}

# ---------------------------------------------------------------- CREATE FIGURE

fig, axes = plt.subplots(
    nrows=len(STATIONS),
    ncols=len(SCENARIOS),
    figsize=(15, 10),
    sharey="row",
    sharex=True,
    constrained_layout=True
)

# Overall time limits
x0 = min(df.index.min() for df in data.values())
x1 = max(df.index.max() for df in data.values())

# ---------------------------------------------------------------- PLOT EACH PANEL

for i, station in enumerate(STATIONS):

    for j, scenario in enumerate(SCENARIOS):

        ax = axes[i, j]

        df = data[(station, scenario)]

        # ------------------------------------------------------------
        # 1. SHADED NEAR AND FAR FUTURE PERIODS
        # ------------------------------------------------------------

        for period, (start_year, end_year) in PERIODS.items():

            ax.axvspan(
                pd.Timestamp(start_year, 1, 1),
                pd.Timestamp(end_year, 12, 31),
                color=PERIOD_FILL[period],
                alpha=0.65,
                linewidth=0,
                zorder=0
            )

        # ------------------------------------------------------------
        # 2. SWAT DAILY STREAMFLOW
        # ------------------------------------------------------------

        if df["SWAT"].notna().any():

            ax.plot(
                df.index,
                df["SWAT"],
                color=COLORS["SWAT"],
                linewidth=1.0,
                alpha=0.85,
                zorder=2
            )

        # ------------------------------------------------------------
        # 3. FIVE HYBRID MODELS -- ORIGINAL DAILY VALUES
        # ------------------------------------------------------------

        for model in MODELS:

            if df[model].notna().any():

                ax.plot(
                    df.index,
                    df[model],
                    color=COLORS[model],
                    linewidth=0.75,
                    alpha=0.85,
                    zorder=3
                )

        # ------------------------------------------------------------
        # 4. DAILY ENSEMBLE MEAN
        # ------------------------------------------------------------

        ax.plot(
            df.index,
            df["Ensemble"],
            color=ENS_COLOR,
            linewidth=2.0,
            alpha=1.0,
            zorder=5
        )

        # ------------------------------------------------------------
        # 5. NEAR AND FAR PERIOD LABELS -- TOP ROW ONLY
        # ------------------------------------------------------------

        if i == 0:

            for period, (start_year, end_year) in PERIODS.items():

                midpoint = (
                    pd.Timestamp(start_year, 1, 1)
                    +
                    (
                        pd.Timestamp(end_year, 12, 31)
                        -
                        pd.Timestamp(start_year, 1, 1)
                    ) / 2
                )

                ax.text(
                    midpoint,
                    1.025,
                    f"{PERIOD_NAME[period]} "
                    f"({start_year}\u2013{end_year})",
                    transform=ax.get_xaxis_transform(),
                    horizontalalignment="center",
                    verticalalignment="bottom",
                    fontsize=11,
                    fontweight="bold",
                    color="#333333"
                )

            ax.set_title(
                scenario,
                fontweight="bold",
                fontsize=14,
                pad=28
            )

        # ------------------------------------------------------------
        # 6. AXIS FORMATTING
        # ------------------------------------------------------------

        ax.set_xlim(x0, x1)

        if LOG_Y:
            ax.set_yscale("log")

        ax.grid(
            axis="y",
            linestyle="--",
            linewidth=0.55,
            alpha=0.35,
            zorder=1
        )

        ax.xaxis.set_major_locator(
            mdates.YearLocator(10)
        )

        ax.xaxis.set_major_formatter(
            mdates.DateFormatter("%Y")
        )

        # Station labels on left-hand panels
        if j == 0:

            ax.set_ylabel(
                f"{station}\nStreamflow",
                labelpad=10
            )

        # X-axis labels on bottom row
        if i == len(STATIONS) - 1:

            ax.set_xlabel(
                "Year",
                fontweight="bold"
            )

# ---------------------------------------------------------------- LEGEND

handles = [
    Line2D(
        [0], [0],
        color=COLORS["SWAT"],
        linewidth=1.6,
        label="SWAT"
    )
]

for model in MODELS:

    handles.append(
        Line2D(
            [0], [0],
            color=COLORS[model],
            linewidth=1.8,
            label=MODEL_LABEL.get(model, model)
        )
    )

handles.extend([

    Line2D(
        [0], [0],
        color=ENS_COLOR,
        linewidth=2.8,
        label="Ensemble mean"
    ),

    Patch(
        facecolor=PERIOD_FILL["Near"],
        edgecolor="#C9A64B",
        alpha=0.8,
        label="Near future"
    ),

    Patch(
        facecolor=PERIOD_FILL["Far"],
        edgecolor="#719BC5",
        alpha=0.8,
        label="Far future"
    )
])

fig.legend(
    handles=handles,
    loc="outside lower center",
    ncol=5,
    frameon=False,
    columnspacing=1.8,
    handlelength=2.5
)

# ---------------------------------------------------------------- SAVE FIGURES

tag = "raw_daily" + ("_log" if LOG_Y else "")

output_png = (
    PLOT_DIR /
    f"daily_flow_future_all_stations_{tag}.png"
)

output_pdf = (
    PLOT_DIR /
    f"daily_flow_future_all_stations_{tag}.pdf"
)

fig.savefig(
    output_png,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    output_pdf,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

# ---------------------------------------------------------------- CONFIRM OUTPUT

print("=" * 70)
print("RAW DAILY STREAMFLOW FIGURE GENERATED SUCCESSFULLY")
print("=" * 70)
print(f"Temporal smoothing: NONE")
print(f"Temporal averaging: NONE")
print(f"Ensemble: Daily mean of {len(MODELS)} hybrid models")
print(f"Stations: {', '.join(STATIONS)}")
print(f"Scenarios: {', '.join(SCENARIOS)}")
print(f"PNG: {output_png}")
print(f"PDF: {output_pdf}")


# EXTRAPOLATION BEYOND TRAINING ENVELOPE AND STRUCTURAL DIVERGENCE

In [ ]:
# =============================================================================
# EXTRAPOLATION BEYOND TRAINING ENVELOPE AND STRUCTURAL DIVERGENCE

# =============================================================================

import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")

# =============================================================================
# 1. PATHS AND SETTINGS
# =============================================================================

DATA_DIR = PROJECT_DIR / "data"

PREPARED_DIR = RESULTS_ROOT / "LSTM_Prepared_v4"
PROJECTION_DIR = RESULTS_ROOT / "Projection_v4"
PERCENT_CHANGE_FILE = PROJECTION_DIR / "percent_change_by_model.csv"

OUTPUT_DIR = PROJECTION_DIR / "Extrapolation_Structural_Divergence"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STATIONS = ["Shebe", "Asendabo", "Abelti"]
SCENARIOS = ["SSP245", "SSP585"]
MODELS = ["LSTM", "GRU", "TCN", "MLP", "LSTM_FCN"]

MODEL_LOOKBACKS = {
    "LSTM": 45,
    "GRU": 90,
    "TCN": 90,
    "MLP": 30,
    "LSTM_FCN": 120,
}

PERIODS = {
    "Near": ("2041-01-01", "2070-12-31"),
    "Far":  ("2071-01-01", "2100-12-31"),
}

LOWER_Q = 0.005
UPPER_Q = 0.995

# =============================================================================
# 2. LOAD SCALER AND FEATURE METADATA
# =============================================================================

scaler_path = PREPARED_DIR / "scaler.pkl"
if not scaler_path.exists():
    raise FileNotFoundError(f"Missing scaler file: {scaler_path}")

with open(scaler_path, "rb") as f:
    saved = pickle.load(f)

metadata = saved["metadata"]
FEATURES = metadata["feature_cols"]
STATION_TO_ID = metadata["station_to_id"]

print("Number of features:", len(FEATURES))

# =============================================================================
# 3. LOADERS
# =============================================================================

def load_historical_training(lookback):
    path = (PREPARED_DIR / "historical"
            / f"lookback_{lookback:03d}" / "train.npz")
    if not path.exists():
        raise FileNotFoundError(f"Historical training file missing:\n{path}")

    with np.load(path, allow_pickle=False) as z:
        X = z["X"].astype(np.float64)
        station = z["station"].astype(int)
        dates = pd.to_datetime(z["dates"])

    df = pd.DataFrame(X[:, -1, :], columns=FEATURES)
    df["Station_ID"] = station
    df["Date"] = dates
    return df


def load_future_data(scenario, station, lookback):
    path = (PREPARED_DIR / "future" / scenario
            / f"lookback_{lookback:03d}" / f"{station}.npz")
    if not path.exists():
        raise FileNotFoundError(f"Future input file missing:\n{path}")

    with np.load(path, allow_pickle=False) as z:
        X = z["X"].astype(np.float64)
        dates = pd.to_datetime(z["dates"])

    df = pd.DataFrame(X[:, -1, :], columns=FEATURES)
    df["Date"] = dates
    return df

# =============================================================================
# 4. HISTORICAL TRAINING ENVELOPES
# =============================================================================

historical_cache = {}
envelope_rows = []

for model, lookback in MODEL_LOOKBACKS.items():
    if lookback not in historical_cache:
        historical_cache[lookback] = load_historical_training(lookback)
    train_df = historical_cache[lookback]

    for station in STATIONS:
        train_station = train_df[
            train_df["Station_ID"] == STATION_TO_ID[station]
        ]
        if train_station.empty:
            raise ValueError(f"No training data for {station}, "
                             f"lookback={lookback}")

        for feature in FEATURES:
            values = train_station[feature].dropna().to_numpy()
            envelope_rows.append({
                "Model": model,
                "Lookback": lookback,
                "Station": station,
                "Feature": feature,
                "Lower_Bound": np.quantile(values, LOWER_Q),
                "Upper_Bound": np.quantile(values, UPPER_Q),
                "Training_N": len(values),
            })

envelope_df = pd.DataFrame(envelope_rows)
envelope_df.to_csv(OUTPUT_DIR / "historical_training_envelopes.csv",
                   index=False)

# =============================================================================
# 5. FUTURE FEATURE EXCEEDANCE
# =============================================================================

feature_rows = []
overall_rows = []

for model, lookback in MODEL_LOOKBACKS.items():
    print(f"Processing {model} | lookback={lookback}")

    for scenario in SCENARIOS:
        for station in STATIONS:

            future_df = load_future_data(scenario, station, lookback)

            env = envelope_df[
                (envelope_df["Model"] == model)
                & (envelope_df["Station"] == station)
            ].set_index("Feature")

            for period, (start, end) in PERIODS.items():

                fp = future_df[
                    (future_df["Date"] >= pd.Timestamp(start))
                    & (future_df["Date"] <= pd.Timestamp(end))
                ]
                if fp.empty:
                    print(f"WARNING: no data for {model}, {station}, "
                          f"{scenario}, {period}")
                    continue

                rates = []
                for feature in FEATURES:
                    lower = env.loc[feature, "Lower_Bound"]
                    upper = env.loc[feature, "Upper_Bound"]
                    values = fp[feature].to_numpy()
                    outside = (values < lower) | (values > upper)
                    n_total = len(values)
                    pct = 100.0 * outside.sum() / n_total
                    rates.append(pct)

                    feature_rows.append({
                        "Model": model,
                        "Lookback": lookback,
                        "Station": station,
                        "Scenario": scenario,
                        "Period": period,
                        "Feature": feature,
                        "N_Future": n_total,
                        "N_Outside": int(outside.sum()),
                        "Exceedance_%": pct,
                        "Lower_Training_Bound": lower,
                        "Upper_Training_Bound": upper,
                    })

                overall_rows.append({
                    "Model": model,
                    "Lookback": lookback,
                    "Station": station,
                    "Scenario": scenario,
                    "Period": period,
                    "Mean_Feature_Exceedance_%": np.nanmean(rates),
                    "Max_Feature_Exceedance_%": np.nanmax(rates),
                    "N_Features": len(rates),
                })

feature_df = pd.DataFrame(feature_rows)
overall_df = pd.DataFrame(overall_rows)

feature_df.to_csv(OUTPUT_DIR / "feature_level_extrapolation.csv", index=False)
overall_df.to_csv(OUTPUT_DIR / "overall_extrapolation_by_model.csv",
                  index=False)

# =============================================================================
# 6. SUMMARIES ACROSS ARCHITECTURES
# =============================================================================

extrapolation_summary = (
    feature_df
    .groupby(["Station", "Scenario", "Period", "Feature"], as_index=False)
    .agg(
        Mean_Exceedance=("Exceedance_%", "mean"),
        SD_Exceedance=("Exceedance_%", "std"),
        Minimum=("Exceedance_%", "min"),
        Maximum=("Exceedance_%", "max"),
    )
)

overall_summary = (
    overall_df
    .groupby(["Station", "Scenario", "Period"], as_index=False)
    .agg(
        Mean_Extrapolation=("Mean_Feature_Exceedance_%", "mean"),
        SD_Extrapolation=("Mean_Feature_Exceedance_%", "std"),
        Maximum_Extrapolation=("Mean_Feature_Exceedance_%", "max"),
    )
)

extrapolation_summary.to_csv(OUTPUT_DIR / "extrapolation_feature_summary.csv",
                             index=False)
overall_summary.to_csv(OUTPUT_DIR / "extrapolation_overall_summary.csv",
                       index=False)

# =============================================================================
# 7. STRUCTURAL DIVERGENCE
# =============================================================================

if not PERCENT_CHANGE_FILE.exists():
    raise FileNotFoundError(f"Missing projection summary:\n"
                            f"{PERCENT_CHANGE_FILE}")

projection_df = pd.read_csv(PERCENT_CHANGE_FILE)

required_columns = ["Station", "Model", "Scenario", "Period", "mean_change_%"]
missing = [c for c in required_columns if c not in projection_df.columns]
if missing:
    raise ValueError(f"Missing columns in percent_change_by_model.csv: "
                     f"{missing}")

projection_df = projection_df[projection_df["Model"].isin(MODELS)].copy()

structural_df = (
    projection_df
    .groupby(["Station", "Scenario", "Period"], as_index=False)
    .agg(
        Architecture_Min=("mean_change_%", "min"),
        Architecture_Max=("mean_change_%", "max"),
        Architecture_Mean=("mean_change_%", "mean"),
        Architecture_SD=("mean_change_%", "std"),
        N_Architectures=("mean_change_%", "count"),
    )
)
structural_df["Architecture_Range"] = (
    structural_df["Architecture_Max"] - structural_df["Architecture_Min"]
)
structural_df.to_csv(OUTPUT_DIR / "structural_divergence_summary.csv",
                     index=False)

combined = structural_df.merge(
    overall_summary, on=["Station", "Scenario", "Period"], how="inner"
)
combined.to_csv(OUTPUT_DIR / "extrapolation_structural_divergence.csv",
                index=False)

print(combined[["Station", "Scenario", "Period", "Mean_Extrapolation",
                "Architecture_Range", "Architecture_SD"]]
      .round(2).to_string(index=False))

pearson = combined[["Mean_Extrapolation", "Architecture_Range"]] \
    .corr(method="pearson").iloc[0, 1]
spearman = combined[["Mean_Extrapolation", "Architecture_Range"]] \
    .corr(method="spearman").iloc[0, 1]

print(f"\nPearson r = {pearson:.3f} | Spearman rho = {spearman:.3f} "
      f"(n = {len(combined)})")

# =============================================================================
# 8. FIGURE STYLE
# =============================================================================

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 14,
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "axes.linewidth": 0.9,
    "figure.dpi": 300,
    "savefig.dpi": 600,
})

# Heatmap colormap: cream -> gold -> orange -> crimson -> deep plum
heat_cmap = LinearSegmentedColormap.from_list(
    "extrapolation",
    ["#FFFDF3", "#FFF0B3", "#FFD166", "#F9A03F",
     "#E4572E", "#B3173D", "#5B0A3A"],
    N=256,
)

# Colour-blind-safe scenario colours (Okabe-Ito)
SCENARIO_COLORS = {"SSP245": "#0072B2", "SSP585": "#D55E00"}
STATION_MARKERS = {"Shebe": "o", "Asendabo": "s", "Abelti": "^"}
PERIOD_SIZE = {"Near": 120, "Far": 190}

# Individual label positions for panel (b): (dx, dy) in points.
# Negative dx -> label is placed to the left of the marker (right-aligned).
LABEL_OFFSETS = {
    # Abelti
    ("Abelti", "SSP585", "Far"):  (14, 2),
    ("Abelti", "SSP245", "Far"):  (16, -5),
    ("Abelti", "SSP245", "Near"): (21, -8),
    ("Abelti", "SSP585", "Near"): (12, 0),

    # Shebe
    ("Shebe", "SSP585", "Near"):  (12, 0),
    ("Shebe", "SSP585", "Far"):   (0, -17, "center"),   # below the point
    ("Shebe", "SSP245", "Near"):  (12, -12),
    ("Shebe", "SSP245", "Far"):   (12, 11),

    # Asendabo
    ("Asendabo", "SSP585", "Near"): (0, 17, "center"),  # above the point
    ("Asendabo", "SSP585", "Far"):  (12, 0),
    ("Asendabo", "SSP245", "Near"): (12, 0),
    ("Asendabo", "SSP245", "Far"):  (12, 10),
}

fig, axes = plt.subplots(
    nrows=1, ncols=2, figsize=(19, 8),
    gridspec_kw={"width_ratios": [1.65, 1.0]},
)

# -----------------------------------------------------------------------------
# PANEL (a) -- HEATMAP
# -----------------------------------------------------------------------------

ax = axes[0]

row_order = [(st, sc, pe) for st in STATIONS for sc in SCENARIOS
             for pe in ["Near", "Far"]]

heatmap_data = np.asarray([
    extrapolation_summary[
        (extrapolation_summary["Station"] == st)
        & (extrapolation_summary["Scenario"] == sc)
        & (extrapolation_summary["Period"] == pe)
    ].set_index("Feature")["Mean_Exceedance"].reindex(FEATURES).to_numpy()
    for st, sc, pe in row_order
])

im = ax.imshow(
    heatmap_data, aspect="auto", cmap=heat_cmap,
    vmin=0, vmax=100, interpolation="nearest",
)

# White cell borders
ax.set_xticks(np.arange(-0.5, len(FEATURES), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(row_order), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=0.8)
ax.tick_params(which="minor", length=0)

ax.set_xticks(np.arange(len(FEATURES)))
ax.set_xticklabels(FEATURES, rotation=65, ha="right")
ax.set_yticks(np.arange(len(row_order)))
ax.set_yticklabels([f"{st} | {sc} | {pe}" for st, sc, pe in row_order])

ax.set_title("(a) Future input extrapolation beyond the training envelope",
             loc="left", pad=12)
ax.set_xlabel("Input features")
ax.set_ylabel("Station | Climate scenario | Projection period")

cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
cbar.set_label("Future values outside training envelope (%)")
cbar.outline.set_linewidth(0.8)

# Station separators
for y in [3.5, 7.5]:
    ax.axhline(y, color="#222222", linewidth=1.6)

# Cell annotations (low values muted so the strong departures stand out)
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        v = heatmap_data[i, j]
        if not np.isfinite(v):
            continue
        if v > 50:
            color, weight = "white", "bold"
        elif v >= 10:
            color, weight = "#1a1a1a", "bold"
        else:
            color, weight = "#8a8a8a", "normal"
        ax.text(j, i, f"{v:.0f}", ha="center", va="center",
                fontsize=9, color=color, fontweight=weight)

# -----------------------------------------------------------------------------
# PANEL (b) -- EXTRAPOLATION VS ARCHITECTURAL DIVERGENCE
# -----------------------------------------------------------------------------

ax = axes[1]

# Dashed least-squares guide line
x_all = combined["Mean_Extrapolation"].to_numpy()
y_all = combined["Architecture_Range"].to_numpy()
slope, intercept = np.polyfit(x_all, y_all, 1)
xx = np.linspace(2.0, 10.0, 100)
ax.plot(xx, slope * xx + intercept, color="#6b6b6b", linestyle="--",
        linewidth=1.1, alpha=0.8, zorder=1)

for _, row in combined.iterrows():
    station, scenario, period = row["Station"], row["Scenario"], row["Period"]
    x, y = row["Mean_Extrapolation"], row["Architecture_Range"]

    ax.scatter(
        x, y,
        marker=STATION_MARKERS[station],
        color=SCENARIO_COLORS[scenario],
        s=PERIOD_SIZE[period],
        edgecolor="black", linewidth=0.9, alpha=0.92, zorder=3,
    )

    label = f"{station[:3]}-{scenario[-3:]}-{period[0]}"
    off = LABEL_OFFSETS[(station, scenario, period)]
    dx, dy = off[0], off[1]
    ha = off[2] if len(off) > 2 else ("left" if dx >= 0 else "right")
    ax.annotate(
        label, (x, y), xytext=(dx, dy),
        textcoords="offset points",
        fontsize=13, ha=ha, va="center", zorder=4,
    )

ax.set_xlim(2.0, 10.0)
ax.set_ylim(10, 185)

ax.set_xlabel("Mean feature-level extrapolation burden (%)")
ax.set_ylabel("Inter-architecture range in projected mean-flow change\n"
              "(percentage points)")
ax.set_title("(b) Association between extrapolation and structural divergence",
             loc="left", pad=12)

ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.35, zorder=0)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)

# Correlation note
ax.text(
    0.98, 0.60,
    f"Pearson $r$ = {pearson:.2f}\nSpearman $\\rho$ = {spearman:.2f}\n"
    f"$n$ = {len(combined)}",
    transform=ax.transAxes, ha="right", va="center", fontsize=14,
    bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
              edgecolor="#bbbbbb", linewidth=0.8),
)

# Legends
scenario_handles = [
    Line2D([0], [0], marker="o", color="w",
           markerfacecolor=SCENARIO_COLORS[s], markeredgecolor="black",
           markersize=9, label=s)
    for s in SCENARIOS
]
station_handles = [
    Line2D([0], [0], marker=STATION_MARKERS[s], color="black",
           linestyle="None", markersize=8, label=s)
    for s in STATIONS
]
period_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#bdbdbd",
           markeredgecolor="black", markersize=np.sqrt(PERIOD_SIZE[p]) * 0.85,
           label=f"{p} (N)" if p == "Near" else f"{p} (F)")
    for p in ["Near", "Far"]
]

leg1 = ax.legend(handles=scenario_handles, title="Scenario",
                 loc="upper left", frameon=True, edgecolor="#bbbbbb")
ax.add_artist(leg1)
leg2 = ax.legend(handles=period_handles, title="Period",
                 loc="lower center", frameon=True, bbox_to_anchor=(0.55, 0.0), edgecolor="#bbbbbb")
ax.add_artist(leg2)
ax.legend(handles=station_handles, title="Station",
          loc="lower right", frameon=True, edgecolor="#bbbbbb")

# =============================================================================
# 9. EXPORT
# =============================================================================

plt.tight_layout()

png_path = OUTPUT_DIR / "Figure_Extrapolation_Structural_Divergence.png"
plt.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white")

plt.show()

print("PNG:", png_path)

In [ ]:
# =============================================================================
# PROJECTED STREAMFLOW CHANGES UNDER SSP245 AND SSP585
#
# Mean streamflow
# High flows (Q5) and low flows (Q95)
# Annual maximum streamflow
#
# For each metric this script produces
#   (1) a heatmap figure   : architecture x period, stations x scenarios
#   (2) a grouped-bar figure: same layout, bars per architecture
#   (3) summary tables for the Results text:
#         - direction agreement among the five architectures
#         - near -> far difference
#         - SSP585 minus SSP245 difference
#
# =============================================================================

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")

# =============================================================================
# 1. SETTINGS  (edit here)
# =============================================================================

DATA_DIR = PROJECT_DIR / "data"
PROJECTION_DIR = RESULTS_ROOT / "Projection_v4"
INPUT_FILE = PROJECTION_DIR / "percent_change_by_model.csv"

OUTPUT_DIR = PROJECTION_DIR / "Section_3_3_Projected_Changes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STATIONS = ["Shebe", "Asendabo", "Abelti"]
SCENARIOS = ["SSP245", "SSP585"]
PERIODS = ["Near", "Far"]
PERIOD_LABELS = {
    "Near": "Near future\n(2041-2070)",
    "Far": "Far future\n(2071-2100)",
}

# The five hybrid architectures as named in the CSV "Model" column.
DL_MODELS_RAW = ["LSTM", "GRU", "TCN", "MLP", "LSTM_FCN"]

# Metric -> column name in the CSV.
# Leave as None to auto-detect; set an exact column name to override.
# Convention: Q5 = flow exceeded 5% of the time (high flow),
#             Q95 = flow exceeded 95% of the time (low flow).
# If your file uses the opposite naming, swap the two entries below.
METRICS = {
    "mean":   {"label": "Mean flow",             "column": "mean_change_%"},
    "q5":     {"label": "High flow (Q5)",        "column": None},
    "q95":    {"label": "Low flow (Q95)",        "column": None},
    "annmax": {"label": "Annual maximum flow",   "column": None},
}

# Display order in the heatmaps
ORDER = ["LSTM", "GRU", "TCN", "MLP", "LSTM-FCN", "SWAT", "Ensemble mean"]

# Colours for the grouped-bar figures (Okabe-Ito, colour-blind safe)
MODEL_COLORS = {
    "LSTM":     "#0072B2",
    "GRU":      "#D55E00",
    "TCN":      "#009E73",
    "MLP":      "#CC79A7",
    "LSTM-FCN": "#E69F00",
    "SWAT":     "#9a9a9a",
}

# =============================================================================
# 2. LOAD AND CLEAN
# =============================================================================

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Missing file:\n{INPUT_FILE}")

raw = pd.read_csv(INPUT_FILE)
print("Columns found in the CSV:\n ", list(raw.columns), "\n")

for c in ["Station", "Model", "Scenario", "Period"]:
    if c not in raw.columns:
        raise ValueError(f"Required column '{c}' not found.")


def display_name(name):
    n = str(name).strip()
    low = n.lower().replace("_", "-").replace(" ", "-")
    if "swat" in low:
        return "SWAT"
    if "ensemble" in low:
        return "Ensemble mean"
    if low in ("lstm-fcn", "lstmfcn"):
        return "LSTM-FCN"
    return n.upper()


def clean_scenario(s):
    return re.sub(r"[^A-Z0-9]", "", str(s).upper())


def clean_period(p):
    p = str(p).lower()
    if "near" in p:
        return "Near"
    if "far" in p:
        return "Far"
    return str(p)


raw["Model_disp"] = raw["Model"].map(display_name)
raw["Scenario"] = raw["Scenario"].map(clean_scenario)
raw["Period"] = raw["Period"].map(clean_period)
raw["Station"] = raw["Station"].astype(str).str.strip()

DL_MODELS = [display_name(m) for m in DL_MODELS_RAW]

print("Models in file :", sorted(raw["Model_disp"].unique()))
print("Scenarios      :", sorted(raw["Scenario"].unique()))
print("Periods        :", sorted(raw["Period"].unique()), "\n")


def autodetect_column(metric_key, columns):
    pats = {
        "mean":   r"mean.*change|change.*mean",
        "q5":     r"q_?5(?!\d)",
        "q95":    r"q_?95",
        "annmax": r"annual|ann_?max|a_?max|peak",
    }
    for c in columns:
        if re.search(pats[metric_key], c.lower()):
            return c
    return None


for key, info in METRICS.items():
    if info["column"] is None:
        info["column"] = autodetect_column(key, raw.columns)
    status = info["column"] if info["column"] else "NOT FOUND (will be skipped)"
    print(f"{info['label']:<22} -> {status}")

# =============================================================================
# 3. TIDY TABLE PER METRIC (with ensemble mean)
# =============================================================================

def build_tidy(column):
    t = raw[["Station", "Scenario", "Period", "Model_disp", column]].copy()
    t = t.rename(columns={"Model_disp": "Model", column: "Value"})
    t = t[t["Station"].isin(STATIONS) & t["Scenario"].isin(SCENARIOS)]

    # Ensemble mean = mean of the five architecture changes,
    # unless the file already contains an ensemble row.
    if "Ensemble mean" not in t["Model"].unique():
        ens = (
            t[t["Model"].isin(DL_MODELS)]
            .groupby(["Station", "Scenario", "Period"], as_index=False)["Value"]
            .mean()
        )
        ens["Model"] = "Ensemble mean"
        t = pd.concat([t, ens], ignore_index=True)

    return t


def to_matrix(tidy, station, scenario, models):
    sub = tidy[(tidy["Station"] == station) & (tidy["Scenario"] == scenario)]
    piv = sub.pivot_table(index="Model", columns="Period", values="Value")
    piv = piv.reindex(index=models, columns=PERIODS)
    return piv.to_numpy()

# =============================================================================
# 4. STYLE
# =============================================================================

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.linewidth": 0.9,
    "figure.dpi": 300,
    "savefig.dpi": 600,
})

# =============================================================================
# 5. FIGURE TYPE A -- HEATMAPS
# =============================================================================

def make_heatmap(tidy, label, fname):

    models = [m for m in ORDER if m in tidy["Model"].unique()]

    mats = {
        (st, sc): to_matrix(tidy, st, sc, models)
        for sc in SCENARIOS for st in STATIONS
    }
    allv = np.concatenate([m.ravel() for m in mats.values()])
    vmin = min(-1.0, np.nanmin(allv))
    vmax = max(1.0, np.nanmax(allv))
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

    fig, axes = plt.subplots(
        len(SCENARIOS), len(STATIONS),
        figsize=(11.5, 3.6 * len(SCENARIOS)),
        sharey=True,
    )

    for r, sc in enumerate(SCENARIOS):
        for c, st in enumerate(STATIONS):
            ax = axes[r, c]
            mat = mats[(st, sc)]

            im = ax.imshow(mat, cmap="RdYlGn", norm=norm, aspect="auto")

            ax.set_xticks(np.arange(len(PERIODS)))
            ax.set_xticklabels([PERIOD_LABELS[p] for p in PERIODS])
            ax.set_yticks(np.arange(len(models)))
            ax.set_yticklabels(models if c == 0 else [])
            ax.set_title(f"{st} ({sc})", loc="left", fontweight="bold")

            # white cell borders
            ax.set_xticks(np.arange(-0.5, len(PERIODS), 1), minor=True)
            ax.set_yticks(np.arange(-0.5, len(models), 1), minor=True)
            ax.grid(which="minor", color="white", linewidth=1.2)
            ax.tick_params(which="minor", length=0)

            # separate SWAT / ensemble rows from the five architectures
            n_dl = sum(m in DL_MODELS for m in models)
            ax.axhline(n_dl - 0.5, color="#222222", linewidth=1.0)

            for i in range(mat.shape[0]):
                for j in range(mat.shape[1]):
                    v = mat[i, j]
                    if np.isfinite(v):
                        ax.text(j, i, f"{v:.0f}%", ha="center", va="center",
                                fontsize=8.5, color="black")

            if c == 0:
                ax.set_ylabel("Hybrid architecture")

    cbar = fig.colorbar(
        im, ax=axes, orientation="horizontal",
        fraction=0.04, pad=0.14, aspect=50,
    )
    cbar.set_label(f"Change in {label.lower()} relative to the historical period (%)")

    fig.savefig(OUTPUT_DIR / f"{fname}_heatmap.png",
                dpi=600, bbox_inches="tight", facecolor="white")
    fig.savefig(OUTPUT_DIR / f"{fname}_heatmap.pdf",
                bbox_inches="tight", facecolor="white")
    plt.close(fig)

# =============================================================================
# 6. FIGURE TYPE B -- GROUPED BARS
# =============================================================================

def make_barplot(tidy, label, fname):

    bar_models = [m for m in DL_MODELS + ["SWAT"]
                  if m in tidy["Model"].unique()]
    n = len(bar_models)
    width = 0.12
    offsets = (np.arange(n) - (n - 1) / 2) * width

    fig, axes = plt.subplots(
        len(SCENARIOS), len(STATIONS),
        figsize=(13, 4.0 * len(SCENARIOS)),
        sharey=True,
    )

    panel_letters = "abcdefghijkl"
    k = 0

    for r, sc in enumerate(SCENARIOS):
        for c, st in enumerate(STATIONS):
            ax = axes[r, c]
            sub = tidy[(tidy["Station"] == st) & (tidy["Scenario"] == sc)]

            for gi, period in enumerate(PERIODS):
                for mi, m in enumerate(bar_models):
                    val = sub[(sub["Model"] == m)
                              & (sub["Period"] == period)]["Value"]
                    if val.empty:
                        continue
                    ax.bar(
                        gi + offsets[mi], val.iloc[0], width=width * 0.92,
                        color=MODEL_COLORS.get(m, "#777777"),
                        edgecolor="black", linewidth=0.6,
                        hatch="//" if m == "SWAT" else None,
                        zorder=3,
                    )

                ens = sub[(sub["Model"] == "Ensemble mean")
                          & (sub["Period"] == period)]["Value"]
                if not ens.empty:
                    ax.scatter(
                        gi, ens.iloc[0], marker="D", s=55,
                        color="white", edgecolor="black", linewidth=1.2,
                        zorder=5,
                    )

            ax.axhline(0, color="black", linewidth=0.9, zorder=4)
            ax.set_xticks(np.arange(len(PERIODS)))
            ax.set_xticklabels([PERIOD_LABELS[p].replace("\n", " ")
                                for p in PERIODS])
            ax.set_xlim(-0.55, len(PERIODS) - 0.45)
            ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.35,
                    zorder=0)
            for side in ["top", "right"]:
                ax.spines[side].set_visible(False)

            ax.set_title(f"({panel_letters[k]}) {st} - {sc}",
                         loc="left", fontweight="bold")
            k += 1

            if c == 0:
                ax.set_ylabel(f"Change in {label.lower()} (%)")

    handles = [
        Patch(facecolor=MODEL_COLORS[m], edgecolor="black",
              hatch="//" if m == "SWAT" else None, label=m)
        for m in bar_models
    ]
    handles.append(
        Line2D([0], [0], marker="D", color="w", markerfacecolor="white",
               markeredgecolor="black", markersize=8,
               label="Ensemble mean")
    )
    fig.legend(handles=handles, loc="lower center", ncol=len(handles),
               frameon=False, bbox_to_anchor=(0.5, -0.02))

    fig.tight_layout(rect=(0, 0.04, 1, 1))
    fig.savefig(OUTPUT_DIR / f"{fname}_bars.png",
                dpi=600, bbox_inches="tight", facecolor="white")
    fig.savefig(OUTPUT_DIR / f"{fname}_bars.pdf",
                bbox_inches="tight", facecolor="white")
    plt.close(fig)

# =============================================================================
# 7. SUMMARY TABLES FOR THE RESULTS TEXT
# =============================================================================

def agreement_label(vals):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    n_inc = int((vals > 0).sum())
    n_dec = int((vals < 0).sum())
    n = len(vals)
    if n_inc == n:
        return f"All {n} increase"
    if n_dec == n:
        return f"All {n} decrease"
    return f"Split ({n_inc} increase / {n_dec} decrease)"


def make_summaries(tidy, label, fname):

    dl = tidy[tidy["Model"].isin(DL_MODELS)]

    # --- (i) Direction agreement among the five architectures -----------------
    rows = []
    for (st, sc, pe), g in dl.groupby(["Station", "Scenario", "Period"]):
        v = g["Value"].to_numpy()
        ens = tidy[(tidy["Station"] == st) & (tidy["Scenario"] == sc)
                   & (tidy["Period"] == pe)
                   & (tidy["Model"] == "Ensemble mean")]["Value"]
        swat = tidy[(tidy["Station"] == st) & (tidy["Scenario"] == sc)
                    & (tidy["Period"] == pe)
                    & (tidy["Model"] == "SWAT")]["Value"]
        rows.append({
            "Station": st, "Scenario": sc, "Period": pe,
            "Ensemble_mean_%": ens.iloc[0] if len(ens) else np.nan,
            "SWAT_%": swat.iloc[0] if len(swat) else np.nan,
            "Arch_min_%": v.min(), "Arch_max_%": v.max(),
            "N_increase": int((v > 0).sum()),
            "N_decrease": int((v < 0).sum()),
            "Agreement": agreement_label(v),
        })
    agree = pd.DataFrame(rows)
    agree["Station"] = pd.Categorical(agree["Station"], STATIONS, ordered=True)
    agree = agree.sort_values(["Station", "Scenario", "Period"])
    agree.to_csv(OUTPUT_DIR / f"{fname}_direction_agreement.csv", index=False)

    # --- (ii) Near -> far difference (Far minus Near, percentage points) ------
    piv = tidy.pivot_table(index=["Station", "Scenario", "Model"],
                           columns="Period", values="Value").reset_index()
    if {"Near", "Far"}.issubset(piv.columns):
        piv["Far_minus_Near_pp"] = piv["Far"] - piv["Near"]
        piv.to_csv(OUTPUT_DIR / f"{fname}_near_far_difference.csv",
                   index=False)

    # --- (iii) SSP585 minus SSP245 (percentage points) ------------------------
    piv2 = tidy.pivot_table(index=["Station", "Period", "Model"],
                            columns="Scenario", values="Value").reset_index()
    if {"SSP245", "SSP585"}.issubset(piv2.columns):
        piv2["SSP585_minus_SSP245_pp"] = piv2["SSP585"] - piv2["SSP245"]
        piv2.to_csv(OUTPUT_DIR / f"{fname}_scenario_difference.csv",
                    index=False)

    print("\n" + "=" * 78)
    print(f"{label.upper()} -- DIRECTION AGREEMENT AMONG ARCHITECTURES")
    print("=" * 78)
    print(agree.round(1).to_string(index=False))

# =============================================================================
# 8. RUN ALL METRICS
# =============================================================================

for key, info in METRICS.items():

    col = info["column"]
    label = info["label"]

    if col is None or col not in raw.columns:
        print(f"\nSKIPPED: {label} -- no matching column in the CSV.")
        continue

    print(f"\nProcessing: {label}  (column: {col})")

    tidy = build_tidy(col)
    fname = f"Fig_3_3_{key}"

    make_heatmap(tidy, label, fname)
    make_barplot(tidy, label, fname)
    make_summaries(tidy, label, fname)

print("\nAll outputs saved in:")
print(OUTPUT_DIR)

In [ ]:
import calendar
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------- PATHS

DATA_DIR = PROJECT_DIR / "data"
OUT = RESULTS_ROOT / "Projection_v4"
ANNUAL_DIR = OUT / "annual_change"
ANNUAL_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- OPTIONS

BASELINE = (2000, 2021)            # historical baseline years (inclusive)
FUTURE_YEARS = (2041, 2100)
PERIODS = {"Near": (2041, 2070), "Far": (2071, 2100)}
PERIOD_NAME = {"Near": "Near future", "Far": "Far future"}

MIN_COVERAGE = 0.95                # a year needs >=95% of its days to count

MODELS = ["LSTM", "GRU", "TCN", "MLP", "LSTM_FCN"]
MODEL_LABEL = {"LSTM_FCN": "LSTM-FCN"}
STATIONS = ["Shebe", "Asendabo", "Abelti"]
SCENARIOS = ["SSP245", "SSP585"]

SERIES = ["SWAT"] + MODELS + ["Ensemble"]   # Ensemble = daily mean of 5 models

PERIOD_FILL = {"Near": "#FDE0B5", "Far": "#CFE3F5"}
COLORS = {
    "SWAT": "#7f7f7f", "LSTM": "#1f77b4", "GRU": "#ff7f0e",
    "TCN": "#2ca02c", "MLP": "#d62728", "LSTM_FCN": "#9467bd",
}
ENS_COLOR = "#111111"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 12, "axes.labelsize": 12, "axes.titlesize": 12,
    "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 11,
    "axes.spines.top": False, "axes.spines.right": False,
})

# ---------------------------------------------------------------- LOAD

daily = pd.read_csv(OUT / "daily_projections_all_models.csv",
                    parse_dates=["Date"])
daily["SWAT"] = pd.to_numeric(daily["SWAT"], errors="coerce")
daily["Hybrid"] = pd.to_numeric(daily["Hybrid"], errors="coerce")


def daily_series(station, scenario):
    """Daily flows: SWAT, five models, Ensemble (needs all five present)."""
    sub = daily[(daily["Station"] == station)
                & (daily["Scenario"] == scenario)]
    if sub.empty:
        raise ValueError(f"No data for {station} / {scenario}")

    out = sub.groupby("Date")["SWAT"].mean().to_frame("SWAT")
    for m in MODELS:
        s = sub[sub["Model"] == m].set_index("Date")["Hybrid"]
        if s.index.has_duplicates:
            raise ValueError(f"Duplicate dates: {station}, {scenario}, {m}")
        out[m] = s
    out["Ensemble"] = out[MODELS].mean(axis=1, skipna=False)
    return out.sort_index()


def annual_means(df, y0, y1, label):
    """Annual mean flow per series; years with too few days are dropped."""
    df = df[(df.index.year >= y0) & (df.index.year <= y1)]
    cols = {}
    for c in SERIES:
        s = df[c].dropna()
        g = s.groupby(s.index.year)
        n = g.count()
        mean = g.mean()
        need = pd.Series(
            {y: MIN_COVERAGE * (366 if calendar.isleap(y) else 365)
             for y in n.index}
        )
        ok = n >= need
        if (~ok).any():
            print(f"  [{label}] {c}: dropped incomplete years "
                  f"{list(n.index[~ok])}")
        cols[c] = mean[ok]
    return pd.DataFrame(cols)


# ---------------------------------------------------------------- BASELINE

print("Computing baseline (%d-%d)..." % BASELINE)
baseline = {}
for st in STATIONS:
    hist = daily_series(st, "Historical")
    ann = annual_means(hist, BASELINE[0], BASELINE[1], f"{st} Historical")
    n_years = ann.notna().sum()
    if n_years.min() < 15:
        print(f"  WARNING {st}: only {n_years.min()} complete baseline years "
              f"between {BASELINE[0]} and {BASELINE[1]}. Historical data "
              f"span: {hist.index.min().date()} to {hist.index.max().date()}")
    baseline[st] = ann.mean()          # mean of annual means, per series

baseline_df = pd.DataFrame(baseline).T
baseline_df.to_csv(ANNUAL_DIR / "baseline_mean_flow_2000_2021.csv")
print(baseline_df.round(3).to_string())

# ---------------------------------------------------------------- ANNUAL CHANGE

rows = []
for st in STATIONS:
    for sc in SCENARIOS:
        fut = daily_series(st, sc)
        ann = annual_means(fut, FUTURE_YEARS[0], FUTURE_YEARS[1],
                           f"{st} {sc}")
        for c in SERIES:
            b = baseline[st][c]
            for year, val in ann[c].dropna().items():
                period = next((p for p, (a, z) in PERIODS.items()
                               if a <= year <= z), None)
                rows.append({
                    "Station": st, "Scenario": sc, "Model": c,
                    "Year": int(year), "Period": period,
                    "Annual_mean_flow": val,
                    "Baseline_mean_flow": b,
                    "Annual_change_%": 100.0 * (val / b - 1.0),
                })

annual = pd.DataFrame(rows)
annual.to_csv(ANNUAL_DIR / "annual_change_by_year.csv", index=False)

# ---------------------------------------------------------------- PERIOD SUMMARY

summ = (
    annual.dropna(subset=["Period"])
    .groupby(["Station", "Scenario", "Model", "Period"], as_index=False)
    .agg(
        N_years=("Year", "count"),
        Mean_flow=("Annual_mean_flow", "mean"),
        Baseline=("Baseline_mean_flow", "first"),
        Mean_annual_change=("Annual_change_%", "mean"),
        SD_annual_change=("Annual_change_%", "std"),
        Min_annual_change=("Annual_change_%", "min"),
        Max_annual_change=("Annual_change_%", "max"),
        N_years_increase=("Annual_change_%", lambda s: int((s > 0).sum())),
    )
)
summ["Period_change_%"] = 100.0 * (summ["Mean_flow"] / summ["Baseline"] - 1.0)
summ["Pct_years_increase"] = 100.0 * summ["N_years_increase"] / summ["N_years"]

summ["Station"] = pd.Categorical(summ["Station"], STATIONS, ordered=True)
summ["Model"] = pd.Categorical(summ["Model"], SERIES, ordered=True)
summ["Period"] = pd.Categorical(summ["Period"], ["Near", "Far"], ordered=True)
summ = summ.sort_values(["Station", "Scenario", "Model", "Period"])
summ.to_csv(ANNUAL_DIR / "annual_change_period_summary.csv", index=False)

wide = summ.pivot_table(
    index=["Station", "Scenario", "Model"], columns="Period",
    values="Period_change_%", observed=True,
).reset_index()
wide.to_csv(ANNUAL_DIR / "annual_change_period_wide.csv", index=False)

print("\nPeriod change in mean annual flow relative to 2000-2021 (%):")
print(wide.round(1).to_string(index=False))

# ---------------------------------------------------------------- FIGURE

fig, axes = plt.subplots(
    len(STATIONS), len(SCENARIOS), figsize=(15, 10),
    sharey="row", sharex=True, constrained_layout=True,
)
letters = "abcdef"
k = 0

for i, st in enumerate(STATIONS):
    for j, sc in enumerate(SCENARIOS):
        ax = axes[i, j]
        sub = annual[(annual["Station"] == st) & (annual["Scenario"] == sc)]

        for per, (a, z) in PERIODS.items():
            ax.axvspan(a - 0.5, z + 0.5, color=PERIOD_FILL[per],
                       alpha=0.75, lw=0, zorder=0)

        ax.axhline(0, color="black", lw=0.9, zorder=1)

        for c in SERIES:
            s = sub[sub["Model"] == c].sort_values("Year")
            if s.empty:
                continue
            if c == "Ensemble":
                ax.plot(s["Year"], s["Annual_change_%"], color=ENS_COLOR,
                        lw=2.0, zorder=5)
            elif c == "SWAT":
                ax.plot(s["Year"], s["Annual_change_%"], color=COLORS[c],
                        lw=1.0, alpha=0.9, zorder=3)
            else:
                ax.plot(s["Year"], s["Annual_change_%"], color=COLORS[c],
                        lw=0.8, alpha=0.6, zorder=2)

        # ensemble period-mean change as dashed level + label
        for per, (a, z) in PERIODS.items():
            e = summ[(summ["Station"] == st) & (summ["Scenario"] == sc)
                     & (summ["Model"] == "Ensemble")
                     & (summ["Period"] == per)]
            if e.empty:
                continue
            v = float(e["Period_change_%"].iloc[0])
            ax.hlines(v, a, z, colors=ENS_COLOR, linestyles="--",
                      lw=1.2, zorder=6)
            ax.text((a + z) / 2, 0.97, f"{v:+.0f}%",
                    transform=ax.get_xaxis_transform(), ha="center",
                    va="top", fontsize=12, fontweight="bold", zorder=7,
                    bbox=dict(facecolor="white", edgecolor="none",
                              alpha=0.8, pad=1.5))

        if i == 0:
            for per, (a, z) in PERIODS.items():
                ax.text((a + z) / 2, 1.02,
                        f"{PERIOD_NAME[per]} ({a}\u2013{z})",
                        transform=ax.get_xaxis_transform(), ha="center",
                        va="bottom", fontsize=11, fontweight="bold")
            ax.set_title(sc, fontweight="bold", pad=28)

        ax.set_xlim(FUTURE_YEARS[0] - 0.5, FUTURE_YEARS[1] + 0.5)
        ax.grid(axis="y", linestyle="--", lw=0.55, alpha=0.35, zorder=1)
        ax.set_xticks(range(2045, 2101, 10))
        if j == 0:
            ax.set_ylabel(f"{st}\nAnnual change in mean flow (%)")
        if i == len(STATIONS) - 1:
            ax.set_xlabel("Year")
        ax.text(0.01, 0.97, f"({letters[k]})", transform=ax.transAxes,
                ha="left", va="top", fontweight="bold", zorder=8)
        k += 1

handles = [Line2D([0], [0], color=COLORS["SWAT"], lw=1.4, label="SWAT")]
handles += [Line2D([0], [0], color=COLORS[m], lw=1.4, alpha=0.8,
                   label=MODEL_LABEL.get(m, m)) for m in MODELS]
handles += [
    Line2D([0], [0], color=ENS_COLOR, lw=2.4, label="Ensemble mean"),
    Line2D([0], [0], color=ENS_COLOR, lw=1.2, ls="--",
           label="Period mean change (ensemble)"),
    Patch(facecolor=PERIOD_FILL["Near"], label="Near future"),
    Patch(facecolor=PERIOD_FILL["Far"], label="Far future"),
]
fig.legend(handles=handles, loc="outside lower center", ncol=5,
           frameon=False)

png = ANNUAL_DIR / "annual_change_timeseries.png"
pdf = ANNUAL_DIR / "annual_change_timeseries.pdf"
fig.savefig(png, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(pdf, bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)

print("\nSaved in:", ANNUAL_DIR)

In [ ]:
# =============================================================================
# MEMORY-HORIZON RESPONSE CURVE  

# =============================================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------- PATHS

DATA_DIR = PROJECT_DIR / "data"

RESULT_DIRS = {
    "LSTM":     RESULTS_ROOT / "LSTM_Results_v4",
    "GRU":      RESULTS_ROOT / "GRU_Results_v4",
    "TCN":      RESULTS_ROOT / "TCN_Results_v4",
    "MLP":      RESULTS_ROOT / "MLP_Results_v4",
    "LSTM_FCN": RESULTS_ROOT / "LSTM_FCN_Results_v4",
}

SELECTION_FILE = (RESULTS_ROOT / "Future_SSP_Results_v4" / "tables"
                  / "Final_historical_model_selection.csv")

OUT_DIR = RESULTS_ROOT / "Memory_Horizon"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL = {"LSTM_FCN": "LSTM-FCN"}
COLORS = {
    "LSTM": "#0072B2", "GRU": "#D55E00", "TCN": "#009E73",
    "MLP": "#CC79A7", "LSTM_FCN": "#E69F00",
}

# Optional manual y-limits, e.g. (0.4, 0.95). None = automatic.
YLIM = (0.50, 0.87)

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10, "axes.labelsize": 11, "axes.titlesize": 11,
    "legend.fontsize": 9, "axes.spines.top": False,
    "axes.spines.right": False, "figure.dpi": 300, "savefig.dpi": 600,
})

# ---------------------------------------------------------------- LOAD

frames = []
higher_is_better = {}

for name, folder in RESULT_DIRS.items():
    f = folder / "validation_results.csv"
    if not f.exists():
        print(f"WARNING: missing {f}")
        continue

    d = pd.read_csv(f)
    for c in ["composite", "lookback", "config_id", "seed"]:
        if c not in d.columns:
            raise ValueError(f"{f.name} lacks column '{c}'")

    d["composite"] = pd.to_numeric(d["composite"], errors="coerce")
    d = d.replace([np.inf, -np.inf], np.nan).dropna(subset=["composite"])

    # a resumed search can duplicate runs: keep the last entry per trial
    n0 = len(d)
    d = d.drop_duplicates(subset=["lookback", "config_id", "seed"],
                          keep="last")
    if len(d) < n0:
        print(f"{name}: removed {n0 - len(d)} duplicate trials")

    # Is a higher composite better? Compare with the saved best model.
    hib = True
    bf = folder / "best_model_summary.csv"
    if bf.exists():
        best_val = pd.to_numeric(pd.read_csv(bf)["composite"],
                                 errors="coerce").iloc[0]
        if abs(best_val - d["composite"].min()) < abs(best_val
                                                      - d["composite"].max()):
            hib = False
    higher_is_better[name] = hib

    d["Backbone"] = name
    frames.append(d[["Backbone", "lookback", "config_id", "seed",
                     "composite"]])
    print(f"{name:<9} trials={len(d):>5} | lookbacks="
          f"{[int(x) for x in sorted(d['lookback'].unique())]} | "
          f"higher is better={hib}")

if not frames:
    raise SystemExit("No validation_results.csv files were found.")

df = pd.concat(frames, ignore_index=True)
backbones = [b for b in RESULT_DIRS if b in df["Backbone"].unique()]

# ---------------------------------------------------------------- SUMMARISE

rows = []
for name in backbones:
    hib = higher_is_better[name]
    d = df[df["Backbone"] == name]
    for lb, g in d.groupby("lookback"):
        s = g["composite"]
        rows.append({
            "Backbone": name, "Lookback": int(lb), "N_trials": len(s),
            "N_configs": g["config_id"].nunique(),
            "N_seeds": g["seed"].nunique(),
            "Best": s.max() if hib else s.min(),
            "Median": s.median(),
            "Q25": s.quantile(0.25), "Q75": s.quantile(0.75),
        })

summary = pd.DataFrame(rows)
summary["IQR"] = summary["Q75"] - summary["Q25"]
summary["Best_minus_Median"] = (summary["Best"] - summary["Median"]).abs()

# seed sensitivity: SD over seeds within each configuration, then median
sd = (
    df.groupby(["Backbone", "lookback", "config_id"])["composite"].std()
    .reset_index(name="sd")
    .groupby(["Backbone", "lookback"])["sd"].median()
    .reset_index(name="Median_seed_SD")
    .rename(columns={"lookback": "Lookback"})
)
sd["Lookback"] = sd["Lookback"].astype(int)
summary = summary.merge(sd, on=["Backbone", "Lookback"], how="left")

summary.to_csv(OUT_DIR / "memory_horizon_summary.csv", index=False)

best_rows = []
for name in backbones:
    s = summary[summary["Backbone"] == name]
    j = s["Best"].idxmax() if higher_is_better[name] else s["Best"].idxmin()
    best_rows.append(s.loc[j, ["Backbone", "Lookback", "Best", "Median",
                               "N_trials"]])
best_df = pd.DataFrame(best_rows)
best_df.to_csv(OUT_DIR / "best_lookback_per_backbone.csv", index=False)
print("\nBest lookback per backbone (by best validation composite):")
print(best_df.round(4).to_string(index=False))

# ---------------------------------------------------------------- SELECTED LOOKBACKS

selected_lb = {}
try:
    if SELECTION_FILE.exists():
        # utf-8-sig removes a hidden BOM that can corrupt the first header
        sel = pd.read_csv(SELECTION_FILE, encoding="utf-8-sig")
        sel.columns = [str(c).strip().lower() for c in sel.columns]
        print("\nSelection file columns:", list(sel.columns))

        arch_col = next((c for c in sel.columns
                         if c in ("architecture", "model", "backbone")), None)
        lb_col = next((c for c in sel.columns if "lookback" in c), None)

        if arch_col is None or lb_col is None:
            print("Selected lookbacks not drawn: could not find "
                  "architecture/lookback columns.")
        else:
            for _, r in sel.iterrows():
                key = (str(r[arch_col]).strip().upper()
                       .replace("-", "_").replace(" ", "_"))
                selected_lb[key] = int(r[lb_col])
            print("Selected lookbacks:", selected_lb)
    else:
        print(f"Selection file not found: {SELECTION_FILE}")
except Exception as e:
    print(f"Selected lookbacks not drawn ({type(e).__name__}: {e})")

all_lookbacks = sorted(summary["Lookback"].unique())

# ---------------------------------------------------------------- FIGURE

n = len(backbones)
ncols = 3
nrows = int(np.ceil((n + 1) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.2 * nrows))
flat = np.atleast_1d(axes).ravel()
letters = "abcdefgh"

if YLIM is None:
    lo = summary["Median"].min()
    hi = summary["Best"].max()
    pad = 0.08 * (hi - lo if hi > lo else 1.0)
    ylim = (lo - pad, hi + 2.5 * pad)
else:
    ylim = YLIM

for k, name in enumerate(backbones):
    ax = flat[k]
    hib = higher_is_better[name]
    s = summary[summary["Backbone"] == name].sort_values("Lookback")
    col = COLORS[name]

    ax.fill_between(s["Lookback"], s["Q25"], s["Q75"], color=col,
                    alpha=0.18, linewidth=0, zorder=1)
    ax.plot(s["Lookback"], s["Median"], color=col, lw=1.6, ls="--",
            marker="o", ms=5, mfc="white", mec=col, zorder=3)
    ax.plot(s["Lookback"], s["Best"], color=col, lw=2.2, marker="o",
            ms=6, zorder=4)

    j = s["Best"].idxmax() if hib else s["Best"].idxmin()
    ax.scatter(s.loc[j, "Lookback"], s.loc[j, "Best"], marker="*", s=260,
               color="gold", edgecolor="black", linewidth=0.9, zorder=6)
    ax.annotate(f"{int(s.loc[j, 'Lookback'])} d",
                (s.loc[j, "Lookback"], s.loc[j, "Best"]),
                xytext=(0, 12), textcoords="offset points", ha="center",
                fontsize=9, fontweight="bold")

    if name in selected_lb:
        ax.axvline(selected_lb[name], color="#555555", ls=":", lw=1.2,
                   zorder=0)

    ax.set_title(f"({letters[k]}) {LABEL.get(name, name)}", loc="left",
                 fontweight="bold")
    ax.set_xticks(all_lookbacks)
    ax.set_ylim(*ylim)
    ax.grid(True, linestyle="--", lw=0.6, alpha=0.35)
    if k % ncols == 0:
        ax.set_ylabel("Validation composite score")
    ax.set_xlabel("Lookback window (days)")

# overlay of best curves
ax = flat[n]
for name in backbones:
    s = summary[summary["Backbone"] == name].sort_values("Lookback")
    ax.plot(s["Lookback"], s["Best"], color=COLORS[name], lw=2.0,
            marker="o", ms=5, label=LABEL.get(name, name))
ax.set_title(f"({letters[n]}) Best score, all backbones", loc="left",
             fontweight="bold")
ax.set_xticks(all_lookbacks)
ax.set_ylim(*ylim)
ax.grid(True, linestyle="--", lw=0.6, alpha=0.35)
ax.set_xlabel("Lookback window (days)")
ax.legend(frameon=False, ncol=2)

for ax in flat[n + 1:]:
    ax.set_visible(False)

legend_handles = [
    Line2D([0], [0], color="#444444", lw=2.2, marker="o", label="Best"),
    Line2D([0], [0], color="#444444", lw=1.6, ls="--", marker="o",
           mfc="white", label="Median"),
    Patch(facecolor="#444444", alpha=0.2,
          label="25th-75th percentile (configs x seeds)"),
    Line2D([0], [0], marker="*", color="w", markerfacecolor="gold",
           markeredgecolor="black", markersize=14, label="Best lookback"),
]
if selected_lb:
    legend_handles.append(Line2D([0], [0], color="#555555", ls=":", lw=1.2,
                                 label="Selected lookback"))
fig.legend(handles=legend_handles, loc="lower center",
           ncol=len(legend_handles), frameon=False,
           bbox_to_anchor=(0.5, -0.02))

fig.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(OUT_DIR / "memory_horizon_response_curve.png", dpi=600,
            bbox_inches="tight", facecolor="white")

plt.show()

print("\nSaved in:", OUT_DIR)

In [ ]:
# =============================================================================
# FLOW DURATION CURVES (FDC) -- dual-axis style
# =============================================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FixedFormatter, NullFormatter

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------- PATHS

DATA_DIR = PROJECT_DIR / "data"
PREPARED_DIR = RESULTS_ROOT / "LSTM_Prepared_v4"
OUT = RESULTS_ROOT / "Projection_v4"
FDC_DIR = OUT / "fdc_plots"
FDC_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- OPTIONS

BASELINE = (2000, 2021)
PERIODS = {"Near": (2041, 2070), "Far": (2071, 2100)}
MODELS = ["LSTM", "GRU", "TCN", "MLP", "LSTM_FCN"]
MODEL_LABEL = {"LSTM_FCN": "LSTM-FCN"}
STATIONS = ["Shebe", "Asendabo", "Abelti"]
SCENARIOS = ["SSP245", "SSP585"]
STATION_IDS = {0: "Shebe", 1: "Asendabo", 2: "Abelti"}

SHOW_INDIVIDUAL_MODELS = False   # Figure A: add the five models as thin lines
SHOW_SWAT_FUTURE = True          # Figure B: add SWAT-only near/far (dashed)
LOG_FLOOR = None                 # lower limit of the log axis; None = auto

COL = {
    "Observed": "#000000", "SWAT": "#D55E00", "Ensemble": "#009E73",
    "Historical": "#6e6e6e", "Near": "#E69F00", "Far": "#0072B2",
    "LSTM": "#1f77b4", "GRU": "#ff7f0e", "TCN": "#2ca02c",
    "MLP": "#d62728", "LSTM_FCN": "#9467bd",
}

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 11, "axes.labelsize": 11, "axes.titlesize": 12,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 300, "savefig.dpi": 600,
})

# ---------------------------------------------------------------- LOAD

daily = pd.read_csv(OUT / "daily_projections_all_models.csv",
                    parse_dates=["Date"])
daily["SWAT"] = pd.to_numeric(daily["SWAT"], errors="coerce")
daily["Hybrid"] = pd.to_numeric(daily["Hybrid"], errors="coerce")

parts = []
for split in ("train", "val", "test"):
    with np.load(PREPARED_DIR / "historical" / "lookback_030"
                 / f"{split}.npz", allow_pickle=False) as z:
        parts.append(pd.DataFrame({
            "Date": pd.to_datetime(z["dates"]),
            "Station": [STATION_IDS[int(s)] for s in z["station"]],
            "Observed": z["Observed"].astype(float),
        }))
obs = pd.concat(parts, ignore_index=True)


def series(station, scenario):
    sub = daily[(daily["Station"] == station)
                & (daily["Scenario"] == scenario)]
    if sub.empty:
        raise ValueError(f"No data for {station} / {scenario}")
    out = sub.groupby("Date")["SWAT"].mean().to_frame("SWAT")
    for m in MODELS:
        s = sub[sub["Model"] == m].set_index("Date")["Hybrid"]
        if s.index.has_duplicates:
            raise ValueError(f"Duplicate dates: {station}, {scenario}, {m}")
        out[m] = s
    out["Ensemble"] = out[MODELS].mean(axis=1, skipna=False)
    if scenario == "Historical":
        o = obs[obs["Station"] == station].drop_duplicates("Date")
        out["Observed"] = o.set_index("Date")["Observed"]
    return out.sort_index()


def window(df, y0, y1):
    return df[(df.index.year >= y0) & (df.index.year <= y1)]


# ---------------------------------------------------------------- FDC helpers

def fdc(values):
    """Weibull exceedance probability (%) and flows sorted high -> low."""
    v = np.asarray(values, dtype=float)
    v = np.sort(v[np.isfinite(v)])[::-1]
    p = np.arange(1, len(v) + 1) / (len(v) + 1) * 100.0
    return p, v


def draw_fdc(ax, curves, ymax=None):
    """Dual-axis FDC. curves: list of dicts(label, values, color, ls, lw,
    alpha, z)."""
    ax2 = ax.inset_axes([0, 0, 1, 1])
    ax2.patch.set_alpha(0)

    allpos = []
    top = 0.0
    for c in curves:
        p, v = fdc(c["values"])
        kw = dict(color=c["color"], ls=c.get("ls", "-"),
                  lw=c.get("lw", 1.6), alpha=c.get("alpha", 1.0),
                  zorder=c.get("z", 3))
        ax.plot(p, v, label=c["label"], **kw)
        vp = np.where(v > 0, v, np.nan)
        ax2.plot(p, vp, **kw)
        if len(v):
            top = max(top, np.nanmax(v))
            allpos.append(v[v > 0])

    pos = np.concatenate(allpos) if allpos else np.array([1.0])
    floor = LOG_FLOOR if LOG_FLOOR else 10 ** np.floor(
        np.log10(np.nanpercentile(pos, 0.5)))
    ceil = 10 ** np.ceil(np.log10(top)) if top > 0 else 10

    # main axes: log x, linear y (peaks)
    ax.set_xscale("log")
    ax.set_xlim(0.01, 100)
    ax.set_ylim(0, (ymax or top) * 1.05)
    ax.xaxis.set_major_locator(FixedLocator([0.01, 0.1, 1, 10, 100]))
    ax.xaxis.set_major_formatter(FixedFormatter(["0.01", "0.1", "1",
                                                 "10", "100"]))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel("Exceedance probability (%) - log scale")
    ax.set_ylabel("Streamflow (m$^3$/s)")
    ax.grid(True, which="major", linestyle="--", lw=0.5, alpha=0.3)

    # overlay axes: linear x, log y (low flows)
    ax2.set_xlim(0, 100)
    ax2.set_yscale("log")
    ax2.set_ylim(floor, ceil)
    ax2.xaxis.tick_top()
    ax2.xaxis.set_label_position("top")
    ax2.yaxis.tick_right()
    ax2.yaxis.set_label_position("right")
    ax2.set_xlabel("Exceedance probability (%) - linear scale")
    ax2.set_ylabel("Streamflow (m$^3$/s) - log scale")
    for side in ("left", "bottom"):
        ax2.spines[side].set_visible(False)
    for side in ("top", "right"):
        ax2.spines[side].set_visible(True)
    return ax2


# ---------------------------------------------------------------- QUANTILES

QUANTILE_ROWS = []


def record_quantiles(station, scenario, label, values):
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if not len(v):
        return
    row = {"Station": station, "Scenario": scenario, "Curve": label,
           "N_days": len(v), "Mean": v.mean()}
    # Qx = flow exceeded x % of the time
    for q in (1, 5, 10, 50, 90, 95, 99):
        row[f"Q{q}"] = np.percentile(v, 100 - q)
    QUANTILE_ROWS.append(row)


# ---------------------------------------------------------------- FIGURE A

hist = {}
for st in STATIONS:
    h = window(series(st, "Historical"), *BASELINE)
    h = h.dropna(subset=["Observed", "SWAT", "Ensemble"])
    hist[st] = h
    print(f"{st}: {len(h)} common historical days "
          f"({h.index.min().date()} to {h.index.max().date()})")

fig, axes = plt.subplots(1, len(STATIONS), figsize=(16, 5.2),
                         constrained_layout=True)
letters = "abcdef"

for k, st in enumerate(STATIONS):
    ax = axes[k]
    h = hist[st]

    curves = []
    if SHOW_INDIVIDUAL_MODELS:
        for m in MODELS:
            curves.append(dict(label=MODEL_LABEL.get(m, m), values=h[m],
                               color=COL[m], lw=0.9, alpha=0.7, z=2))
    curves += [
        dict(label="Observed", values=h["Observed"], color=COL["Observed"],
             lw=2.0, z=5),
        dict(label="SWAT", values=h["SWAT"], color=COL["SWAT"], lw=1.8,
             z=4),
        dict(label="Hybrid ensemble mean", values=h["Ensemble"],
             color=COL["Ensemble"], ls="--", lw=2.0, z=6),
    ]
    draw_fdc(ax, curves)
    ax.set_title(f"({letters[k]}) {st}", loc="left", fontweight="bold",
                 pad=40)

    for c in curves:
        record_quantiles(st, "Historical", c["label"], c["values"])

h_, l_ = axes[0].get_legend_handles_labels()
fig.legend(h_, l_, loc="outside lower center", ncol=len(l_), frameon=False)
fig.savefig(FDC_DIR / "FDC_historical.png", dpi=600, bbox_inches="tight",
            facecolor="white")
fig.savefig(FDC_DIR / "FDC_historical.pdf", bbox_inches="tight",
            facecolor="white")
plt.show()
plt.close(fig)

# ---------------------------------------------------------------- FIGURE B

fig, axes = plt.subplots(len(STATIONS), len(SCENARIOS), figsize=(13, 14),
                         constrained_layout=True)
n = 0

for i, st in enumerate(STATIONS):
    for j, sc in enumerate(SCENARIOS):
        ax = axes[i, j]
        fut = series(st, sc)
        h = hist[st]

        curves = [dict(label="Historical (2000-2021), ensemble",
                       values=h["Ensemble"], color=COL["Historical"],
                       lw=2.0, z=4)]
        for per, (a, z) in PERIODS.items():
            w = window(fut, a, z)
            curves.append(dict(
                label=f"{per} future ({a}-{z}), ensemble",
                values=w["Ensemble"], color=COL[per], lw=2.0, z=5))
            if SHOW_SWAT_FUTURE:
                curves.append(dict(
                    label=f"{per} future, SWAT-only",
                    values=w["SWAT"], color=COL[per], ls="--", lw=1.2,
                    alpha=0.9, z=3))

        draw_fdc(ax, curves)
        ax.set_title(f"({letters[n]}) {st} - {sc}", loc="left",
                     fontweight="bold", pad=40)
        n += 1

        for c in curves:
            record_quantiles(st, sc, c["label"], c["values"])

h_, l_ = axes[0, 0].get_legend_handles_labels()
fig.legend(h_, l_, loc="outside lower center", ncol=3, frameon=False)
fig.savefig(FDC_DIR / "FDC_future.png", dpi=600, bbox_inches="tight",
            facecolor="white")
fig.savefig(FDC_DIR / "FDC_future.pdf", bbox_inches="tight",
            facecolor="white")
plt.show()
plt.close(fig)

# ---------------------------------------------------------------- TABLE

q = pd.DataFrame(QUANTILE_ROWS)
q.to_csv(FDC_DIR / "FDC_flow_quantiles.csv", index=False)
print("\nSaved in:", FDC_DIR)
print(q.round(2).head(12).to_string(index=False))


In [ ]:

# =============================================================================
# MULTI-ARCHITECTURE DIRECTIONAL AGREEMENT HEATMAP

# =============================================================================

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from pathlib import Path
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# =============================================================================
# 1. PATHS AND SETTINGS
# =============================================================================

DATA_DIR = PROJECT_DIR / "data"

ANNUAL_DIR = RESULTS_ROOT / "Projection_v4" / "annual_change"

INPUT = ANNUAL_DIR / "annual_change_by_year.csv"

OUTPUT = ANNUAL_DIR / "Directional_Agreement"
OUTPUT.mkdir(parents=True, exist_ok=True)

STATIONS = ["Shebe", "Asendabo", "Abelti"]

MODELS = ["LSTM", "GRU", "TCN", "MLP", "LSTM_FCN"]

SCENARIO_PERIODS = [
    ("SSP245", "Near"),
    ("SSP245", "Far"),
    ("SSP585", "Near"),
    ("SSP585", "Far"),
]

PERIODS = {
    "Near": (2041, 2070),
    "Far": (2071, 2100),
}

MODEL_LABELS = {
    "LSTM": "LSTM",
    "GRU": "GRU",
    "TCN": "TCN",
    "MLP": "MLP",
    "LSTM_FCN": "LSTM-FCN",
}

# =============================================================================
# 2. FIGURE STYLE
# =============================================================================

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 13,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 13,
    "figure.dpi": 300,
    "savefig.dpi": 600,
    "axes.linewidth": 0.9,
})

# Cream -> yellow -> orange -> dark red
from matplotlib.colors import LinearSegmentedColormap

AGREEMENT_CMAP = LinearSegmentedColormap.from_list(
    "agreement",
    [
        "#FFF8E7",
        "#FEE8A6",
        "#FDBB62",
        "#E76F51",
        "#9B2226",
    ],
    N=256
)

# =============================================================================
# 3. LOAD DATA
# =============================================================================

if not INPUT.exists():
    raise FileNotFoundError(f"Missing input file: {INPUT}")

df = pd.read_csv(INPUT)

required = [
    "Station", "Scenario", "Model",
    "Year", "Annual_change_%"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df[df["Model"].isin(MODELS)].copy()

df["Annual_change_%"] = pd.to_numeric(
    df["Annual_change_%"], errors="coerce"
)

df["Year"] = pd.to_numeric(
    df["Year"], errors="coerce"
)

df = df.dropna(
    subset=["Station", "Scenario", "Model",
            "Year", "Annual_change_%"]
)

# =============================================================================
# 4. CALCULATE PERIOD-MEAN CHANGE FOR EACH ARCHITECTURE
# =============================================================================

model_period = []

for station in STATIONS:

    for scenario, period in SCENARIO_PERIODS:

        start, end = PERIODS[period]

        sub = df[
            (df["Station"] == station) &
            (df["Scenario"] == scenario) &
            (df["Year"].between(start, end))
        ]

        for model in MODELS:

            s = sub[sub["Model"] == model]

            if s.empty:
                continue

            model_period.append({
                "Station": station,
                "Scenario": scenario,
                "Period": period,
                "Model": model,
                "Mean_Annual_Change_%":
                    s["Annual_change_%"].mean(),
                "N_years": s["Year"].nunique(),
            })

model_period = pd.DataFrame(model_period)

model_period.to_csv(
    OUTPUT / "model_directional_changes.csv",
    index=False
)

# =============================================================================
# 5. CALCULATE DIRECTIONAL AGREEMENT
# =============================================================================

agreement_rows = []

for station in STATIONS:

    for scenario, period in SCENARIO_PERIODS:

        sub = model_period[
            (model_period["Station"] == station) &
            (model_period["Scenario"] == scenario) &
            (model_period["Period"] == period)
        ]

        # Require all five architectures
        if sub["Model"].nunique() != 5:
            print(
                f"WARNING: incomplete architectures: "
                f"{station}, {scenario}, {period}"
            )
            continue

        changes = sub.set_index("Model")[
            "Mean_Annual_Change_%"
        ].reindex(MODELS)

        if changes.isna().any():
            continue

        n_increase = int((changes > 0).sum())
        n_decrease = int((changes < 0).sum())
        n_unchanged = int((changes == 0).sum())

        if n_increase > n_decrease:
            direction = "Increase"
            count = n_increase

        elif n_decrease > n_increase:
            direction = "Decrease"
            count = n_decrease

        else:
            direction = "Tie"
            count = max(n_increase, n_decrease)

        agreement = count / len(MODELS) * 100

        agreement_rows.append({
            "Station": station,
            "Scenario": scenario,
            "Period": period,
            "Direction": direction,
            "Agreement_%": agreement,
            "N_Increase": n_increase,
            "N_Decrease": n_decrease,
            "N_Unchanged": n_unchanged,
            "Mean_Model_Change_%": changes.mean(),
            "Minimum_Model_Change_%": changes.min(),
            "Maximum_Model_Change_%": changes.max(),
            "Architecture_Range_pp":
                changes.max() - changes.min(),
            "Models": "; ".join(
                f"{m}: {changes[m]:.2f}%"
                for m in MODELS
            ),
        })

agreement_df = pd.DataFrame(agreement_rows)

agreement_df.to_csv(
    OUTPUT / "directional_agreement_summary.csv",
    index=False
)

print("\nDIRECTIONAL AGREEMENT RESULTS")
print("=" * 75)

print(
    agreement_df[
        [
            "Station", "Scenario", "Period",
            "Direction", "Agreement_%",
            "Mean_Model_Change_%"
        ]
    ].round(2).to_string(index=False)
)

# =============================================================================
# 6. PREPARE HEATMAP MATRICES
# =============================================================================

fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 6),
    sharey=True
)

letters = ["(a)", "(b)", "(c)"]

for i, station in enumerate(STATIONS):

    ax = axes[i]

    sub = agreement_df[
        agreement_df["Station"] == station
    ]

    matrix = np.full((1, 4), np.nan)

    annotations = [["" for _ in range(4)]]

    for j, (scenario, period) in enumerate(
        SCENARIO_PERIODS
    ):

        r = sub[
            (sub["Scenario"] == scenario) &
            (sub["Period"] == period)
        ]

        if r.empty:
            continue

        row = r.iloc[0]

        matrix[0, j] = row["Agreement_%"]

        if row["Direction"] == "Increase":
            symbol = "\u2191"
        elif row["Direction"] == "Decrease":
            symbol = "\u2193"
        else:
            symbol = "="

        annotations[0][j] = (
            f"{symbol} {row['Agreement_%']:.0f}%\n"
            f"{row['Direction']}"
        )

    im = ax.imshow(
        matrix,
        aspect="auto",
        cmap=AGREEMENT_CMAP,
        norm=Normalize(vmin=20, vmax=100),
        interpolation="nearest"
    )

    ax.set_title(
        f"{letters[i]} {station}",
        loc="left",
        
        pad=12
    )

    ax.set_xticks(range(4))

    ax.set_xticklabels([
        "Near\nSSP245",
        "Far\nSSP245",
        "Near\nSSP585",
        "Far\nSSP585",
    ])

    ax.set_yticks([0])
    ax.set_yticklabels(["Architecture\nagreement"])

    ax.tick_params(length=0)

    # White boundaries
    ax.set_xticks(
        np.arange(-0.5, 4, 1),
        minor=True
    )

    ax.set_yticks(
        np.arange(-0.5, 1, 1),
        minor=True
    )

    ax.grid(
        which="minor",
        color="white",
        linewidth=2
    )

    ax.tick_params(which="minor", length=0)

    # Cell annotations
    for j in range(4):

        if not np.isfinite(matrix[0, j]):
            continue

        value = matrix[0, j]

        color = "white" if value >= 80 else "#333333"

        ax.text(
            j, 0,
            annotations[0][j],
            ha="center",
            va="center",
            fontsize=12,
            color=color
        )

    ax.set_xlim(-0.5, 3.5)

# =============================================================================
# 7. COLORBAR AND LABELS
# =============================================================================

fig.subplots_adjust(
    left=0.12,
    right=0.88,
    bottom=0.23,
    top=0.82,
    wspace=0.25
)

sm = ScalarMappable(
    norm=Normalize(vmin=20, vmax=100),
    cmap=AGREEMENT_CMAP
)

sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=axes,
    fraction=0.025,
    pad=0.035
)

cbar.set_label(
    "Directional agreement among architectures (%)",
    fontsize=12
)

fig.suptitle(
    "Multi-architecture agreement in projected streamflow direction",
    fontsize=15,
       y=0.92
)

fig.text(
    0.5, 0.09,
    "Projection period and climate scenario",
    ha="center",
    fontsize=13
)

fig.text(
    0.5, 0.035,
    "\u2191 Increasing streamflow   "
    "\u2193 Decreasing streamflow   "
    "Agreement calculated from five architectures",
    ha="center",
    fontsize=12
)

# =============================================================================
# 8. EXPORT
# =============================================================================

png = OUTPUT / "directional_agreement_heatmap.png"


fig.savefig(
    png,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

print("\nFigures and tables saved in:")
print(OUTPUT)


In [ ]:
# =============================================================================
# ARCHITECTURE-SPECIFIC DISTRIBUTIONS OF ANNUAL STREAMFLOW CHANGE

# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# =============================================================================
# 1. PATHS AND SETTINGS
# =============================================================================

DATA_DIR = PROJECT_DIR / "data"
ANNUAL_DIR = RESULTS_ROOT / "Projection_v4" / "annual_change"
INPUT = Path(os.environ.get("ANNUAL_INPUT", ANNUAL_DIR / "annual_change_by_year.csv"))

OUTPUT = Path(os.environ.get("ANNUAL_OUTPUT",
                             ANNUAL_DIR / "Architecture_Distributions"))
OUTPUT.mkdir(parents=True, exist_ok=True)

STATIONS = ["Shebe", "Asendabo", "Abelti"]
SCENARIOS = ["SSP245", "SSP585"]
MODELS = ["LSTM", "GRU", "TCN", "MLP", "LSTM_FCN"]
LABELS = ["LSTM", "GRU", "TCN", "MLP", "LSTM-FCN", "Ens.\nmean"]
PERIODS = {"Near": (2041, 2070), "Far": (2071, 2100)}

SIDE = {"SSP245": -1, "SSP585": +1}                 # left / right half
FILL = {"SSP245": "#56B4E9", "SSP585": "#E69F00"}   # violin fill
DARK = {"SSP245": "#0072B2", "SSP585": "#D55E00"}   # IQR bar / markers

# =============================================================================
# 2. STYLE
# =============================================================================

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 18,
    "axes.labelsize": 18,
    "axes.titlesize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "axes.linewidth": 0.9,
})

# =============================================================================
# 3. LOAD DATA
# =============================================================================

df = pd.read_csv(INPUT)
need = ["Station", "Scenario", "Model", "Year", "Annual_change_%"]
miss = [c for c in need if c not in df.columns]
if miss:
    raise ValueError(f"Missing columns: {miss}")

df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Annual_change_%"] = pd.to_numeric(df["Annual_change_%"], errors="coerce")
df = df[df["Model"].isin(MODELS)].dropna(subset=need)
df = df[df["Year"].between(2041, 2100)]

# Ensemble mean per station / scenario / year (only years with all 5 models)
ens = (df.pivot_table(index=["Station", "Scenario", "Year"],
                      columns="Model", values="Annual_change_%")
         .reindex(columns=MODELS).dropna().mean(axis=1)
         .rename("Annual_change_%").reset_index())
ens["Model"] = "Ensemble"
df = pd.concat([df, ens], ignore_index=True)

CATEGORIES = MODELS + ["Ensemble"]

# =============================================================================
# 4. HELPERS
# =============================================================================

def half_violin(ax, vals, pos, side, color):
    """Draw one half of a violin (side = -1 left, +1 right)."""
    parts = ax.violinplot(vals, positions=[pos], widths=0.92,
                          showextrema=False, showmeans=False,
                          showmedians=False, bw_method=0.35)
    body = parts["bodies"][0]
    verts = body.get_paths()[0].vertices
    if side < 0:
        verts[:, 0] = np.minimum(verts[:, 0], pos)
    else:
        verts[:, 0] = np.maximum(verts[:, 0], pos)
    body.set_facecolor(color)
    body.set_edgecolor("#222222")
    body.set_linewidth(0.7)
    body.set_alpha(0.80)
    body.set_zorder(2)

# =============================================================================
# 5. FIGURE
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 7.6))
summary_rows = []

for k, (ax, station) in enumerate(zip(axes, STATIONS)):
    sub = df[df["Station"] == station]

    ymin = sub["Annual_change_%"].min()
    ymax = sub["Annual_change_%"].max()
    pad = 0.06 * (ymax - ymin)
    ymin, ymax = min(ymin - pad, -10), ymax + pad

    # Decrease / increase backdrop
    ax.axhspan(ymin, 0, color="#EDEDED", zorder=0)
    ax.axhline(0, color="#222222", linewidth=1.2, zorder=1)

    for i, cat in enumerate(CATEGORIES):
        for sc in SCENARIOS:
            s = sub[(sub["Model"] == cat) & (sub["Scenario"] == sc)]
            vals = s["Annual_change_%"].to_numpy()
            if len(vals) < 5:
                continue
            side = SIDE[sc]

            half_violin(ax, vals, i, side, FILL[sc])

            # IQR bar and median
            q1, med, q3 = np.percentile(vals, [25, 50, 75])
            xo = i + side * 0.07
            ax.plot([xo, xo], [q1, q3], color=DARK[sc], linewidth=5,
                    solid_capstyle="round", zorder=4)
            ax.scatter(xo, med, s=34, color="white", edgecolor="black",
                       linewidth=0.8, zorder=5)

            # Period means
            for per, marker, dx in [("Near", "D", 0.20), ("Far", "o", 0.33)]:
                a, b = PERIODS[per]
                m = s[s["Year"].between(a, b)]["Annual_change_%"].mean()
                ax.scatter(i + side * dx, m, marker=marker, s=62,
                           color=DARK[sc], edgecolor="black",
                           linewidth=0.9, zorder=6)
                summary_rows.append({
                    "Station": station, "Model": cat, "Scenario": sc,
                    "Period": per, "Mean_change_%": m,
                    "Share_years_increase_%": 100 * (vals > 0).mean(),
                })

    # Separator for ensemble
    ax.axvline(len(MODELS) - 0.5, color="#555555", linestyle=":",
               linewidth=1.1, zorder=1)

    ax.set_xlim(-0.6, len(CATEGORIES) - 0.4)
    ax.set_ylim(ymin, ymax)
    ax.set_xticks(range(len(CATEGORIES)))
    ax.set_xticklabels(LABELS)
    ax.set_title(f"({'abc'[k]}) {station}", loc="left", pad=10)
    ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.35, zorder=0)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    ax.tick_params(axis="x", length=0)

    ax.text(0.99, 0.015, "Decrease", transform=ax.transAxes, ha="right",
            va="bottom", fontsize=11, color="#555555", style="italic")
    ax.text(0.99, 0.985, "Increase", transform=ax.transAxes, ha="right",
            va="top", fontsize=11, color="#555555", style="italic")

    if k == 0:
        ax.set_ylabel("Annual change in mean flow, 2041\u20132100 (%)")

# =============================================================================
# 6. LEGEND AND TITLE
# =============================================================================

handles = [
    Patch(facecolor=FILL["SSP245"], edgecolor="#222222", label="SSP245 (left half)"),
    Patch(facecolor=FILL["SSP585"], edgecolor="#222222", label="SSP585 (right half)"),
    Line2D([0], [0], color="#444444", linewidth=5, solid_capstyle="round",
           label="Interquartile range"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="white",
           markeredgecolor="black", markersize=8, label="Median"),
    Line2D([0], [0], marker="D", color="w", markerfacecolor="#888888",
           markeredgecolor="black", markersize=9, label="Near-future mean (2041\u20132070)"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#888888",
           markeredgecolor="black", markersize=9, label="Far-future mean (2071\u20132100)"),
]
fig.legend(handles=handles, loc="lower center", ncol=6, frameon=False,
           fontsize=14, bbox_to_anchor=(0.5, -0.005), columnspacing=1.6)

fig.suptitle("Distribution of annual projected streamflow change by "
             "hybrid architecture", fontsize=17, y=0.985)

plt.tight_layout(rect=[0, 0.07, 1, 0.96])

# =============================================================================
# 7. EXPORT
# =============================================================================

pd.DataFrame(summary_rows).to_csv(
    OUTPUT / "architecture_distribution_summary.csv", index=False)

fig.savefig(OUTPUT / "Architecture_Split_Violins.png", dpi=600,
            bbox_inches="tight", facecolor="white")

plt.show()
plt.close(fig)
print("Saved:", OUTPUT)

In [ ]:
# =============================================================================
# FIGURE: TAYLOR DIAGRAMS OF THE HYBRID SWAT-DL MODELS

# =============================================================================

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D

# =============================================================================
# 1. SETTINGS (edit here)
# =============================================================================

DATA_DIR = PROJECT_DIR / "data"
OUTPUT = RESULTS_ROOT / "Taylor_Diagrams"

# Results folder of each architecture (LSTM folder name: change if different)
RESULT_DIRS = {
    "LSTM":     RESULTS_ROOT / "LSTM_Results_v4",
    "GRU":      RESULTS_ROOT / "GRU_Results_v4",
    "TCN":      RESULTS_ROOT / "TCN_Results_v4",
    "MLP":      RESULTS_ROOT / "MLP_Results_v4",
    "LSTM-FCN": RESULTS_ROOT / "LSTM_FCN_Results_v4",
}

STATIONS = ["Shebe", "Asendabo", "Abelti"]

# Rows of the figure: (label, start date, end date)
PERIODS = [
    ("Validation 2016\u20132018", "2016-01-01", "2018-12-31"),
    ("Test 2019\u20132021",       "2019-01-01", "2021-12-31"),
]

MODELS = ["SWAT", "LSTM", "GRU", "TCN", "MLP", "LSTM-FCN", "Ensemble"]
ARCHS = ["LSTM", "GRU", "TCN", "MLP", "LSTM-FCN"]

STYLE = {  # colour, marker, size  (Okabe-Ito, colour-blind safe)
    "SWAT":     ("#7F7F7F", "P", 120),
    "LSTM":     ("#0072B2", "o", 105),
    "GRU":      ("#E69F00", "s", 105),
    "TCN":      ("#009E73", "^", 120),
    "MLP":      ("#CC79A7", "D", 95),
    "LSTM-FCN": ("#D55E00", "v", 120),
    "Ensemble": ("#000000", "*", 260),
}

CORR_TICKS = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0]
RMS_LEVELS = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5]

# =============================================================================
# 2. STYLE
# =============================================================================

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 16,
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "axes.linewidth": 0.9,
})

# =============================================================================
# 3. LOAD AND CHECK DATA
# =============================================================================

NEEDED = ["Station", "Date", "Observed", "FLOW_OUT", "Corrected_Flow"]

def load_arch(arch, folder):
    """Read every prediction CSV in <folder>/predictions that has the needed
    columns (test, validation, ... whatever the training cell saved)."""
    pred_dir = folder / "predictions"
    frames, seen = [], []
    for f in sorted(pred_dir.glob("*.csv")):
        seen.append(f.name)
        try:
            d = pd.read_csv(f)
        except Exception:
            continue
        if set(NEEDED) <= set(d.columns):
            d = d[NEEDED].copy()
            d["Date"] = pd.to_datetime(d["Date"])
            frames.append(d)
    if not frames:
        raise FileNotFoundError(
            f"No prediction CSV with columns {NEEDED} in {pred_dir}. "
            f"Files seen: {seen}")
    out = pd.concat(frames).drop_duplicates(["Station", "Date"])
    print(f"{arch}: {len(out)} rows from {len(frames)} file(s) in {pred_dir.name}")
    return out

df = None
for arch, folder in RESULT_DIRS.items():
    d = load_arch(arch, folder)
    if df is None:
        df = d.rename(columns={"FLOW_OUT": "SWAT", "Corrected_Flow": arch})
    else:
        d = d.rename(columns={"Observed": "Observed_chk",
                              "Corrected_Flow": arch})[
            ["Station", "Date", "Observed_chk", arch]]
        df = df.merge(d, on=["Station", "Date"], how="inner")
        diff = (df["Observed"] - df["Observed_chk"]).abs().max()
        if diff > 1e-6:
            print(f"WARNING: Observed differs between LSTM and {arch} "
                  f"(max abs diff {diff:.3g}); check the folders.")
        df = df.drop(columns="Observed_chk")

df["Ensemble"] = df[ARCHS].mean(axis=1)

# keep only the periods that have data for all stations
PERIODS = [p for p in PERIODS
           if all(((df["Station"] == st) & (df["Date"] >= p[1])
                   & (df["Date"] <= p[2])).sum() >= 30 for st in STATIONS)]
if not PERIODS:
    raise ValueError("No period has data for all stations; check the dates.")
print("Periods used:", [p[0] for p in PERIODS])

# =============================================================================
# 4. TAYLOR STATISTICS
# =============================================================================

def taylor_stats(obs, sim):
    """Normalised std, correlation, normalised centred RMS difference."""
    so = obs.std(ddof=0)
    ss = sim.std(ddof=0)
    r = np.corrcoef(obs, sim)[0, 1]
    s = ss / so
    crmsd = np.sqrt(1.0 + s ** 2 - 2.0 * s * r)
    return s, r, crmsd, ss, so

rows = []
panel_stats = {}
for plabel, start, end in PERIODS:
    for st in STATIONS:
        sub = df[(df["Station"] == st)
                 & (df["Date"] >= start) & (df["Date"] <= end)]
        sub = sub.dropna(subset=["Observed"] + MODELS)   # same days for all models
        if len(sub) < 30:
            print(f"WARNING: only {len(sub)} days for {st}, {plabel}")
        res = {}
        for m in MODELS:
            s, r, c, ss, so = taylor_stats(sub["Observed"].to_numpy(),
                                           sub[m].to_numpy())
            res[m] = (s, r, c)
            rows.append({"Period": plabel, "Station": st, "Model": m,
                         "N_days": len(sub), "Std_obs": so, "Std_sim": ss,
                         "Norm_std": s, "Correlation": r,
                         "Norm_centred_RMSD": c})
        panel_stats[(plabel, st)] = res

OUTPUT.mkdir(parents=True, exist_ok=True)
stats_df = pd.DataFrame(rows)
stats_df.to_csv(OUTPUT / "taylor_statistics.csv", index=False)
print(stats_df.round(3).to_string(index=False))

# Common radial limit so every panel has the same scale
max_s = max(v[0] for res in panel_stats.values() for v in res.values())
R = float(np.ceil(max(1.5, max_s * 1.08) * 4) / 4)       # multiple of 0.25

# =============================================================================
# 5. DRAWING FUNCTION
# =============================================================================

def draw_frame(ax, R):
    th = np.linspace(0, np.pi / 2, 400)

    # standard-deviation arcs
    for s in np.arange(0.25, R, 0.25):
        if abs(s - 1.0) < 1e-9:
            continue
        ax.plot(s * np.cos(th), s * np.sin(th), color="#9a9a9a",
                linestyle=":", linewidth=0.8, zorder=1)
    ax.plot(R * np.cos(th), R * np.sin(th), color="black", linewidth=1.4,
            zorder=2)
    # reference (observed) arc
    ax.plot(np.cos(th), np.sin(th), color="black", linestyle="--",
            linewidth=1.1, zorder=2)

    # correlation rays and labels
    for r in CORR_TICKS:
        a = np.arccos(r)
        ax.plot([0, R * np.cos(a)], [0, R * np.sin(a)], color="#777777",
                linestyle=":", linewidth=0.8, zorder=1)
        ax.text(1.035 * R * np.cos(a), 1.035 * R * np.sin(a),
                f"{r:g}", ha="left" if r < 0.99 else "left", va="bottom"
                if r > 0.5 else "center", fontsize=16, color="#222222",
                rotation=0)
    ax.text(0.80 * R, 0.80 * R * 0.78 + 0.22 * R, "Correlation",
            rotation=-45, ha="center", va="center", fontsize=16)

    # centred RMS difference circles around the observed point (1, 0)
    for rho in RMS_LEVELS:
        t = np.linspace(0, np.pi, 600)
        x = 1 + rho * np.cos(t)
        y = rho * np.sin(t)
        keep = (x ** 2 + y ** 2 <= R ** 2 + 1e-9) & (x >= 0) & (y >= 0)
        xx, yy = np.where(keep, x, np.nan), np.where(keep, y, np.nan)
        ax.plot(xx, yy, color="#C0392B", linewidth=0.9, zorder=1)
        # label near 135 degrees from the observed point
        tx, ty = 1 + rho * np.cos(np.radians(125)), rho * np.sin(np.radians(125))
        if tx > 0.03 and tx ** 2 + ty ** 2 < R ** 2:
            ax.text(tx, ty, f"{rho:g}", color="#C0392B", fontsize=16,
                    ha="center", va="center",
                    bbox=dict(facecolor="white", edgecolor="none",
                              pad=0.6, alpha=0.9), zorder=3)

    ax.set_xlim(-0.02 * R, 1.10 * R)
    ax.set_ylim(-0.02 * R, 1.10 * R)
    ax.set_aspect("equal")
    ax.set_xticks(np.arange(0, R + 1e-9, 0.5))
    ax.set_yticks(np.arange(0, R + 1e-9, 0.5))
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)

# =============================================================================
# 6. FIGURE
# =============================================================================

nrows, ncols = len(PERIODS), len(STATIONS)
fig, axes = plt.subplots(nrows, ncols, figsize=(6.4 * ncols, 6.3 * nrows),
                         squeeze=False)
letters = "abcdefghijkl"
k = 0
for i, (plabel, start, end) in enumerate(PERIODS):
    for j, st in enumerate(STATIONS):
        ax = axes[i, j]
        draw_frame(ax, R)
        res = panel_stats[(plabel, st)]

        # observed reference point
        ax.scatter(1, 0, s=140, color="black", edgecolor="white",
                   linewidth=1.0, zorder=6, clip_on=False)

        for m in MODELS:
            s, r, c = res[m]
            col, mk, sz = STYLE[m]
            if r < 0:
                print(f"NOTE: negative correlation for {m}, {st}, {plabel} "
                      f"(not shown on the quarter diagram)")
                continue
            th = np.arccos(min(r, 1.0))
            ax.scatter(s * np.cos(th), s * np.sin(th), marker=mk, s=sz,
                       color=col, edgecolor="black", linewidth=0.8, zorder=5)

        ax.set_title(f"({letters[k]}) {st} | {plabel}", loc="left", pad=22)
        if j == 0:
            ax.set_ylabel("Normalised standard deviation")
        if i == nrows - 1:
            ax.set_xlabel("Normalised standard deviation")
        k += 1

handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor="black",
                  markeredgecolor="white", markersize=11, label="Observed")]
for m in MODELS:
    col, mk, sz = STYLE[m]
    label = "Ensemble mean" if m == "Ensemble" else m
    handles.append(Line2D([0], [0], marker=mk, color="w", markerfacecolor=col,
                          markeredgecolor="black",
                          markersize=np.sqrt(sz) * (0.95 if mk != "*" else 0.85),
                          label=label))
handles.append(Line2D([0], [0], color="#C0392B", linewidth=1.2,
                      label="Centred RMS difference"))
fig.legend(handles=handles, loc="lower center", ncol=len(handles),
           frameon=False, fontsize=16, bbox_to_anchor=(0.5, -0.005),
           columnspacing=1.2, handletextpad=0.4)


fig.savefig(OUTPUT / "Taylor_Diagrams.png", dpi=600, bbox_inches="tight",
            facecolor="white")
plt.show()
plt.close(fig)
print("Saved:", OUTPUT)

In [ ]:
# =============================================================================
# ALLUVIAL (FLOW) DIAGRAM OF THE PROJECTED-CHANGE CLASS BY ARCHITECTURE

# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.path import Path as MPath
from matplotlib.patches import PathPatch, Rectangle, Patch
from matplotlib.lines import Line2D

# =============================================================================
# 1. SETTINGS
# =============================================================================

DATA_DIR = PROJECT_DIR / "data"
INPUT = Path(os.environ.get(
    "ALLUV_INPUT", RESULTS_ROOT / "Projection_v4" / "percent_change_by_model.csv"))
OUTPUT = Path(os.environ.get(
    "ALLUV_OUT", RESULTS_ROOT / "Projection_v4" / "Architecture_Alluvial"))
OUTPUT.mkdir(parents=True, exist_ok=True)

STATIONS = ["Shebe", "Asendabo", "Abelti"]
MODELS = ["LSTM", "GRU", "TCN", "MLP", "LSTM_FCN"]
LABEL = {"LSTM": "LSTM", "GRU": "GRU", "TCN": "TCN", "MLP": "MLP",
         "LSTM_FCN": "LSTM-FCN"}
COLORS = {"LSTM": "#0072B2", "GRU": "#E69F00", "TCN": "#009E73",
          "MLP": "#CC79A7", "LSTM_FCN": "#D55E00"}

# Change classes, top lane first: (name, lower bound, upper bound) in percent
CLASSES = [
    ("Large increase\n(> +50%)",       50.0,  np.inf),
    ("Moderate increase\n(+10 to +50%)", 10.0, 50.0),
    ("Little change\n(\u221210 to +10%)", -10.0, 10.0),
    ("Decrease\n(< \u221210%)",        -np.inf, -10.0),
]

GROUPS = [("SSP245", ["Near", "Far"]), ("SSP585", ["Near", "Far"])]
COLS = [(sc, per) for sc, pers in GROUPS for per in pers]
X = [0.0, 1.7, 3.5, 5.2]            # x position of the four columns
NW = 0.20                            # node width

LANE_H, LANE_GAP = 5.0, 0.9          # lane height = room for 5 architectures

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 18, "axes.titlesize": 16, "axes.linewidth": 0.9,
})

# =============================================================================
# 2. DATA
# =============================================================================

df = pd.read_csv(INPUT)
need = ["Station", "Model", "Scenario", "Period", "mean_change_%"]
miss = [c for c in need if c not in df.columns]
if miss:
    raise ValueError(f"Missing columns: {miss}")
df = df[df["Model"].isin(MODELS)].copy()

ens = (df.groupby(["Station", "Scenario", "Period"], as_index=False)
         ["mean_change_%"].mean().assign(Model="Ensemble"))
df = pd.concat([df, ens], ignore_index=True)

def klass(v):
    for k, (_, lo, hi) in enumerate(CLASSES):
        if lo <= v < hi:
            return k
    return None

df["Class"] = df["mean_change_%"].apply(klass)
df.to_csv(OUTPUT / "alluvial_classes.csv", index=False)

def lane_bottom(k):                  # k = 0 is the top lane
    return (len(CLASSES) - 1 - k) * (LANE_H + LANE_GAP)

# =============================================================================
# 3. DRAWING HELPERS
# =============================================================================

def ribbon(ax, xa, ya_hi, ya_lo, xb, yb_hi, yb_lo, color, alpha=0.62):
    xm = (xa + xb) / 2.0
    verts = [(xa, ya_hi), (xm, ya_hi), (xm, yb_hi), (xb, yb_hi),
             (xb, yb_lo), (xm, yb_lo), (xm, ya_lo), (xa, ya_lo), (xa, ya_hi)]
    codes = [MPath.MOVETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4,
             MPath.LINETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4,
             MPath.CLOSEPOLY]
    ax.add_patch(PathPatch(MPath(verts, codes), facecolor=color,
                           edgecolor="white", linewidth=0.5,
                           alpha=alpha, zorder=3))

# =============================================================================
# 4. FIGURE
# =============================================================================

fig, axes = plt.subplots(1, len(STATIONS), figsize=(19, 8.2), sharey=True)
total_h = len(CLASSES) * LANE_H + (len(CLASSES) - 1) * LANE_GAP

for pi, (ax, st) in enumerate(zip(axes, STATIONS)):
    sub = df[df["Station"] == st]

    # lane backgrounds
    for k in range(len(CLASSES)):
        ax.add_patch(Rectangle((-0.45, lane_bottom(k)), X[-1] + NW + 0.9,
                               LANE_H, facecolor="#F3F3F3" if k % 2 == 0
                               else "#EAEAEA", edgecolor="none", zorder=0))

    # class of each architecture in each column
    cls = {}
    for ci, (sc, per) in enumerate(COLS):
        for m in MODELS + ["Ensemble"]:
            r = sub[(sub["Model"] == m) & (sub["Scenario"] == sc)
                    & (sub["Period"] == per)]
            cls[(ci, m)] = None if r.empty else int(r["Class"].iloc[0])

    # slot of each architecture inside its node (top to bottom, fixed order)
    slot = {}
    node_y0, node_cnt = {}, {}
    for ci in range(len(COLS)):
        for k in range(len(CLASSES)):
            members = [m for m in MODELS if cls[(ci, m)] == k]
            c = len(members)
            node_cnt[(ci, k)] = c
            y0 = lane_bottom(k) + (LANE_H - c) / 2.0
            node_y0[(ci, k)] = y0
            for i, m in enumerate(members):
                slot[(ci, m)] = (y0 + c - i, y0 + c - i - 1)   # (top, bottom)

    # ribbons: near -> far inside each scenario
    for first in (0, 2):
        a, b = first, first + 1
        for m in MODELS:
            if (a, m) in slot and (b, m) in slot:
                ta, ba = slot[(a, m)]
                tb, bb = slot[(b, m)]
                ribbon(ax, X[a] + NW, ta, ba, X[b], tb, bb, COLORS[m])

    # nodes with counts and ensemble markers
    for ci in range(len(COLS)):
        for k in range(len(CLASSES)):
            c = node_cnt[(ci, k)]
            if c == 0:
                continue
            ax.add_patch(Rectangle((X[ci], node_y0[(ci, k)]), NW, c,
                                   facecolor="#2b2b2b", edgecolor="white",
                                   linewidth=0.8, zorder=4))
            ax.text(X[ci] + NW / 2, node_y0[(ci, k)] + c / 2, str(c),
                    ha="center", va="center", color="white", fontsize=16, zorder=5)
        ke = cls[(ci, "Ensemble")]
        if ke is not None:
            xm = X[ci] - 0.17 if ci in (0, 2) else X[ci] + NW + 0.17
            ax.scatter(xm, lane_bottom(ke) + LANE_H / 2, marker="D", s=95,
                       color="black", edgecolor="white", linewidth=0.8,
                       zorder=6)

    # column and group labels
    for ci, (sc, per) in enumerate(COLS):
        ax.text(X[ci] + NW / 2, total_h + 0.35, per, ha="center",
                va="bottom", fontsize=16)
    for g, (sc, pers) in enumerate(GROUPS):
        i0 = 2 * g
        xc = (X[i0] + X[i0 + 1] + NW) / 2
        ax.text(xc, total_h + 1.35, sc, ha="center", va="bottom",
                fontsize=16)
        ax.plot([X[i0] - 0.1, X[i0 + 1] + NW + 0.1],
                [total_h + 1.25, total_h + 1.25], color="black", linewidth=1.2)

    ax.set_xlim(-0.55, X[-1] + NW + 0.5)
    ax.set_ylim(-0.4, total_h + 2.6)
    ax.set_xticks([])
    for s in ["top", "right", "bottom", "left"]:
        ax.spines[s].set_visible(False)
    ax.set_title(f"({'abc'[pi]}) {st}", loc="left", pad=8, fontsize=17)

# class lane labels on the first panel
axes[0].set_yticks([lane_bottom(k) + LANE_H / 2 for k in range(len(CLASSES))])
axes[0].set_yticklabels([c[0] for c in CLASSES], fontsize=16)
axes[0].tick_params(axis="y", length=0)
for a in axes[1:]:
    a.tick_params(axis="y", left=False, labelleft=False)

# legend
handles = [Patch(facecolor=COLORS[m], alpha=0.8, label=LABEL[m]) for m in MODELS]
handles.append(Line2D([0], [0], marker="D", color="w", markerfacecolor="black",
                      markeredgecolor="white", markersize=10,
                      label="Ensemble mean class"))
handles.append(Patch(facecolor="#2b2b2b", label="Node = number of architectures"))
fig.legend(handles=handles, loc="lower center", ncol=len(handles),
           frameon=False, fontsize=16, bbox_to_anchor=(0.5, 0.0),
           columnspacing=1.5)


plt.tight_layout(rect=[0, 0.07, 1, 0.97])

fig.savefig(OUTPUT / "Architecture_Alluvial.png", dpi=600,
            bbox_inches="tight", facecolor="white")

plt.show()
plt.close(fig)
print("Saved:", OUTPUT)

In [ ]:
# =============================================================================
# COMBINED FIGURE
#   Row 1, panels (a)-(c): alluvial diagram of projected-change class by architecture
#   Row 2, panels (d)-(f): multi-architecture directional agreement heatmap

# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.gridspec import GridSpec
from matplotlib.path import Path as MPath
from matplotlib.patches import PathPatch, Rectangle, Patch
from matplotlib.lines import Line2D
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable

# =============================================================================
# 1. SETTINGS
# =============================================================================

FS = 18   # one font size for everything

DATA_DIR = PROJECT_DIR / "data"
PROJ_DIR = RESULTS_ROOT / "Projection_v4"

INPUT_ALLUV = PROJ_DIR / "percent_change_by_model.csv"
INPUT_ANNUAL = PROJ_DIR / "annual_change" / "annual_change_by_year.csv"

OUTPUT = PROJ_DIR / "Combined_Figure"
OUTPUT.mkdir(parents=True, exist_ok=True)

STATIONS = ["Shebe", "Asendabo", "Abelti"]
MODELS = ["LSTM", "GRU", "TCN", "MLP", "LSTM_FCN"]
LABEL = {"LSTM": "LSTM", "GRU": "GRU", "TCN": "TCN", "MLP": "MLP",
         "LSTM_FCN": "LSTM-FCN"}
COLORS = {"LSTM": "#0072B2", "GRU": "#E69F00", "TCN": "#009E73",
          "MLP": "#CC79A7", "LSTM_FCN": "#D55E00"}

CLASSES = [
    ("Large increase\n(> +50%)",         50.0,  np.inf),
    ("Moderate increase\n(+10 to +50%)", 10.0,  50.0),
    ("Little change\n(\u221210 to +10%)", -10.0, 10.0),
    ("Decrease\n(< \u221210%)",          -np.inf, -10.0),
]

GROUPS = [("SSP245", ["Near", "Far"]), ("SSP585", ["Near", "Far"])]
COLS = [(sc, per) for sc, pers in GROUPS for per in pers]
SCENARIO_PERIODS = COLS
PERIODS = {"Near": (2041, 2070), "Far": (2071, 2100)}

X = [0.0, 1.7, 3.5, 5.2]
NW = 0.20
LANE_H, LANE_GAP = 5.0, 0.9

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": FS,
    "axes.labelsize": FS,
    "axes.titlesize": FS,
    "xtick.labelsize": FS,
    "ytick.labelsize": FS,
    "legend.fontsize": FS,
    "figure.titlesize": FS,
    "axes.linewidth": 0.9,
})

AGREEMENT_CMAP = LinearSegmentedColormap.from_list(
    "agreement", ["#FFF8E7", "#FEE8A6", "#FDBB62", "#E76F51", "#9B2226"], N=256)

# =============================================================================
# 2. DATA: ALLUVIAL CLASSES
# =============================================================================

df = pd.read_csv(INPUT_ALLUV)
need = ["Station", "Model", "Scenario", "Period", "mean_change_%"]
miss = [c for c in need if c not in df.columns]
if miss:
    raise ValueError(f"Missing columns in {INPUT_ALLUV.name}: {miss}")
df = df[df["Model"].isin(MODELS)].copy()

ens = (df.groupby(["Station", "Scenario", "Period"], as_index=False)
         ["mean_change_%"].mean().assign(Model="Ensemble"))
df = pd.concat([df, ens], ignore_index=True)


def klass(v):
    for k, (_, lo, hi) in enumerate(CLASSES):
        if lo <= v < hi:
            return k
    return None


df["Class"] = df["mean_change_%"].apply(klass)
df.to_csv(OUTPUT / "alluvial_classes.csv", index=False)


def lane_bottom(k):
    return (len(CLASSES) - 1 - k) * (LANE_H + LANE_GAP)


total_h = len(CLASSES) * LANE_H + (len(CLASSES) - 1) * LANE_GAP

# =============================================================================
# 3. DATA: DIRECTIONAL AGREEMENT
# =============================================================================

ann = pd.read_csv(INPUT_ANNUAL)
req = ["Station", "Scenario", "Model", "Year", "Annual_change_%"]
miss = [c for c in req if c not in ann.columns]
if miss:
    raise ValueError(f"Missing columns in {INPUT_ANNUAL.name}: {miss}")

ann = ann[ann["Model"].isin(MODELS)].copy()
ann["Annual_change_%"] = pd.to_numeric(ann["Annual_change_%"], errors="coerce")
ann["Year"] = pd.to_numeric(ann["Year"], errors="coerce")
ann = ann.dropna(subset=req)

model_period = []
for station in STATIONS:
    for scenario, period in SCENARIO_PERIODS:
        start, end = PERIODS[period]
        sub = ann[(ann["Station"] == station) & (ann["Scenario"] == scenario)
                  & (ann["Year"].between(start, end))]
        for model in MODELS:
            s = sub[sub["Model"] == model]
            if s.empty:
                continue
            model_period.append({
                "Station": station, "Scenario": scenario, "Period": period,
                "Model": model,
                "Mean_Annual_Change_%": s["Annual_change_%"].mean(),
                "N_years": s["Year"].nunique()})
model_period = pd.DataFrame(model_period)
model_period.to_csv(OUTPUT / "model_directional_changes.csv", index=False)

agreement_rows = []
for station in STATIONS:
    for scenario, period in SCENARIO_PERIODS:
        sub = model_period[(model_period["Station"] == station)
                           & (model_period["Scenario"] == scenario)
                           & (model_period["Period"] == period)]
        if sub["Model"].nunique() != len(MODELS):
            print(f"WARNING: incomplete architectures: {station}, {scenario}, {period}")
            continue
        changes = sub.set_index("Model")["Mean_Annual_Change_%"].reindex(MODELS)
        if changes.isna().any():
            continue

        n_inc = int((changes > 0).sum())
        n_dec = int((changes < 0).sum())
        n_unc = int((changes == 0).sum())
        if n_inc > n_dec:
            direction, count = "Increase", n_inc
        elif n_dec > n_inc:
            direction, count = "Decrease", n_dec
        else:
            direction, count = "Tie", max(n_inc, n_dec)

        agreement_rows.append({
            "Station": station, "Scenario": scenario, "Period": period,
            "Direction": direction,
            "Agreement_%": count / len(MODELS) * 100,
            "N_Increase": n_inc, "N_Decrease": n_dec, "N_Unchanged": n_unc,
            "Mean_Model_Change_%": changes.mean(),
            "Minimum_Model_Change_%": changes.min(),
            "Maximum_Model_Change_%": changes.max(),
            "Architecture_Range_pp": changes.max() - changes.min(),
            "Models": "; ".join(f"{m}: {changes[m]:.2f}%" for m in MODELS)})

agreement_df = pd.DataFrame(agreement_rows)
agreement_df.to_csv(OUTPUT / "directional_agreement_summary.csv", index=False)

print("\nDIRECTIONAL AGREEMENT RESULTS")
print("=" * 75)
print(agreement_df[["Station", "Scenario", "Period", "Direction",
                    "Agreement_%", "Mean_Model_Change_%"]]
      .round(2).to_string(index=False))

# =============================================================================
# 4. DRAWING HELPERS
# =============================================================================


def ribbon(ax, xa, ya_hi, ya_lo, xb, yb_hi, yb_lo, color, alpha=0.62):
    xm = (xa + xb) / 2.0
    verts = [(xa, ya_hi), (xm, ya_hi), (xm, yb_hi), (xb, yb_hi),
             (xb, yb_lo), (xm, yb_lo), (xm, ya_lo), (xa, ya_lo), (xa, ya_hi)]
    codes = [MPath.MOVETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4,
             MPath.LINETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4,
             MPath.CLOSEPOLY]
    ax.add_patch(PathPatch(MPath(verts, codes), facecolor=color,
                           edgecolor="white", linewidth=0.5,
                           alpha=alpha, zorder=3))


def draw_alluvial(ax, st, letter):
    sub = df[df["Station"] == st]

    for k in range(len(CLASSES)):
        ax.add_patch(Rectangle((-0.45, lane_bottom(k)), X[-1] + NW + 0.9,
                               LANE_H, facecolor="#F3F3F3" if k % 2 == 0
                               else "#EAEAEA", edgecolor="none", zorder=0))

    cls = {}
    for ci, (sc, per) in enumerate(COLS):
        for m in MODELS + ["Ensemble"]:
            r = sub[(sub["Model"] == m) & (sub["Scenario"] == sc)
                    & (sub["Period"] == per)]
            cls[(ci, m)] = None if r.empty else int(r["Class"].iloc[0])

    slot, node_y0, node_cnt = {}, {}, {}
    for ci in range(len(COLS)):
        for k in range(len(CLASSES)):
            members = [m for m in MODELS if cls[(ci, m)] == k]
            c = len(members)
            node_cnt[(ci, k)] = c
            y0 = lane_bottom(k) + (LANE_H - c) / 2.0
            node_y0[(ci, k)] = y0
            for i, m in enumerate(members):
                slot[(ci, m)] = (y0 + c - i, y0 + c - i - 1)

    for first in (0, 2):
        a, b = first, first + 1
        for m in MODELS:
            if (a, m) in slot and (b, m) in slot:
                ta, ba = slot[(a, m)]
                tb, bb = slot[(b, m)]
                ribbon(ax, X[a] + NW, ta, ba, X[b], tb, bb, COLORS[m])

    for ci in range(len(COLS)):
        for k in range(len(CLASSES)):
            c = node_cnt[(ci, k)]
            if c == 0:
                continue
            ax.add_patch(Rectangle((X[ci], node_y0[(ci, k)]), NW, c,
                                   facecolor="#2b2b2b", edgecolor="white",
                                   linewidth=0.8, zorder=4))
            ax.text(X[ci] + NW / 2, node_y0[(ci, k)] + c / 2, str(c),
                    ha="center", va="center", color="white",
                    fontsize=FS, zorder=5)
        ke = cls[(ci, "Ensemble")]
        if ke is not None:
            xm = X[ci] - 0.17 if ci in (0, 2) else X[ci] + NW + 0.17
            ax.scatter(xm, lane_bottom(ke) + LANE_H / 2, marker="D", s=95,
                       color="black", edgecolor="white", linewidth=0.8,
                       zorder=6)

    for ci, (sc, per) in enumerate(COLS):
        ax.text(X[ci] + NW / 2, total_h + 0.35, per, ha="center",
                va="bottom", fontsize=FS)
    for g, (sc, pers) in enumerate(GROUPS):
        i0 = 2 * g
        xc = (X[i0] + X[i0 + 1] + NW) / 2
        ax.text(xc, total_h + 1.35, sc, ha="center", va="bottom", fontsize=FS)
        ax.plot([X[i0] - 0.1, X[i0 + 1] + NW + 0.1],
                [total_h + 1.25, total_h + 1.25], color="black", linewidth=1.2)

    ax.set_xlim(-0.55, X[-1] + NW + 0.5)
    ax.set_ylim(-0.4, total_h + 2.6)
    ax.set_xticks([])
    for s in ["top", "right", "bottom", "left"]:
        ax.spines[s].set_visible(False)
    ax.set_title(f"{letter} {st}", loc="left", pad=8, fontsize=FS)


def draw_heatmap(ax, st, letter, show_ylabel):
    sub = agreement_df[agreement_df["Station"] == st]
    matrix = np.full((1, 4), np.nan)
    annotations = [""] * 4

    for j, (scenario, period) in enumerate(SCENARIO_PERIODS):
        r = sub[(sub["Scenario"] == scenario) & (sub["Period"] == period)]
        if r.empty:
            continue
        row = r.iloc[0]
        matrix[0, j] = row["Agreement_%"]
        symbol = {"Increase": "\u2191", "Decrease": "\u2193"}.get(row["Direction"], "=")
        annotations[j] = f"{symbol} {row['Agreement_%']:.0f}%\n{row['Direction']}"

    ax.imshow(matrix, aspect="auto", cmap=AGREEMENT_CMAP,
              norm=Normalize(vmin=20, vmax=100), interpolation="nearest")
    ax.set_title(f"{letter} {st}", loc="left", pad=8, fontsize=FS)

    ax.set_xticks(range(4))
    ax.set_xticklabels(["Near\nSSP245", "Far\nSSP245",
                        "Near\nSSP585", "Far\nSSP585"], fontsize=FS)
    ax.set_yticks([0])
    ax.set_yticklabels(["Architecture\nagreement"] if show_ylabel else [],
                       fontsize=FS)
    ax.tick_params(length=0)

    ax.set_xticks(np.arange(-0.5, 4, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 1, 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=2)
    ax.tick_params(which="minor", length=0)

    for j in range(4):
        if not np.isfinite(matrix[0, j]):
            continue
        color = "white" if matrix[0, j] >= 80 else "#333333"
        ax.text(j, 0, annotations[j], ha="center", va="center",
                fontsize=FS, color=color)
    ax.set_xlim(-0.5, 3.5)

    for s in ax.spines.values():
        s.set_visible(False)



# =============================================================================
# 5. FIGURE LAYOUT — NESTED GRIDSPEC
#
# =============================================================================

fig = plt.figure(figsize=(19, 14.5))

# Main layout: upper alluvial section and lower heatmap section
outer_gs = fig.add_gridspec(
    2, 1,
    height_ratios=[9.0, 3.05],
    left=0.11,
    right=0.99,
    top=0.96,
    bottom=0.04,
    hspace=0.05
)

# -------------------------------------------------------------------------
# UPPER SECTION: alluvial diagrams + legend
# -------------------------------------------------------------------------

top_gs = outer_gs[0].subgridspec(
    2, 3,
    height_ratios=[8.2, 0.8],
    hspace=0.01,       # Controls alluvial-to-legend spacing
    wspace=0.08
)

# --- Row 1: alluvial panels (a)-(c) --------------------------------------

allu_axes = []

for i, st in enumerate(STATIONS):
    ax = fig.add_subplot(
        top_gs[0, i],
        sharey=allu_axes[0] if allu_axes else None
    )

    draw_alluvial(ax, st, f"({'abc'[i]})")
    allu_axes.append(ax)

allu_axes[0].set_yticks(
    [lane_bottom(k) + LANE_H / 2 for k in range(len(CLASSES))]
)

allu_axes[0].set_yticklabels(
    [c[0] for c in CLASSES],
    fontsize=FS
)

allu_axes[0].tick_params(axis="y", length=0)

for a in allu_axes[1:]:
    a.tick_params(axis="y", left=False, labelleft=False)


# --- Row 2: alluvial legend ----------------------------------------------

leg_ax = fig.add_subplot(top_gs[1, :])
leg_ax.axis("off")

handles = [
    Patch(facecolor=COLORS[m], alpha=0.8, label=LABEL[m])
    for m in MODELS
]

handles.append(
    Line2D(
        [0], [0],
        marker="D",
        color="w",
        markerfacecolor="black",
        markeredgecolor="white",
        markersize=10,
        label="Ensemble mean class"
    )
)

handles.append(
    Patch(
        facecolor="#2b2b2b",
        label="Node = number of architectures"
    )
)

leg_ax.legend(
    handles=handles,
    loc="center",
    bbox_to_anchor=(0.5, 0.5),
    ncol=4,
    frameon=False,
    fontsize=FS,
    columnspacing=1.8,
    handlelength=1.6,
    borderaxespad=0.0,
    labelspacing=0.2,
    handletextpad=0.5
)


# -------------------------------------------------------------------------
# LOWER SECTION: heatmaps + colour bar
# -------------------------------------------------------------------------

heat_gs = outer_gs[1].subgridspec(
    2, 3,
    height_ratios=[2.6, 0.45],
    hspace=0.6,       # Controls heatmap-to-colourbar spacing
    wspace=0.08
)

# --- Row 1: heatmaps (d)-(f) ---------------------------------------------

heat_axes = []

for i, st in enumerate(STATIONS):
    ax = fig.add_subplot(heat_gs[0, i])

    draw_heatmap(
        ax,
        st,
        f"({'def'[i]})",
        show_ylabel=(i == 0)
    )

    heat_axes.append(ax)


# --- Row 2: heatmap colour bar -------------------------------------------

cax = fig.add_subplot(heat_gs[1, 1])

sm = ScalarMappable(
    norm=Normalize(vmin=20, vmax=100),
    cmap=AGREEMENT_CMAP
)

sm.set_array([])

cbar = fig.colorbar(
    sm,
    cax=cax,
    orientation="horizontal"
)

cbar.set_label(
    "Directional agreement among architectures (%)",
    fontsize=FS
)

cbar.ax.tick_params(labelsize=FS)


# =============================================================================
# 6. EXPORT
# =============================================================================

fig.savefig(
    OUTPUT / "Combined_Alluvial_Agreement.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    OUTPUT / "Combined_Alluvial_Agreement.pdf",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

print("\nSaved:", OUTPUT)

# =============================================================================
# 6. EXPORT
# =============================================================================

fig.savefig(OUTPUT / "Combined_Alluvial_Agreement.png", dpi=600,
            bbox_inches="tight", facecolor="white")
fig.savefig(OUTPUT / "Combined_Alluvial_Agreement.pdf",
            bbox_inches="tight", facecolor="white")

plt.show()
plt.close(fig)
print("\nSaved:", OUTPUT)

In [ ]:
# =============================================================================
#  MODEL PERFORMANCE COMPARISON (five architectures + SWAT)
#   (a) NSE   validation vs test
#   (b) KGE   validation vs test
#   (c) Composite score (test)
#   (d) PBIAS (test)
#   (e) RMSE  (test)
# Times New Roman, 16 pt
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Patch

FS = 16

OUTPUT = Path(os.environ.get(
    "PERF_OUT",
    str(RESULTS_ROOT / "Projection_v4" / "Model_Performance")))
OUTPUT.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": FS, "axes.labelsize": FS, "axes.titlesize": FS,
    "xtick.labelsize": FS, "ytick.labelsize": FS, "legend.fontsize": FS,
    "axes.linewidth": 0.9,
})

# -----------------------------------------------------------------------------
# DATA (from the table)
# -----------------------------------------------------------------------------
rows = [
    # model,      lookback, config,            val_NSE, val_KGE, NSE,  KGE,  PBIAS, RMSE,   Composite
    ("GRU",       90,  "GRU128_64_D64",  0.72, 0.83, 0.66, 0.66, 24.68, 99.59,  0.68),
    ("MLP",       30,  "MLP256_128_D64", 0.59, 0.79, 0.65, 0.71, 18.49, 100.74, 0.70),
    ("TCN",       90,  "TCN64_B6",       0.77, 0.82, 0.58, 0.59, 24.13, 110.52, 0.62),
    ("LSTM",      45,  "LSTM128_D32",    0.74, 0.84, 0.55, 0.56, 29.75, 114.45, 0.58),
    ("LSTM-FCN",  120, "LSTMFCN64_F64",  0.76, 0.82, 0.38, 0.37, 34.76, 134.47, 0.43),
    ("SWAT",      np.nan, "-",           0.54, 0.58, 0.61, 0.62, 25.5,  108.3,  np.nan),
]
cols = ["Model", "Lookback", "Config", "Val_NSE", "Val_KGE",
        "Test_NSE", "Test_KGE", "Test_PBIAS", "Test_RMSE", "Composite"]
d = pd.DataFrame(rows, columns=cols)
d.to_csv(OUTPUT / "model_performance_table.csv", index=False)

names = d["Model"].tolist()
x = np.arange(len(names))
is_swat = np.array([n == "SWAT" for n in names])

C_VAL, C_TEST, C_SWAT = "#9ECAE1", "#08519C", "#7F7F7F"
MODEL_COL = {"GRU": "#E69F00", "MLP": "#CC79A7", "TCN": "#009E73",
             "LSTM": "#0072B2", "LSTM-FCN": "#D55E00", "SWAT": C_SWAT}


def style(ax, ylabel, title, ymax, ticks=None):
    ax.set_title(title, loc="left", pad=8, fontsize=FS)
    ax.set_ylabel(ylabel, fontsize=FS)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=45, ha="right", fontsize=FS)
    ax.set_ylim(0, ymax)
    if ticks is not None:
        ax.set_yticks(ticks)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="#DDDDDD", linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)


def grouped(ax, vcol, tcol, ylabel, title, ymax, ticks):
    w = 0.38
    for i, n in enumerate(names):
        hatch = "//" if is_swat[i] else None
        ax.bar(x[i] - w / 2, d[vcol][i], w, color=C_VAL, edgecolor="black",
               linewidth=0.6, hatch=hatch, zorder=3)
        ax.bar(x[i] + w / 2, d[tcol][i], w, color=C_TEST, edgecolor="black",
               linewidth=0.6, hatch=hatch, zorder=3)
        for xx, v in ((x[i] - w / 2, d[vcol][i]), (x[i] + w / 2, d[tcol][i])):
            ax.text(xx, v + ymax * 0.015, f"{v:.2f}", rotation=90,
                    ha="center", va="bottom", fontsize=FS)
    style(ax, ylabel, title, ymax, ticks)


def single(ax, col, ylabel, title, ymax, fmt, ticks=None):
    for i, n in enumerate(names):
        v = d[col][i]
        if np.isnan(v):
            ax.text(x[i], ymax * 0.02, "\u2013", ha="center", va="bottom",
                    fontsize=FS)
            continue
        ax.bar(x[i], v, 0.6, color=MODEL_COL[n], edgecolor="black",
               linewidth=0.6, hatch="//" if is_swat[i] else None, zorder=3)
        ax.text(x[i], v + ymax * 0.015, fmt.format(v), ha="center",
                va="bottom", fontsize=FS)
    style(ax, ylabel, title, ymax, ticks)


fig, axes = plt.subplots(2, 3, figsize=(19, 11.5))
ax = axes.ravel()

grouped(ax[0], "Val_NSE", "Test_NSE", "NSE (\u2013)", "(a) NSE",
        1.25, np.arange(0, 1.01, 0.25))
grouped(ax[1], "Val_KGE", "Test_KGE", "KGE (\u2013)", "(b) KGE",
        1.25, np.arange(0, 1.01, 0.25))
single(ax[2], "Composite", "Composite score (\u2013)", "(c) Composite (test)",
       0.9, "{:.2f}", np.arange(0, 0.81, 0.2))
single(ax[3], "Test_PBIAS", "PBIAS (%)", "(d) PBIAS (test)",
       45, "{:.1f}")
single(ax[4], "Test_RMSE", "RMSE", "(e) RMSE (test)",
       165, "{:.1f}")

# legend in the empty sixth panel
ax[5].axis("off")
handles = [
    Patch(facecolor=C_VAL, edgecolor="black", label="Validation"),
    Patch(facecolor=C_TEST, edgecolor="black", label="Test"),
    Patch(facecolor="white", edgecolor="black", hatch="//",
          label="SWAT (benchmark)"),
]
ax[5].legend(handles=handles, loc="center", frameon=False, fontsize=FS,
             title="Panels (a)\u2013(b)", title_fontsize=FS)

fig.tight_layout()
fig.savefig(OUTPUT / "Model_Performance.png", dpi=600,
            bbox_inches="tight", facecolor="white")
fig.savefig(OUTPUT / "Model_Performance.pdf",
            bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)
print("Saved:", OUTPUT)

In [ ]:
# =============================================================================
# COMBINED FIGURE: TAYLOR DIAGRAMS + MODEL PERFORMANCE COMPARISON
#
#   Rows 1-2, panels (a)-(f): Taylor diagrams (validation row, test row;
#                              one column per station)
#   Rows 3-4, panels (g)-(k): performance comparison of the five architectures
#                              and SWAT
#       (g) NSE   validation vs test
#       (h) KGE   validation vs test
#       (i) Composite score (test)
#       (j) PBIAS (test)
#       (k) RMSE  (test)
#
# Times New Roman, 16 pt everywhere.
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

FS = 18

# =============================================================================
# 1. SETTINGS (edit here)
# =============================================================================

DATA_DIR = PROJECT_DIR / "data"
OUTPUT = RESULTS_ROOT / "Combined_Taylor_Performance"
OUTPUT.mkdir(parents=True, exist_ok=True)

RESULT_DIRS = {
    "LSTM":     RESULTS_ROOT / "LSTM_Results_v4",
    "GRU":      RESULTS_ROOT / "GRU_Results_v4",
    "TCN":      RESULTS_ROOT / "TCN_Results_v4",
    "MLP":      RESULTS_ROOT / "MLP_Results_v4",
    "LSTM-FCN": RESULTS_ROOT / "LSTM_FCN_Results_v4",
}

STATIONS = ["Shebe", "Asendabo", "Abelti"]

PERIODS = [
    ("Validation 2016\u20132018", "2016-01-01", "2018-12-31"),
    ("Test 2019\u20132021",       "2019-01-01", "2021-12-31"),
]

MODELS = ["SWAT", "LSTM", "GRU", "TCN", "MLP", "LSTM-FCN", "Ensemble"]
ARCHS = ["LSTM", "GRU", "TCN", "MLP", "LSTM-FCN"]

STYLE = {  # colour, marker, size (Okabe-Ito)
    "SWAT":     ("#7F7F7F", "P", 120),
    "LSTM":     ("#0072B2", "o", 105),
    "GRU":      ("#E69F00", "s", 105),
    "TCN":      ("#009E73", "^", 120),
    "MLP":      ("#CC79A7", "D", 95),
    "LSTM-FCN": ("#D55E00", "v", 120),
    "Ensemble": ("#000000", "*", 260),
}

CORR_TICKS = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0]
RMS_LEVELS = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5]

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": FS, "axes.labelsize": FS, "axes.titlesize": FS,
    "xtick.labelsize": FS, "ytick.labelsize": FS, "legend.fontsize": FS,
    "axes.linewidth": 0.9,
})

# =============================================================================
# 2. TAYLOR: LOAD AND CHECK DATA
# =============================================================================

NEEDED = ["Station", "Date", "Observed", "FLOW_OUT", "Corrected_Flow"]


def load_arch(arch, folder):
    pred_dir = folder / "predictions"
    frames, seen = [], []
    for f in sorted(pred_dir.glob("*.csv")):
        seen.append(f.name)
        try:
            d = pd.read_csv(f)
        except Exception:
            continue
        if set(NEEDED) <= set(d.columns):
            d = d[NEEDED].copy()
            d["Date"] = pd.to_datetime(d["Date"])
            frames.append(d)
    if not frames:
        raise FileNotFoundError(
            f"No prediction CSV with columns {NEEDED} in {pred_dir}. "
            f"Files seen: {seen}")
    out = pd.concat(frames).drop_duplicates(["Station", "Date"])
    print(f"{arch}: {len(out)} rows from {len(frames)} file(s) in {pred_dir.name}")
    return out


df = None
for arch, folder in RESULT_DIRS.items():
    d = load_arch(arch, folder)
    if df is None:
        df = d.rename(columns={"FLOW_OUT": "SWAT", "Corrected_Flow": arch})
    else:
        d = d.rename(columns={"Observed": "Observed_chk",
                              "Corrected_Flow": arch})[
            ["Station", "Date", "Observed_chk", arch]]
        df = df.merge(d, on=["Station", "Date"], how="inner")
        diff = (df["Observed"] - df["Observed_chk"]).abs().max()
        if diff > 1e-6:
            print(f"WARNING: Observed differs between LSTM and {arch} "
                  f"(max abs diff {diff:.3g}); check the folders.")
        df = df.drop(columns="Observed_chk")

df["Ensemble"] = df[ARCHS].mean(axis=1)

PERIODS = [p for p in PERIODS
           if all(((df["Station"] == st) & (df["Date"] >= p[1])
                   & (df["Date"] <= p[2])).sum() >= 30 for st in STATIONS)]
if not PERIODS:
    raise ValueError("No period has data for all stations; check the dates.")
print("Periods used:", [p[0] for p in PERIODS])

# =============================================================================
# 3. TAYLOR STATISTICS
# =============================================================================


def taylor_stats(obs, sim):
    so = obs.std(ddof=0)
    ss = sim.std(ddof=0)
    r = np.corrcoef(obs, sim)[0, 1]
    s = ss / so
    crmsd = np.sqrt(1.0 + s ** 2 - 2.0 * s * r)
    return s, r, crmsd, ss, so


rows = []
panel_stats = {}
for plabel, start, end in PERIODS:
    for st in STATIONS:
        sub = df[(df["Station"] == st)
                 & (df["Date"] >= start) & (df["Date"] <= end)]
        sub = sub.dropna(subset=["Observed"] + MODELS)
        if len(sub) < 30:
            print(f"WARNING: only {len(sub)} days for {st}, {plabel}")
        res = {}
        for m in MODELS:
            s, r, c, ss, so = taylor_stats(sub["Observed"].to_numpy(),
                                           sub[m].to_numpy())
            res[m] = (s, r, c)
            rows.append({"Period": plabel, "Station": st, "Model": m,
                         "N_days": len(sub), "Std_obs": so, "Std_sim": ss,
                         "Norm_std": s, "Correlation": r,
                         "Norm_centred_RMSD": c})
        panel_stats[(plabel, st)] = res

stats_df = pd.DataFrame(rows)
stats_df.to_csv(OUTPUT / "taylor_statistics.csv", index=False)
print(stats_df.round(3).to_string(index=False))

max_s = max(v[0] for res in panel_stats.values() for v in res.values())
R = float(np.ceil(max(1.5, max_s * 1.08) * 4) / 4)

# =============================================================================
# 4. PERFORMANCE TABLE
# =============================================================================

perf_rows = [
    # model,      lookback, config,           val_NSE, val_KGE, NSE,  KGE,  PBIAS, RMSE,   Composite
    ("GRU",       90,  "GRU128_64_D64",  0.72, 0.83, 0.66, 0.66, 24.68, 99.59,  0.68),
    ("MLP",       30,  "MLP256_128_D64", 0.59, 0.79, 0.65, 0.71, 18.49, 100.74, 0.70),
    ("TCN",       90,  "TCN64_B6",       0.77, 0.82, 0.58, 0.59, 24.13, 110.52, 0.62),
    ("LSTM",      45,  "LSTM128_D32",    0.74, 0.84, 0.55, 0.56, 29.75, 114.45, 0.58),
    ("LSTM-FCN",  120, "LSTMFCN64_F64",  0.76, 0.82, 0.38, 0.37, 34.76, 134.47, 0.43),
    ("SWAT",      np.nan, "-",           0.54, 0.58, 0.61, 0.62, 25.5,  108.3,  np.nan),
]
perf_cols = ["Model", "Lookback", "Config", "Val_NSE", "Val_KGE",
             "Test_NSE", "Test_KGE", "Test_PBIAS", "Test_RMSE", "Composite"]
perf = pd.DataFrame(perf_rows, columns=perf_cols)
perf.to_csv(OUTPUT / "model_performance_table.csv", index=False)

names = perf["Model"].tolist()
xb = np.arange(len(names))
is_swat = np.array([n == "SWAT" for n in names])

C_VAL, C_TEST, C_SWAT = "#9ECAE1", "#08519C", "#7F7F7F"
MODEL_COL = {"GRU": "#E69F00", "MLP": "#CC79A7", "TCN": "#009E73",
             "LSTM": "#0072B2", "LSTM-FCN": "#D55E00", "SWAT": C_SWAT}

# =============================================================================
# 5. DRAWING FUNCTIONS: TAYLOR
# =============================================================================


def draw_frame(ax, R):
    th = np.linspace(0, np.pi / 2, 400)

    for s in np.arange(0.25, R, 0.25):
        if abs(s - 1.0) < 1e-9:
            continue
        ax.plot(s * np.cos(th), s * np.sin(th), color="#9a9a9a",
                linestyle=":", linewidth=0.8, zorder=1)
    ax.plot(R * np.cos(th), R * np.sin(th), color="black", linewidth=1.4,
            zorder=2)
    ax.plot(np.cos(th), np.sin(th), color="black", linestyle="--",
            linewidth=1.1, zorder=2)

    for r in CORR_TICKS:
        a = np.arccos(r)
        ax.plot([0, R * np.cos(a)], [0, R * np.sin(a)], color="#777777",
                linestyle=":", linewidth=0.8, zorder=1)
        ax.text(1.035 * R * np.cos(a), 1.035 * R * np.sin(a),
                f"{r:g}", ha="left", va="bottom" if r > 0.5 else "center",
                fontsize=FS, color="#222222")
    ax.text(0.80 * R, 0.80 * R * 0.78 + 0.22 * R, "Correlation",
            rotation=-45, ha="center", va="center", fontsize=FS)

    for rho in RMS_LEVELS:
        t = np.linspace(0, np.pi, 600)
        x = 1 + rho * np.cos(t)
        y = rho * np.sin(t)
        keep = (x ** 2 + y ** 2 <= R ** 2 + 1e-9) & (x >= 0) & (y >= 0)
        ax.plot(np.where(keep, x, np.nan), np.where(keep, y, np.nan),
                color="#C0392B", linewidth=0.9, zorder=1)
        tx, ty = 1 + rho * np.cos(np.radians(125)), rho * np.sin(np.radians(125))
        if tx > 0.03 and tx ** 2 + ty ** 2 < R ** 2:
            ax.text(tx, ty, f"{rho:g}", color="#C0392B", fontsize=FS,
                    ha="center", va="center",
                    bbox=dict(facecolor="white", edgecolor="none",
                              pad=0.6, alpha=0.9), zorder=3)

    ax.set_xlim(-0.02 * R, 1.10 * R)
    ax.set_ylim(-0.02 * R, 1.10 * R)
    ax.set_aspect("equal")
    ax.set_xticks(np.arange(0, R + 1e-9, 0.5))
    ax.set_yticks(np.arange(0, R + 1e-9, 0.5))
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)


# =============================================================================
# 6. DRAWING FUNCTIONS: PERFORMANCE BARS
# =============================================================================


def style_bar(ax, ylabel, title, ymax, ticks=None):
    ax.set_title(title, loc="left", pad=8, fontsize=FS)
    ax.set_ylabel(ylabel, fontsize=FS)
    ax.set_xticks(xb)
    ax.set_xticklabels(names, rotation=45, ha="right", fontsize=FS)
    ax.set_ylim(0, ymax)
    if ticks is not None:
        ax.set_yticks(ticks)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="#DDDDDD", linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)


def grouped(ax, vcol, tcol, ylabel, title, ymax, ticks):
    w = 0.38
    for i, n in enumerate(names):
        hatch = "//" if is_swat[i] else None
        ax.bar(xb[i] - w / 2, perf[vcol][i], w, color=C_VAL, edgecolor="black",
               linewidth=0.6, hatch=hatch, zorder=3)
        ax.bar(xb[i] + w / 2, perf[tcol][i], w, color=C_TEST, edgecolor="black",
               linewidth=0.6, hatch=hatch, zorder=3)
        for xx, v in ((xb[i] - w / 2, perf[vcol][i]),
                      (xb[i] + w / 2, perf[tcol][i])):
            ax.text(xx, v + ymax * 0.015, f"{v:.2f}", rotation=90,
                    ha="center", va="bottom", fontsize=FS)
    style_bar(ax, ylabel, title, ymax, ticks)


def single(ax, col, ylabel, title, ymax, fmt, ticks=None):
    for i, n in enumerate(names):
        v = perf[col][i]
        if np.isnan(v):
            ax.text(xb[i], ymax * 0.02, "\u2013", ha="center", va="bottom",
                    fontsize=FS)
            continue
        ax.bar(xb[i], v, 0.6, color=MODEL_COL[n], edgecolor="black",
               linewidth=0.6, hatch="//" if is_swat[i] else None, zorder=3)
        ax.text(xb[i], v + ymax * 0.015, fmt.format(v), ha="center",
                va="bottom", fontsize=FS)
    style_bar(ax, ylabel, title, ymax, ticks)



# =============================================================================
# 7. FIGURE LAYOUT — INDEPENDENT SPACING CONTROL
# =============================================================================

nT = len(PERIODS)
ncols = len(STATIONS)

# -------------------------------------------------------------------------
# INDEPENDENT SPACING CONTROLS — EDIT THESE VALUES
# -------------------------------------------------------------------------

TAYLOR_LEGEND_GAP = 0.01   # Gap between Taylor diagrams and Taylor legend
LEGEND_BAR_GAP    = 0.06   # Gap between Taylor legend and performance bars

# Taylor legend row height
TAYLOR_LEGEND_HEIGHT = 0.85

# Spacing between Taylor diagram columns
TAYLOR_WSPACE = 0.25

# Spacing between performance bar graphs
BAR_WSPACE = 0.25

# Spacing between the two rows of performance graphs
BAR_HSPACE = 0.35

# -------------------------------------------------------------------------
# MAIN FIGURE
# -------------------------------------------------------------------------

fig = plt.figure(
    figsize=(6.4 * ncols + 0.8,
             6.3 * nT + 1.0 + 10.4 + 1.5)
)

# Outer grid:
# Row 1 = Taylor diagrams + Taylor legend
# Row 2 = performance graphs
outer_gs = fig.add_gridspec(
    2, 1,
    height_ratios=[
        6.3 * nT + TAYLOR_LEGEND_HEIGHT,
        10.4
    ],
    left=0.08,
    right=0.99,
    top=0.975,
    bottom=0.04,
    hspace=LEGEND_BAR_GAP
)

# -------------------------------------------------------------------------
# UPPER SECTION: TAYLOR DIAGRAMS + THEIR LEGEND
# -------------------------------------------------------------------------

top_gs = outer_gs[0].subgridspec(
    nT + 1,
    ncols,
    height_ratios=[6.3] * nT + [TAYLOR_LEGEND_HEIGHT],
    wspace=TAYLOR_WSPACE,
    hspace=TAYLOR_LEGEND_GAP
)

# -------------------------------------------------------------------------
# LOWER SECTION: PERFORMANCE BAR GRAPHS
# -------------------------------------------------------------------------

bar_gs = outer_gs[1].subgridspec(
    2, 3,
    wspace=BAR_WSPACE,
    hspace=BAR_HSPACE
)

letters = "abcdefghijklmnop"
k = 0

# =============================================================================
# TAYLOR DIAGRAMS
# =============================================================================

for i, (plabel, start, end) in enumerate(PERIODS):

    for j, st in enumerate(STATIONS):

        ax = fig.add_subplot(top_gs[i, j])

        draw_frame(ax, R)

        res = panel_stats[(plabel, st)]

        # Observed reference point
        ax.scatter(
            1, 0,
            s=140,
            color="black",
            edgecolor="white",
            linewidth=1.0,
            zorder=6,
            clip_on=False
        )

        # Model points
        for m in MODELS:

            s, r, c = res[m]
            col, mk, sz = STYLE[m]

            if r < 0:
                print(
                    f"NOTE: negative correlation for {m}, "
                    f"{st}, {plabel} (not shown on the quarter diagram)"
                )
                continue

            th = np.arccos(min(r, 1.0))

            ax.scatter(
                s * np.cos(th),
                s * np.sin(th),
                marker=mk,
                s=sz,
                color=col,
                edgecolor="black",
                linewidth=0.8,
                zorder=5
            )

        ax.set_title(
            f"({letters[k]}) {st} | {plabel}",
            loc="left",
            pad=22
        )

        if j == 0:
            ax.set_ylabel("Normalised standard deviation")

        if i == nT - 1:
            ax.set_xlabel("Normalised standard deviation")

        k += 1


# =============================================================================
# TAYLOR LEGEND
# =============================================================================

leg_ax = fig.add_subplot(top_gs[nT, :])
leg_ax.axis("off")

handles = [
    Line2D(
        [0], [0],
        marker="o",
        color="w",
        markerfacecolor="black",
        markeredgecolor="white",
        markersize=11,
        label="Observed"
    )
]

for m in MODELS:

    col, mk, sz = STYLE[m]

    label = "Ensemble mean" if m == "Ensemble" else m

    handles.append(
        Line2D(
            [0], [0],
            marker=mk,
            color="w",
            markerfacecolor=col,
            markeredgecolor="black",
            markersize=np.sqrt(sz) * (0.95 if mk != "*" else 0.85),
            label=label
        )
    )

handles.append(
    Line2D(
        [0], [0],
        color="#C0392B",
        linewidth=1.2,
        label="Centred RMS difference"
    )
)

leg_ax.legend(
    handles=handles,
    loc="center",
    ncol=5,
    frameon=False,
    fontsize=FS,
    columnspacing=1.6,
    handletextpad=0.4
)


# =============================================================================
# PERFORMANCE BAR GRAPHS
# =============================================================================

# Six slots arranged in 2 rows × 3 columns
bax = [
    fig.add_subplot(bar_gs[n // 3, n % 3])
    for n in range(6)
]

# (g) NSE
grouped(
    bax[0],
    "Val_NSE",
    "Test_NSE",
    "NSE (–)",
    f"({letters[k]}) NSE",
    1.25,
    np.arange(0, 1.01, 0.25)
)
k += 1

# (h) KGE
grouped(
    bax[1],
    "Val_KGE",
    "Test_KGE",
    "KGE (–)",
    f"({letters[k]}) KGE",
    1.25,
    np.arange(0, 1.01, 0.25)
)
k += 1

# (i) Composite score
single(
    bax[2],
    "Composite",
    "Composite score (–)",
    f"({letters[k]}) Composite (test)",
    0.9,
    "{:.2f}",
    np.arange(0, 0.81, 0.2)
)
k += 1

# (j) PBIAS
single(
    bax[3],
    "Test_PBIAS",
    "PBIAS (%)",
    f"({letters[k]}) PBIAS (test)",
    45,
    "{:.1f}"
)
k += 1

# (k) RMSE
single(
    bax[4],
    "Test_RMSE",
    "RMSE",
    f"({letters[k]}) RMSE (test)",
    165,
    "{:.1f}"
)
k += 1

# Legend in the sixth slot
bax[5].axis("off")

bar_handles = [
    Patch(
        facecolor=C_VAL,
        edgecolor="black",
        label="Validation"
    ),
    Patch(
        facecolor=C_TEST,
        edgecolor="black",
        label="Test"
    ),
    Patch(
        facecolor="white",
        edgecolor="black",
        hatch="//",
        label="SWAT (benchmark)"
    )
]

bax[5].legend(
    handles=bar_handles,
    loc="center",
    frameon=False,
    fontsize=FS,
    title="Performance comparison",
    title_fontsize=FS
)



# =============================================================================
# 8. EXPORT
# =============================================================================

fig.savefig(OUTPUT / "Taylor_and_Performance.png", dpi=600,
            bbox_inches="tight", facecolor="white")
fig.savefig(OUTPUT / "Taylor_and_Performance.pdf",
            bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)
print("Saved:", OUTPUT)

In [ ]:

# =============================================================================
# FIGURE: MULTI-METRIC MODEL PERFORMANCE HEATMAP
# Five deep-learning architectures + SWAT benchmark
#
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from pathlib import Path
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Rectangle

# =============================================================================
# 1. SETTINGS
# =============================================================================

FS = 18

OUTPUT = RESULTS_ROOT / "Projection_v4" / "Model_Performance"

OUTPUT.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": FS,
    "axes.labelsize": FS,
    "axes.titlesize": FS,
    "xtick.labelsize": FS,
    "ytick.labelsize": FS,
    "legend.fontsize": FS,
    "axes.linewidth": 0.9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

# =============================================================================
# 2. PERFORMANCE DATA
# =============================================================================

rows = [
    # Model,      Val_NSE, Val_KGE, Test_NSE, Test_KGE, PBIAS, RMSE, Composite

    ("GRU",       0.72, 0.83, 0.66, 0.66, 24.68,  99.59, 0.68),
    ("MLP",       0.59, 0.79, 0.65, 0.71, 18.49, 100.74, 0.70),
    ("TCN",       0.77, 0.82, 0.58, 0.59, 24.13, 110.52, 0.62),
    ("LSTM",      0.74, 0.84, 0.55, 0.56, 29.75, 114.45, 0.58),
    ("LSTM-FCN",  0.76, 0.82, 0.38, 0.37, 34.76, 134.47, 0.43),
    ("SWAT",      0.54, 0.58, 0.61, 0.62, 25.50, 108.30, np.nan)
]

columns = [
    "Model",
    "Val_NSE",
    "Val_KGE",
    "Test_NSE",
    "Test_KGE",
    "Test_PBIAS",
    "Test_RMSE",
    "Composite"
]

df = pd.DataFrame(rows, columns=columns)

# Save original performance table
df.to_csv(
    OUTPUT / "model_performance_heatmap_data.csv",
    index=False
)

# =============================================================================
# 3. PREPARE PERFORMANCE MATRIX
# =============================================================================

# Display order of metrics
metrics = [
    "Val_NSE",
    "Test_NSE",
    "Val_KGE",
    "Test_KGE",
    "Composite",
    "Test_PBIAS",
    "Test_RMSE"
]

metric_labels = [
    "Validation\nNSE",
    "Test\nNSE",
    "Validation\nKGE",
    "Test\nKGE",
    "Test\nComposite",
    "Test\nPBIAS (%)",
    "Test\nRMSE"
]

# Higher values are better for these metrics.
higher_is_better = [
    "Val_NSE",
    "Test_NSE",
    "Val_KGE",
    "Test_KGE",
    "Composite"
]

# Lower absolute values are better for these metrics.
lower_is_better = [
    "Test_PBIAS",
    "Test_RMSE"
]

# Original values
values = df[metrics].to_numpy(dtype=float)

# Normalized performance matrix
performance = np.full(values.shape, np.nan)

for j, metric in enumerate(metrics):

    column = values[:, j]
    valid = np.isfinite(column)

    if not valid.any():
        continue

    # PBIAS is evaluated by its absolute magnitude.
    if metric == "Test_PBIAS":
        column_for_score = np.abs(column[valid])
    else:
        column_for_score = column[valid]

    minimum = np.min(column_for_score)
    maximum = np.max(column_for_score)

    if np.isclose(maximum, minimum):
        normalized = np.full(len(column_for_score), 0.5)

    else:
        normalized = (
            (column_for_score - minimum) /
            (maximum - minimum)
        )

    # Reverse the scale for metrics where lower is better.
    if metric in lower_is_better:
        normalized = 1.0 - normalized

    performance[valid, j] = normalized

# Save normalized performance scores
norm_df = pd.DataFrame(
    performance,
    columns=metrics
)

norm_df.insert(0, "Model", df["Model"])

norm_df.to_csv(
    OUTPUT / "normalized_performance_scores.csv",
    index=False
)

# =============================================================================
# 4. CUSTOM COLOUR MAP
# =============================================================================

# Low performance: pale cream
# Intermediate: orange
# High performance: dark red

cmap = LinearSegmentedColormap.from_list(
    "performance_map",
    [
        "#FFF7EC",
        "#FDD49E",
        "#FC8D59",
        "#D7301F",
        "#7F0000"
    ],
    N=256
)

cmap.set_bad("#E5E5E5")

# =============================================================================
# 5. DRAW HEATMAP
# =============================================================================

fig, ax = plt.subplots(figsize=(17, 9))

im = ax.imshow(
    performance,
    cmap=cmap,
    norm=Normalize(vmin=0, vmax=1),
    aspect="auto",
    interpolation="nearest"
)

n_models = len(df)
n_metrics = len(metrics)

# -------------------------------------------------------------------------
# Axis labels
# -------------------------------------------------------------------------

ax.set_xticks(np.arange(n_metrics))

ax.set_xticklabels(
    metric_labels,
    fontsize=FS,
    ha="center"
)

ax.set_yticks(np.arange(n_models))

ax.set_yticklabels(
    df["Model"].tolist(),
    fontsize=FS
)

ax.tick_params(
    axis="both",
    length=0,
    pad=12
)

# -------------------------------------------------------------------------
# Cell borders
# -------------------------------------------------------------------------

ax.set_xticks(
    np.arange(-0.5, n_metrics, 1),
    minor=True
)

ax.set_yticks(
    np.arange(-0.5, n_models, 1),
    minor=True
)

ax.grid(
    which="minor",
    color="white",
    linestyle="-",
    linewidth=2
)

ax.tick_params(
    which="minor",
    bottom=False,
    left=False
)

# -------------------------------------------------------------------------
# Original values inside each cell
# -------------------------------------------------------------------------

for i in range(n_models):

    for j, metric in enumerate(metrics):

        original_value = values[i, j]
        score = performance[i, j]

        if not np.isfinite(original_value):

            annotation = "—"
            text_color = "#555555"

        else:

            if metric == "Test_PBIAS":
                annotation = f"{original_value:.1f}%"

            elif metric == "Test_RMSE":
                annotation = f"{original_value:.1f}"

            else:
                annotation = f"{original_value:.2f}"

            # Text contrast according to background intensity.
            text_color = "white" if score >= 0.65 else "#222222"

        ax.text(
            j,
            i,
            annotation,
            ha="center",
            va="center",
            fontsize=FS,
            color=text_color,
            fontweight="normal"
        )

# -------------------------------------------------------------------------
# Separate SWAT benchmark visually
# -------------------------------------------------------------------------

# Horizontal separator above SWAT
ax.axhline(
    4.5,
    color="black",
    linewidth=2.0,
    linestyle="-",
    zorder=5
)

# Add subtle outline around SWAT row
ax.add_patch(
    Rectangle(
        (-0.5, 4.5),
        n_metrics,
        1,
        fill=False,
        edgecolor="#333333",
        linewidth=1.3,
        zorder=6
    )
)

# -------------------------------------------------------------------------
# Color bar
# -------------------------------------------------------------------------

cbar = fig.colorbar(
    im,
    ax=ax,
    orientation="vertical",
    fraction=0.035,
    pad=0.025
)

cbar.set_label(
    "Relative performance within each metric",
    fontsize=FS,
    labelpad=14
)

cbar.set_ticks([0, 0.25, 0.50, 0.75, 1.0])

cbar.set_ticklabels([
    "Lowest",
    "",
    "Intermediate",
    "",
    "Highest"
])

cbar.ax.tick_params(labelsize=FS, length=4)

# -------------------------------------------------------------------------
# Axis and layout
# -------------------------------------------------------------------------

ax.set_xlim(-0.5, n_metrics - 0.5)
ax.set_ylim(n_models - 0.5, -0.5)

ax.set_xlabel(
    "Performance metrics",
    fontsize=FS,
    labelpad=18
)

ax.set_ylabel(
    "Hydrological model / architecture",
    fontsize=FS,
    labelpad=15
)

ax.set_title(
    "(a) Multi-metric comparison of model performance",
    fontsize=FS,
    loc="left",
    pad=18
)

# Remove unnecessary outer spines
for spine in ax.spines.values():
    spine.set_visible(False)

fig.tight_layout()

# =============================================================================
# 6. EXPORT
# =============================================================================

fig.savefig(
    OUTPUT / "Model_Performance_Heatmap.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    OUTPUT / "Model_Performance_Heatmap.pdf",
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    OUTPUT / "Model_Performance_Heatmap.tiff",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

print("\nHeatmap successfully generated.")
print("Output directory:", OUTPUT)
print("\nFiles:")
print("1. Model_Performance_Heatmap.png")
print("2. Model_Performance_Heatmap.pdf")
print("3. Model_Performance_Heatmap.tiff")
print("4. model_performance_heatmap_data.csv")
print("5. normalized_performance_scores.csv")


In [ ]:
# =============================================================================
# MERGED FLOW DURATION CURVES (FDC): 3 x 3 PANELS, FONT SIZE 18

# =============================================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FixedFormatter, NullFormatter

warnings.filterwarnings("ignore")

# =============================================================================
# 1. PATHS
# =============================================================================

DATA_DIR = PROJECT_DIR / "data"
PREPARED_DIR = RESULTS_ROOT / "LSTM_Prepared_v4"
OUT = RESULTS_ROOT / "Projection_v4"
FDC_DIR = OUT / "fdc_plots"
FDC_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# 2. OPTIONS
# =============================================================================

BASELINE = (2000, 2021)
PERIODS = {"Near": (2041, 2070), "Far": (2071, 2100)}

MODELS = ["LSTM", "GRU", "TCN", "MLP", "LSTM_FCN"]
MODEL_LABEL = {"LSTM_FCN": "LSTM-FCN"}

STATIONS = ["Shebe", "Asendabo", "Abelti"]
STATION_IDS = {0: "Shebe", 1: "Asendabo", 2: "Abelti"}
SCENARIOS = ["SSP245", "SSP585"]

SHOW_INDIVIDUAL_MODELS = False   # add the five architectures to historical panels
SHOW_SWAT_FUTURE = True          # SWAT-only near/far curves in future panels
LOG_FLOOR = None                 # None = automatic

FS = 18
FIG_WIDTH, FIG_HEIGHT = 22, 21
LEGEND_COLUMNS = 3

COL = {
    "Observed": "#000000", "SWAT": "#D55E00", "Ensemble": "#009E73",
    "Historical": "#6e6e6e", "Near": "#E69F00", "Far": "#0072B2",
    "LSTM": "#1f77b4", "GRU": "#ff7f0e", "TCN": "#2ca02c",
    "MLP": "#d62728", "LSTM_FCN": "#9467bd",
}

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": FS, "axes.labelsize": FS, "axes.titlesize": FS,
    "xtick.labelsize": FS, "ytick.labelsize": FS, "legend.fontsize": FS,
    "axes.linewidth": 0.9,
    "axes.spines.top": False, "axes.spines.right": False,
})

# =============================================================================
# 3. CHECK INPUT FILES
# =============================================================================

DAILY_FILE = OUT / "daily_projections_all_models.csv"
if not DAILY_FILE.exists():
    raise FileNotFoundError(f"Daily projection file not found:\n{DAILY_FILE}")

for split in ("train", "val", "test"):
    f = PREPARED_DIR / "historical" / "lookback_030" / f"{split}.npz"
    if not f.exists():
        raise FileNotFoundError(f"Historical NPZ file not found:\n{f}")

# =============================================================================
# 4. LOAD DATA
# =============================================================================

daily = pd.read_csv(DAILY_FILE, parse_dates=["Date"])
need = ["Date", "Station", "Scenario", "Model", "SWAT", "Hybrid"]
miss = [c for c in need if c not in daily.columns]
if miss:
    raise ValueError(f"Missing columns in daily projection file: {miss}")

daily["SWAT"] = pd.to_numeric(daily["SWAT"], errors="coerce")
daily["Hybrid"] = pd.to_numeric(daily["Hybrid"], errors="coerce")
daily["Date"] = pd.to_datetime(daily["Date"], errors="coerce")
daily = daily.dropna(subset=["Date", "Station", "Scenario", "Model"])

print("Rows:", len(daily))
print("Stations:", sorted(daily["Station"].unique()))
print("Scenarios:", sorted(daily["Scenario"].unique()))
print("Models:", sorted(daily["Model"].unique()))

parts = []
for split in ("train", "val", "test"):
    f = PREPARED_DIR / "historical" / "lookback_030" / f"{split}.npz"
    with np.load(f, allow_pickle=False) as z:
        miss = [x for x in ("dates", "station", "Observed") if x not in z.files]
        if miss:
            raise ValueError(f"Missing arrays in {f.name}: {miss}")
        parts.append(pd.DataFrame({
            "Date": pd.to_datetime(z["dates"]),
            "Station": [STATION_IDS[int(s)] for s in z["station"]],
            "Observed": z["Observed"].astype(float)}))

obs = (pd.concat(parts, ignore_index=True).sort_values("Date")
       .drop_duplicates(subset=["Station", "Date"], keep="first"))
print("Observed rows:", len(obs))

# =============================================================================
# 5. HELPERS
# =============================================================================


def series(station, scenario):
    """SWAT, five hybrid architectures and ensemble mean (all five required)."""
    sub = daily[(daily["Station"] == station)
                & (daily["Scenario"] == scenario)].copy()
    if sub.empty:
        raise ValueError(f"No data found for {station} / {scenario}")

    out = sub.groupby("Date")["SWAT"].mean().to_frame("SWAT")
    for m in MODELS:
        ms = sub[sub["Model"] == m].copy()
        if ms.empty:
            raise ValueError(f"Missing model {m} for {station} / {scenario}")
        ms = ms.set_index("Date")
        if ms.index.has_duplicates:
            raise ValueError(f"Duplicate dates for {station}, {scenario}, {m}")
        out[m] = ms["Hybrid"].reindex(out.index)

    out["Ensemble"] = out[MODELS].mean(axis=1, skipna=False)

    if scenario == "Historical":
        o = (obs[obs["Station"] == station].drop_duplicates(subset=["Date"])
             .set_index("Date"))
        out["Observed"] = o["Observed"].reindex(out.index)
    return out.sort_index()


def window(df, y0, y1):
    return df[(df.index.year >= y0) & (df.index.year <= y1)].copy()


def fdc(values):
    """Weibull exceedance probability (%) and flows sorted high to low."""
    v = np.asarray(values, dtype=float)
    v = np.sort(v[np.isfinite(v)])[::-1]
    if len(v) == 0:
        return np.array([]), np.array([])
    return np.arange(1, len(v) + 1) / (len(v) + 1) * 100.0, v


def draw_fdc(ax, curves, xlab_bottom=False, xlab_top=False,
             ylab_left=False, ylab_right=False):
    """Dual-axis FDC. Main axis: log probability / linear flow.
    Overlay axis: linear probability / log flow."""
    ax2 = ax.inset_axes([0, 0, 1, 1])
    ax2.patch.set_alpha(0)

    all_positive, top = [], 0.0
    for c in curves:
        p, v = fdc(c["values"])
        if len(v) == 0:
            continue
        kw = dict(color=c["color"], linestyle=c.get("ls", "-"),
                  linewidth=c.get("lw", 1.6), alpha=c.get("alpha", 1.0),
                  zorder=c.get("z", 3))
        ax.plot(p, v, label=c["label"], **kw)
        ax2.plot(p, np.where(v > 0, v, np.nan), **kw)
        top = max(top, float(np.nanmax(v)))
        pos = v[v > 0]
        if len(pos):
            all_positive.append(pos)

    if all_positive:
        pos = np.concatenate(all_positive)
        floor = (float(LOG_FLOOR) if LOG_FLOOR is not None
                 else 10 ** np.floor(np.log10(np.nanpercentile(pos, 0.5))))
    else:
        floor = 1.0
    ceil = 10 ** np.ceil(np.log10(top)) if top > 0 else 10.0
    if floor >= ceil:
        ceil = floor * 10

    # main axis
    ax.set_xscale("log")
    ax.set_xlim(0.01, 100)
    ax.set_ylim(0, top * 1.05 if top > 0 else 1)
    ax.xaxis.set_major_locator(FixedLocator([0.01, 0.1, 1, 10, 100]))
    ax.xaxis.set_major_formatter(FixedFormatter(["0.01", "0.1", "1", "10", "100"]))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.grid(True, which="major", linestyle="--", linewidth=0.5, alpha=0.3)
    if xlab_bottom:
        ax.set_xlabel("Exceedance probability (%), log", labelpad=5)
    if ylab_left:
        ax.set_ylabel("Streamflow (m$^3$/s)", labelpad=6)

    # overlay axis
    ax2.set_xlim(0, 100)
    ax2.set_yscale("log")
    ax2.set_ylim(floor, ceil)
    ax2.xaxis.tick_top()
    ax2.xaxis.set_label_position("top")
    ax2.yaxis.tick_right()
    ax2.yaxis.set_label_position("right")
    if xlab_top:
        ax2.set_xlabel("Exceedance probability (%), linear", labelpad=5)
    if ylab_right:
        ax2.set_ylabel("Streamflow (m$^3$/s), log", labelpad=8)
    ax2.spines["top"].set_visible(True)
    ax2.spines["right"].set_visible(True)
    ax2.spines["left"].set_visible(False)
    ax2.spines["bottom"].set_visible(False)
    ax2.patch.set_alpha(0)
    return ax2


QUANTILE_ROWS = []


def record_quantiles(station, scenario, label, values):
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return
    row = {"Station": station, "Scenario": scenario, "Curve": label,
           "N_days": len(v), "Mean": np.mean(v)}
    for q in (1, 5, 10, 50, 90, 95, 99):
        row[f"Q{q}"] = np.percentile(v, 100 - q)
    QUANTILE_ROWS.append(row)


# =============================================================================
# 6. HISTORICAL DATA
# =============================================================================

hist = {}
for st in STATIONS:
    h = window(series(st, "Historical"), *BASELINE)
    h = h.dropna(subset=["Observed", "SWAT", "Ensemble"])
    if h.empty:
        raise ValueError(f"No common historical observations for {st}")
    hist[st] = h
    print(f"{st}: {len(h)} common historical days "
          f"({h.index.min().date()} to {h.index.max().date()})")

# =============================================================================
# 7. FIGURE: 3 x 3
# =============================================================================

fig = plt.figure(figsize=(FIG_WIDTH, FIG_HEIGHT))
gs = fig.add_gridspec(3, 3, left=0.075, right=0.93, top=0.95, bottom=0.105,
                      wspace=0.42, hspace=0.35)

letters = "abcdefghi"
panel_axes = []
k = 0

for i, st in enumerate(STATIONS):
    for j, col_name in enumerate(["Historical"] + SCENARIOS):
        ax = fig.add_subplot(gs[i, j])
        panel_axes.append(ax)
        h = hist[st]

        if j == 0:
            # ----- historical panel -----
            curves = []
            if SHOW_INDIVIDUAL_MODELS:
                for m in MODELS:
                    curves.append({"label": MODEL_LABEL.get(m, m),
                                   "values": h[m], "color": COL[m],
                                   "lw": 0.9, "alpha": 0.7, "z": 2})
            curves += [
                {"label": "Observed", "values": h["Observed"],
                 "color": COL["Observed"], "lw": 2.0, "z": 5},
                {"label": "SWAT", "values": h["SWAT"],
                 "color": COL["SWAT"], "lw": 1.8, "z": 4},
                {"label": "Hybrid ensemble mean", "values": h["Ensemble"],
                 "color": COL["Ensemble"], "ls": "--", "lw": 2.0, "z": 6},
            ]
            scen_label = "Historical"
        else:
            # ----- future panel -----
            sc = col_name
            fut = series(st, sc)
            curves = [{"label": "Historical ensemble mean",
                       "values": h["Ensemble"], "color": COL["Historical"],
                       "lw": 2.0, "z": 4}]
            for per, (a, z) in PERIODS.items():
                w = window(fut, a, z)
                if w.empty:
                    raise ValueError(f"No data for {st}, {sc}, {per} ({a}-{z})")
                print(f"{st} | {sc} | {per}: {len(w)} daily records "
                      f"({w.index.min().date()} to {w.index.max().date()})")
                curves.append({"label": f"{per} ensemble mean ({a}\u2013{z})",
                               "values": w["Ensemble"], "color": COL[per],
                               "lw": 2.0, "z": 5})
                if SHOW_SWAT_FUTURE:
                    curves.append({"label": f"{per} SWAT-only",
                                   "values": w["SWAT"], "color": COL[per],
                                   "ls": "--", "lw": 1.3, "alpha": 0.9, "z": 3})
            scen_label = sc

        draw_fdc(ax, curves,
                 xlab_bottom=(i == 2),
                 xlab_top=(i == 0),
                 ylab_left=(j == 0),
                 ylab_right=(j == 2))

        ax.set_title(f"({letters[k]}) {st} | {scen_label}", loc="left",
                     fontweight="bold", pad=(62 if i == 0 else 34))

        for c in curves:
            record_quantiles(st, scen_label, c["label"], c["values"])
        k += 1

# =============================================================================
# 8. ONE SHARED LEGEND
# =============================================================================

legend_handles = {}
for ax in panel_axes:
    hs, ls = ax.get_legend_handles_labels()
    for h_, l_ in zip(hs, ls):
        legend_handles.setdefault(l_, h_)

fig.legend(handles=list(legend_handles.values()),
           labels=list(legend_handles.keys()),
           loc="lower center", bbox_to_anchor=(0.5, 0.01),
           ncol=LEGEND_COLUMNS, frameon=False, fontsize=FS,
           columnspacing=2.0, handlelength=2.5, handletextpad=0.6)

# =============================================================================
# 9. SAVE
# =============================================================================

PNG_FILE = FDC_DIR / "FDC_Historical_and_Future_3x3.png"
PDF_FILE = FDC_DIR / "FDC_Historical_and_Future_3x3.pdf"
fig.savefig(PNG_FILE, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(PDF_FILE, bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)

q = pd.DataFrame(QUANTILE_ROWS)
if q.empty:
    raise ValueError("No flow quantiles were generated.")
CSV_FILE = FDC_DIR / "FDC_flow_quantiles.csv"
q.to_csv(CSV_FILE, index=False)

print("\nMERGED FIGURE:", PNG_FILE, "|", PDF_FILE)
print("QUANTILE TABLE:", CSV_FILE, q.shape)
print(q.round(2).head(15).to_string(index=False))

In [ ]:
# =============================================================================
# COMBINED FIGURE: MEMORY-HORIZON RESPONSE CURVES + PERMUTATION RELIANCE

# =============================================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")

FS = 18

# =============================================================================
# 1. PATHS AND SETTINGS
# =============================================================================

DATA_DIR = PROJECT_DIR / "data"

RESULT_DIRS = {
    "LSTM":     RESULTS_ROOT / "LSTM_Results_v4",
    "GRU":      RESULTS_ROOT / "GRU_Results_v4",
    "TCN":      RESULTS_ROOT / "TCN_Results_v4",
    "MLP":      RESULTS_ROOT / "MLP_Results_v4",
    "LSTM_FCN": RESULTS_ROOT / "LSTM_FCN_Results_v4",
}

SELECTION_FILE = (RESULTS_ROOT / "Future_SSP_Results_v4" / "tables"
                  / "Final_historical_model_selection.csv")
RELIANCE_FILE = RESULTS_ROOT / "Ablation_v4" / "permutation_reliance.csv"

OUT_DIR = RESULTS_ROOT / "Memory_Horizon_and_Reliance"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL = {"LSTM_FCN": "LSTM-FCN"}
COLORS = {"LSTM": "#0072B2", "GRU": "#D55E00", "TCN": "#009E73",
          "MLP": "#CC79A7", "LSTM_FCN": "#E69F00"}

YLIM = (0.50, 0.87)       # None = automatic

# heatmap settings
HEAT_MODELS = ["LSTM", "GRU", "TCN", "MLP", "LSTM-FCN"]
HEAT_GROUPS = ["SWAT flow", "SWAT states", "Forcing (P, Tmax, Tmin)",
               "Seasonality + T range", "Short precip memory (7-30 d)",
               "Long precip memory (90-365 d, EWMA)", "Static descriptors"]
HEAT_TICK = {   # display labels (wrapped to save width)
    "SWAT flow": "SWAT flow",
    "SWAT states": "SWAT states",
    "Forcing (P, Tmax, Tmin)": "Forcing\n(P, Tmax, Tmin)",
    "Seasonality + T range": "Seasonality\n+ T range",
    "Short precip memory (7-30 d)": "Short precip memory\n(7\u201330 d)",
    "Long precip memory (90-365 d, EWMA)": "Long precip memory\n(90\u2013365 d, EWMA)",
    "Static descriptors": "Static descriptors",
}
VLIM = 1.0   # colour scale clipped to +/-1

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": FS, "axes.labelsize": FS, "axes.titlesize": FS,
    "xtick.labelsize": FS, "ytick.labelsize": FS, "legend.fontsize": FS,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.8,
})

# =============================================================================
# 2. MEMORY HORIZON: LOAD AND SUMMARISE
# =============================================================================

frames = []
higher_is_better = {}

for name, folder in RESULT_DIRS.items():
    f = folder / "validation_results.csv"
    if not f.exists():
        print(f"WARNING: missing {f}")
        continue

    d = pd.read_csv(f)
    for c in ["composite", "lookback", "config_id", "seed"]:
        if c not in d.columns:
            raise ValueError(f"{f.name} lacks column '{c}'")

    d["composite"] = pd.to_numeric(d["composite"], errors="coerce")
    d = d.replace([np.inf, -np.inf], np.nan).dropna(subset=["composite"])

    n0 = len(d)
    d = d.drop_duplicates(subset=["lookback", "config_id", "seed"], keep="last")
    if len(d) < n0:
        print(f"{name}: removed {n0 - len(d)} duplicate trials")

    hib = True
    bf = folder / "best_model_summary.csv"
    if bf.exists():
        best_val = pd.to_numeric(pd.read_csv(bf)["composite"],
                                 errors="coerce").iloc[0]
        if abs(best_val - d["composite"].min()) < abs(best_val - d["composite"].max()):
            hib = False
    higher_is_better[name] = hib

    d["Backbone"] = name
    frames.append(d[["Backbone", "lookback", "config_id", "seed", "composite"]])
    print(f"{name:<9} trials={len(d):>5} | lookbacks="
          f"{[int(x) for x in sorted(d['lookback'].unique())]} | "
          f"higher is better={hib}")

if not frames:
    raise SystemExit("No validation_results.csv files were found.")

df = pd.concat(frames, ignore_index=True)
backbones = [b for b in RESULT_DIRS if b in df["Backbone"].unique()]

rows = []
for name in backbones:
    hib = higher_is_better[name]
    d = df[df["Backbone"] == name]
    for lb, g in d.groupby("lookback"):
        s = g["composite"]
        rows.append({
            "Backbone": name, "Lookback": int(lb), "N_trials": len(s),
            "N_configs": g["config_id"].nunique(),
            "N_seeds": g["seed"].nunique(),
            "Best": s.max() if hib else s.min(),
            "Median": s.median(),
            "Q25": s.quantile(0.25), "Q75": s.quantile(0.75)})

summary = pd.DataFrame(rows)
summary["IQR"] = summary["Q75"] - summary["Q25"]
summary["Best_minus_Median"] = (summary["Best"] - summary["Median"]).abs()

sd = (df.groupby(["Backbone", "lookback", "config_id"])["composite"].std()
      .reset_index(name="sd")
      .groupby(["Backbone", "lookback"])["sd"].median()
      .reset_index(name="Median_seed_SD")
      .rename(columns={"lookback": "Lookback"}))
sd["Lookback"] = sd["Lookback"].astype(int)
summary = summary.merge(sd, on=["Backbone", "Lookback"], how="left")
summary.to_csv(OUT_DIR / "memory_horizon_summary.csv", index=False)

best_rows = []
for name in backbones:
    s = summary[summary["Backbone"] == name]
    j = s["Best"].idxmax() if higher_is_better[name] else s["Best"].idxmin()
    best_rows.append(s.loc[j, ["Backbone", "Lookback", "Best", "Median", "N_trials"]])
best_df = pd.DataFrame(best_rows)
best_df.to_csv(OUT_DIR / "best_lookback_per_backbone.csv", index=False)
print("\nBest lookback per backbone (by best validation composite):")
print(best_df.round(4).to_string(index=False))

# selected lookbacks
selected_lb = {}
try:
    if SELECTION_FILE.exists():
        sel = pd.read_csv(SELECTION_FILE, encoding="utf-8-sig")
        sel.columns = [str(c).strip().lower() for c in sel.columns]
        arch_col = next((c for c in sel.columns
                         if c in ("architecture", "model", "backbone")), None)
        lb_col = next((c for c in sel.columns if "lookback" in c), None)
        if arch_col is None or lb_col is None:
            print("Selected lookbacks not drawn: columns not found.")
        else:
            for _, r in sel.iterrows():
                key = (str(r[arch_col]).strip().upper()
                       .replace("-", "_").replace(" ", "_"))
                selected_lb[key] = int(r[lb_col])
            print("Selected lookbacks:", selected_lb)
    else:
        print(f"Selection file not found: {SELECTION_FILE}")
except Exception as e:
    print(f"Selected lookbacks not drawn ({type(e).__name__}: {e})")

all_lookbacks = sorted(summary["Lookback"].unique())

# =============================================================================
# 3. PERMUTATION RELIANCE: LOAD
# =============================================================================

res = pd.read_csv(RELIANCE_FILE)

# =============================================================================
# 4. FIGURE LAYOUT
#    outer row 0 : memory-horizon panels (2 x 3)
#    outer row 1 : memory-horizon legend
#    outer row 2 : two heatmaps + colour bar
# =============================================================================
n = len(backbones)
ncols = 3
nrows = int(np.ceil((n + 1) / ncols))

fig = plt.figure(figsize=(19, 5.2 * nrows + 1.3 + 7.2 + 1.0))
outer = GridSpec(3, 1, figure=fig,
                 height_ratios=[5.2 * nrows, 0.65, 7.2],
                 left=0.07, right=0.985, top=0.975, bottom=0.07,
                 hspace=0.12)

gs_top = outer[0].subgridspec(nrows, ncols, wspace=0.22, hspace=0.30)
gs_leg = outer[1]
gs_bot = outer[2].subgridspec(1, 3, width_ratios=[1, 1, 0.04], wspace=0.10)

letters = "abcdefghij"

if YLIM is None:
    lo = summary["Median"].min()
    hi = summary["Best"].max()
    pad = 0.08 * (hi - lo if hi > lo else 1.0)
    ylim = (lo - pad, hi + 2.5 * pad)
else:
    ylim = YLIM

# ---- memory-horizon panels --------------------------------------------------
flat = []
for r in range(nrows):
    for c in range(ncols):
        flat.append(fig.add_subplot(gs_top[r, c]))

for k, name in enumerate(backbones):
    ax = flat[k]
    hib = higher_is_better[name]
    s = summary[summary["Backbone"] == name].sort_values("Lookback")
    col = COLORS[name]

    ax.fill_between(s["Lookback"], s["Q25"], s["Q75"], color=col,
                    alpha=0.18, linewidth=0, zorder=1)
    ax.plot(s["Lookback"], s["Median"], color=col, lw=1.6, ls="--",
            marker="o", ms=5, mfc="white", mec=col, zorder=3)
    ax.plot(s["Lookback"], s["Best"], color=col, lw=2.2, marker="o",
            ms=6, zorder=4)

    j = s["Best"].idxmax() if hib else s["Best"].idxmin()
    ax.scatter(s.loc[j, "Lookback"], s.loc[j, "Best"], marker="*", s=260,
               color="gold", edgecolor="black", linewidth=0.9, zorder=6)
    ax.annotate(f"{int(s.loc[j, 'Lookback'])} d",
                (s.loc[j, "Lookback"], s.loc[j, "Best"]),
                xytext=(0, 12), textcoords="offset points", ha="center",
                fontsize=FS, fontweight="bold")

    if name in selected_lb:
        ax.axvline(selected_lb[name], color="#555555", ls=":", lw=1.2, zorder=0)

    ax.set_title(f"({letters[k]}) {LABEL.get(name, name)}", loc="left",
                 fontweight="bold")
    ax.set_xticks(all_lookbacks)
    ax.set_ylim(*ylim)
    ax.grid(True, linestyle="--", lw=0.6, alpha=0.35)
    if k % ncols == 0:
        ax.set_ylabel("Validation composite score")
    ax.set_xlabel("Lookback window (days)")

ax = flat[n]
for name in backbones:
    s = summary[summary["Backbone"] == name].sort_values("Lookback")
    ax.plot(s["Lookback"], s["Best"], color=COLORS[name], lw=2.0,
            marker="o", ms=5, label=LABEL.get(name, name))
ax.set_title(f"({letters[n]}) Best score, all backbones", loc="left",
             fontweight="bold")
ax.set_xticks(all_lookbacks)
ax.set_ylim(*ylim)
ax.grid(True, linestyle="--", lw=0.6, alpha=0.35)
ax.set_xlabel("Lookback window (days)")
ax.legend(frameon=False, ncol=2, loc="lower center", fontsize=FS,
          columnspacing=1.0, handlelength=1.4)

for ax in flat[n + 1:]:
    ax.set_visible(False)

# ---- memory-horizon legend --------------------------------------------------
leg_ax = fig.add_subplot(gs_leg)
leg_ax.axis("off")
legend_handles = [
    Line2D([0], [0], color="#444444", lw=2.2, marker="o", label="Best"),
    Line2D([0], [0], color="#444444", lw=1.6, ls="--", marker="o",
           mfc="white", label="Median"),
    Patch(facecolor="#444444", alpha=0.2,
          label="25th\u201375th percentile (configs \u00d7 seeds)"),
    Line2D([0], [0], marker="*", color="w", markerfacecolor="gold",
           markeredgecolor="black", markersize=14, label="Best lookback"),
]
if selected_lb:
    legend_handles.append(Line2D([0], [0], color="#555555", ls=":", lw=1.2,
                                 label="Selected lookback"))

leg_ax.legend(handles=legend_handles, loc="center", ncol=5, frameon=False,
              fontsize=FS, columnspacing=1.2, handletextpad=0.5, borderaxespad=0.0, labelspacing=0.3)

# ---- permutation reliance heatmaps ------------------------------------------
k = n + 1
im = None
heat_axes = []
for hi_, (m, title) in enumerate([("dNSE", "Change in NSE"),
                                  ("dKGE", "Change in KGE")]):
    ax = fig.add_subplot(gs_bot[0, hi_])
    heat_axes.append(ax)

    val = res.pivot(index="Group", columns="Model", values=m) \
             .reindex(index=HEAT_GROUPS, columns=HEAT_MODELS)
    sdv = res.pivot(index="Group", columns="Model", values=m + "_sd") \
             .reindex(index=HEAT_GROUPS, columns=HEAT_MODELS)

    im = ax.imshow(val.values, cmap="RdBu", vmin=-VLIM, vmax=VLIM,
                   aspect="auto")
    for i in range(val.shape[0]):
        for j in range(val.shape[1]):
            v, s_ = val.values[i, j], sdv.values[i, j]
            if not np.isfinite(v):
                continue
            noisy = abs(v) <= s_
            ax.text(j, i, f"{v:+.2f}", ha="center", va="center",
                    fontsize=FS,
                    color="0.45" if noisy else "black",
                    style="italic" if noisy else "normal",
                    weight="normal" if noisy else "bold")

    ax.set_xticks(range(len(HEAT_MODELS)))
    ax.set_xticklabels(HEAT_MODELS, rotation=30, ha="right", fontsize=FS)
    ax.set_yticks(range(len(HEAT_GROUPS)))
    if hi_ == 0:
        ax.set_yticklabels([HEAT_TICK[g] for g in HEAT_GROUPS], fontsize=FS)
    else:
        ax.set_yticklabels([])
    ax.set_title(f"({letters[k]}) {title}", loc="left", fontweight="bold")
    ax.set_xticks(np.arange(-.5, len(HEAT_MODELS), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(HEAT_GROUPS), 1), minor=True)
    ax.grid(which="minor", color="white", lw=1.5)
    ax.tick_params(which="minor", length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    k += 1

cax = fig.add_subplot(gs_bot[0, 2])
cb = fig.colorbar(im, cax=cax)
cb.set_label("Change after permutation", fontsize=FS)
cb.ax.tick_params(labelsize=FS)


# =============================================================================
# 5. EXPORT
# =============================================================================

fig.savefig(OUT_DIR / "Memory_Horizon_and_Reliance.png", dpi=600,
            bbox_inches="tight", facecolor="white")
fig.savefig(OUT_DIR / "Memory_Horizon_and_Reliance.pdf",
            bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)
print("\nSaved in:", OUT_DIR)